# 🚪 FACE RECOGNITION & LPR — نسخة البوابة المدمجة — نظام البوابة الذكية
**تعرّف وجوه (InsightFace) + قراءة لوحات سعودية (موديل حروف مدرّب)**

### دليل التشغيل — موديلك مدرّب وجاهز بالدرايف ✅ (ما تحتاج تدرّب)
1. **① الإعداد** — شغّل كل خلاياه (كل جلسة).
2. **② تحميل الموديل من Drive** — يجيب موديلك الجاهز `plate_chars.pt`.
3. **③ (اختياري) رفع الوجوه + الأسماء** — لو تبي التعرّف على الوجوه (قبل التشغيل).
4. **④ التشغيل** — اختر المحرك ← 🚀 ← 🩺 الرابط.
5. **⑤ الاختبار** — جرّب قراءة لوحات.

> **⑥ إعادة التدريب = اختياري** — فقط لو تبي تحسّن الموديل (يحتاج GPU + وقت).
> ⚠️ Runtime ← Change runtime type ← **GPU**.

# ⚙️ ① الإعداد (شغّل كل القسم كل جلسة)

In [ ]:
!pip install -q flask insightface onnxruntime-gpu opencv-python pillow
!pip install -q easyocr
!pip install -q ultralytics huggingface_hub
print('✅ تم')

import os
for d in ['templates','static/css','static/enrolled']:
    os.makedirs(f'/content/app/{d}', exist_ok=True)
os.chdir('/content/app')
print('✅ تم')

### 📝 ملفات التطبيق — شغّل كل خلايا هذا القسم (app.py + CSS + القوالب)

In [ ]:
%%writefile /content/app/app.py
"""
Smart Gate System - Flask Backend (v3)
- اكتشاف الجنس والعمر تلقائياً من الصورة
- قراءة بيانات الموظفين من employees.csv
- صفحة تسجيل أشخاص جدد بصورهم ولوحاتهم
"""

from flask import Flask, render_template, request, jsonify
import os
import cv2
import numpy as np
import pickle
import json
import csv
import random
import hashlib
from datetime import datetime, timezone, timedelta
from PIL import Image, ImageOps
import io
import base64
import threading

app = Flask(__name__)
app.config['MAX_CONTENT_LENGTH'] = 50 * 1024 * 1024

# ============================================================
# إعدادات
# ============================================================
DATASET_PATH = 'smart_gate_dataset/smart_gate_dataset_augmented'
FACE_DB_PATH = 'face_database.pkl'
EMPLOYEES_CSV = 'employees.csv'
EMPLOYEES_DB_PATH = 'employees_database.json'
LOGS_PATH = 'access_logs.json'
PLATES_DIR = 'static/plates'   # صور اللوحات
THRESHOLD = float(os.environ.get('FACE_THRESHOLD', 0.55))
GATE_PI_URL = os.environ.get('GATE_PI_URL', '')   # عنوان Raspberry Pi لفتح البوابة

# توقيت السعودية (UTC+3) — سيرفرات Colab تمشي على UTC فلازم نثبّت المنطقة الزمنية
SA_TZ = timezone(timedelta(hours=3))
def now_sa():
    return datetime.now(SA_TZ)   # عنوان Raspberry Pi لفتح البوابة، مثال: http://192.168.1.50:8000/open

os.makedirs(PLATES_DIR, exist_ok=True)
os.makedirs('static/enrolled', exist_ok=True)

# ============================================================
# جداول البيانات الافتراضية (للتوليد لو ما فيه CSV)
# ============================================================
NATIONALITY_AR = {
    'saudi': 'سعودي', 'egyptian': 'مصري', 'indian': 'هندي',
    'pakistani': 'باكستاني', 'bangladeshi': 'بنغالي', 'nepali': 'نيبالي'
}

# --- مستويات الصلاحية (هرمي: أ أعلى من ب أعلى من ج) ---
CLEARANCE_RANK = {'A': 3, 'B': 2, 'C': 1}
CLEARANCE_AR = {'A': 'أ', 'B': 'ب', 'C': 'ج'}
GATE_ZONE = 'B'   # المنطقة المطلوبة لهذي البوابة — غيّرها A/B/C (A=الأشد، C=الأدنى)

PLATE_LETTERS = [
    ('ا','A'),('ب','B'),('ح','J'),('د','D'),('ر','R'),('س','S'),('ص','X'),
    ('ط','T'),('ع','E'),('ق','G'),('ك','K'),('ل','L'),('م','Z'),('ن','N'),
    ('هـ','H'),('و','U'),('ى','V')
]
# خرائط نظام لوحات المرور السعودي (لاتيني <-> عربي)
LAT2AR = {en: ar for ar, en in PLATE_LETTERS}
AR2LAT = {ar: en for ar, en in PLATE_LETTERS}
AR2LAT['ه'] = 'H'   # على اللوحة تُكتب هـ لكن OCR أحياناً يرجّعها ه مفردة
AR_DIGITS = '٠١٢٣٤٥٦٧٨٩'

def to_ar_digits(s):
    return ''.join(AR_DIGITS[int(d)] if d.isdigit() else d for d in str(s))

def gen_plate(rnd):
    letters = rnd.sample(PLATE_LETTERS, 3)
    digits = ''.join([str(rnd.randint(0,9)) for _ in range(4)])
    return {
        'plate_ar': f"{' '.join(l[0] for l in letters)}  {to_ar_digits(digits)}",
        'plate_en': f"{' '.join(l[1] for l in reversed(letters))}  {digits}"
    }

# ============================================================
# متغيرات عامة
# ============================================================
face_app = None
ACTIVE_MODEL = None
face_database = {}
employees_db = {}
access_logs = []

# --- LPR (قراءة اللوحات) — مستقل تماماً عن نظام الوجوه ---
VEHICLE_LOGS_PATH = 'vehicle_logs.json'
plate_model = None
ocr_reader = None
ocr_reader_en = None   # قارئ إنجليزي مستقل لصف اللوحة اللاتيني
char_model = None       # موديل الحروف المدرّب (اختياري) — لو وُجد يصير الأساس وOCR احتياط
# محرك القراءة الأساسي (نصيحة د. صالح: EasyOCR أساسي يتجنّب التباس موديل الحروف):
#   'easyocr' (افتراضي) = الصف اللاتيني بـEasyOCR ثم تحويل عربي بـLAT2AR، وموديل الحروف احتياط
#   'chars'             = موديل الحروف أساس، EasyOCR احتياط (سلوك Final الأصلي 100%)
#   'hybrid'            = الأرقام من موديل الحروف (دقيق عليها) + الحروف من EasyOCR
PRIMARY_ENGINE = os.environ.get('PRIMARY_ENGINE', 'easyocr')   # 'easyocr' | 'chars' | 'hybrid'
CHAR_CANON = {}         # id الصنف -> رمز قانوني (حرف لاتيني من الـ17 أو رقم)
lpr_ready = False
vehicle_state = {}   # plate_key -> 'inside' / 'outside'
vehicle_logs = []
_plate_last_seen = {}      # plate_key -> آخر وقت قراءة (لمنع تكرار دخول/خروج بالغلط)
PLATE_COOLDOWN_SEC = 15

# آخر لوحة انقرأت عند البوابة — تُقرن بأقرب تعرف وجه خلال النافذة الزمنية
# ملاحظة تصميم: هذا تسجيل سياقي فقط (بأي مركبة وصل الشخص) — مو شرط دخول،
# الشخص ممكن يجي بأي سيارة وقرار البوابة يعتمد على وجهه فقط.
last_gate_plate = None
PLATE_PAIR_WINDOW_SEC = int(os.environ.get('PLATE_PAIR_WINDOW', 120))


def detect_gender_age(face):
    """يستخرج الجنس والعمر من كائن الوجه (InsightFace)"""
    # InsightFace: sex='M'/'F', age=رقم
    gender = 'male' if face.sex == 'M' else 'female'
    age = int(face.age)
    return gender, age


def initialize_system():
    global face_app, ACTIVE_MODEL, face_database, employees_db, access_logs, vehicle_state, vehicle_logs

    print('=' * 60)
    print('Smart Gate System v3 - Starting...')
    print('=' * 60)

    print('[1/5] Loading InsightFace (antelopev2 - دقة عالية)...')
    from insightface.app import FaceAnalysis

    def _load_model(model_name):
        """يحمّل الموديل (GPU وإلا CPU) ويرجّع كائن FaceAnalysis أو يرفع استثناء."""
        import onnxruntime as ort
        use_gpu = 'CUDAExecutionProvider' in ort.get_available_providers()
        try:
            if not use_gpu:
                raise RuntimeError('CUDA غير متاح')
            fa = FaceAnalysis(name=model_name, providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
            fa.prepare(ctx_id=0, det_size=(640, 640))
            print(f'      OK ({model_name} / GPU)')
        except Exception:
            fa = FaceAnalysis(name=model_name, providers=['CPUExecutionProvider'])
            fa.prepare(ctx_id=-1, det_size=(640, 640))
            print(f'      OK ({model_name} / CPU)')
        return fa

    def _fix_nested_model_dir(model_name):
        """يصلّح مشكلة معروفة في antelopev2: الـ zip ينفك بمجلد داخل مجلد فما يلقى ملفات onnx."""
        import glob, shutil
        root = os.path.expanduser(f'~/.insightface/models/{model_name}')
        nested = os.path.join(root, model_name)
        if os.path.isdir(nested) and not glob.glob(os.path.join(root, '*.onnx')):
            for fn in os.listdir(nested):
                shutil.move(os.path.join(nested, fn), os.path.join(root, fn))
            os.rmdir(nested)
            return True
        return False

    # نحاول الموديل الأقوى، ولو فشل نرجع لموديل احتياطي — ونسجّل أي موديل اشتغل فعلاً
    for candidate in ('antelopev2', 'buffalo_l'):
        try:
            face_app = _load_model(candidate)
            ACTIVE_MODEL = candidate
            break
        except Exception as e:
            # نحاول إصلاح مشكلة فك الضغط المتداخل ثم نعيد المحاولة مرة وحدة
            try:
                if _fix_nested_model_dir(candidate):
                    face_app = _load_model(candidate)
                    ACTIVE_MODEL = candidate
                    break
            except Exception as e2:
                e = e2
            print(f'      ⚠️ تعذّر تحميل {candidate}: {e}')
    if face_app is None:
        raise RuntimeError('فشل تحميل أي موديل للوجوه (antelopev2 / buffalo_l)')

    print('[2/5] Building/loading face database...')
    # نبني دايماً عشان نستخرج الجنس والعمر مع الـ embedding
    build_face_database()
    if not face_database:
        print('⚠️⚠️ قاعدة الوجوه فارغة! تأكد إن الداتاسيت مرفوع، وشغّل خلية تنظيف الكاش، ثم أعد التشغيل.')

    print('[3/5] Loading employee data...')
    load_employees()

    print('[4/5] Loading logs...')
    if os.path.exists(LOGS_PATH):
        with open(LOGS_PATH, 'r', encoding='utf-8') as f:
            access_logs = json.load(f)
    print(f'      {len(access_logs)} logs')

    print('[5/5] Loading LPR (plate detection + OCR)...')
    load_lpr()
    if os.path.exists(VEHICLE_LOGS_PATH):
        with open(VEHICLE_LOGS_PATH, 'r', encoding='utf-8') as f:
            vdata = json.load(f)
        vehicle_state = vdata.get('state', {})
        vehicle_logs = vdata.get('logs', [])
    print(f'      {len(vehicle_logs)} vehicle logs')

    print('=' * 60)
    print('System ready!')
    print('=' * 60)


def _save_face_db():
    """يحفظ قاعدة الوجوه مع اسم الموديل اللي بناها (عشان نكشف التعارض)."""
    if not face_database:
        return   # لا نحفظ قاعدة فاضية في الكاش
    with open(FACE_DB_PATH, 'wb') as f:
        pickle.dump({'__model__': ACTIVE_MODEL, 'data': face_database}, f)


def build_face_database():
    """يبني embeddings + يستخرج الجنس والعمر من كل شخص"""
    global face_database

    # نخزن: {'__model__': اسم الموديل, 'data': {person: {'embedding':..., 'gender':..., 'age':...}}}
    if os.path.exists(FACE_DB_PATH):
        with open(FACE_DB_PATH, 'rb') as f:
            cached = pickle.load(f)
        cached_model = cached.get('__model__') if isinstance(cached, dict) and '__model__' in cached else None
        if cached_model == ACTIVE_MODEL and cached.get('data'):
            face_database = cached['data']
            print(f'      Loaded {len(face_database)} faces from cache ({cached_model})')
            return
        # الكاش اتبنى بموديل مختلف -> الـ embeddings غير متوافقة، نعيد البناء
        print(f'      ⚠️ الكاش بموديل ({cached_model}) يخالف الموديل الحالي ({ACTIVE_MODEL}) — إعادة بناء القاعدة')

    if not os.path.exists(DATASET_PATH):
        print(f'      Dataset not found: {DATASET_PATH}')
        return

    people = sorted([d for d in os.listdir(DATASET_PATH)
                     if os.path.isdir(os.path.join(DATASET_PATH, d))])

    for person in people:
        folder = os.path.join(DATASET_PATH, person)
        embeddings = []
        genders = []
        ages = []

        for img_name in os.listdir(folder):
            if not img_name.lower().endswith(('.jpg','.jpeg','.png')):
                continue
            img = cv2.imread(os.path.join(folder, img_name))
            if img is None:
                continue
            faces = face_app.get(img)
            if len(faces) == 0:
                continue
            face = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))
            embeddings.append(face.embedding)
            g, a = detect_gender_age(face)
            genders.append(g)
            ages.append(a)

        if embeddings:
            # الجنس: الأغلبية، العمر: المتوسط
            gender = max(set(genders), key=genders.count)
            age = int(np.mean(ages))
            face_database[person] = {
                'embedding': np.mean(embeddings, axis=0),
                'gender': gender,
                'age': age
            }
            print(f'      {person}: {len(embeddings)} imgs, {gender}, age~{age}')

    _save_face_db()


def load_employees():
    """يقرأ بيانات الموظفين من CSV، ويولّد الناقص تلقائياً"""
    global employees_db

    # نقرأ الـ CSV لو موجود
    csv_data = {}
    if os.path.exists(EMPLOYEES_CSV):
        with open(EMPLOYEES_CSV, 'r', encoding='utf-8-sig') as f:
            reader = csv.DictReader(f)
            if reader.fieldnames and 'folder' in reader.fieldnames:
                for row in reader:
                    csv_data[row['folder']] = row
                print(f'      Loaded {len(csv_data)} rows from CSV')
            else:
                print(f'      ⚠️ employees.csv ما فيه عمود folder — تجاهلنا الملف (الأعمدة: {reader.fieldnames})')
    else:
        # نولّد قالب CSV فاضي للتعبئة
        generate_csv_template()
        print(f'      Created template: {EMPLOYEES_CSV}')

    # نحافظ على بيانات المسجّلين من صفحة التسجيل عبر إعادة التشغيل
    saved = {}
    if os.path.exists(EMPLOYEES_DB_PATH):
        try:
            with open(EMPLOYEES_DB_PATH, 'r', encoding='utf-8') as f:
                saved = json.load(f)
        except Exception:
            saved = {}

    # نبني بيانات كل شخص
    for person, face_info in face_database.items():
        if person.startswith('enrolled_') and person in saved and person not in csv_data:
            employees_db[person] = saved[person]
            continue
        rnd = random.Random(int(hashlib.md5(person.encode('utf-8')).hexdigest(), 16) % (2**32))
        detected_gender = face_info.get('gender', 'male')
        detected_age = face_info.get('age', 30)

        if person in csv_data:
            row = csv_data[person]
            nationality = row.get('nationality', 'saudi').strip() or 'saudi'
            name_ar = row.get('name_ar', '').strip()
            name_en = row.get('name_en', '').strip()
            rank_ar = row.get('rank_ar', '').strip() or 'موظف مدني'
            rank_en = row.get('rank_en', '').strip() or 'Civilian Employee'
            national_id = row.get('national_id', '').strip()
            plate_ar = row.get('plate_ar', '').strip()
            plate_en = row.get('plate_en', '').strip()
            brand_ar = row.get('brand_ar', '').strip() or 'تويوتا'
            color_ar = row.get('color_ar', '').strip() or 'أبيض'
            status = row.get('access_status', 'ACTIVE').strip() or 'ACTIVE'
            clearance_raw = row.get('clearance', '').strip().upper()
        else:
            nationality = 'saudi'
            name_ar = name_en = ''
            rank_ar, rank_en = 'موظف مدني', 'Civilian Employee'
            national_id = ''
            plate_ar = plate_en = ''
            brand_ar, color_ar = 'تويوتا', 'أبيض'
            status = 'ACTIVE'
            clearance_raw = ''

        # توليد الناقص
        if not national_id:
            prefix = '1' if nationality == 'saudi' else '2'
            national_id = prefix + ''.join([str(rnd.randint(0,9)) for _ in range(9)])
        if not plate_ar:
            p = gen_plate(rnd)
            plate_ar, plate_en = p['plate_ar'], p['plate_en']
        if not name_ar:
            name_ar = 'غير محدد'
            name_en = 'Not Set'

        clearance = clearance_raw if clearance_raw in CLEARANCE_RANK else 'A'   # افتراضياً صلاحية كاملة؛ قيّد من employees.csv لو تبي

        employees_db[person] = {
            'folder_name': person,
            'national_id': national_id,
            'id_type_ar': 'هوية وطنية' if nationality == 'saudi' else 'إقامة',
            'name_ar': name_ar,
            'name_en': name_en,
            'gender': detected_gender,        # ← من الصورة تلقائياً
            'gender_ar': 'ذكر' if detected_gender == 'male' else 'أنثى',
            'age': detected_age,              # ← من الصورة تلقائياً
            'nationality': nationality,
            'nationality_ar': NATIONALITY_AR.get(nationality, nationality),
            'rank_ar': rank_ar,
            'rank_en': rank_en,
            'access_status': status,
            'access_status_ar': 'موقوف' if status == 'SUSPENDED' else 'نشط',
            'clearance': clearance,
            'clearance_ar': CLEARANCE_AR.get(clearance, 'ج'),
            'vehicle': {
                'plate_ar': plate_ar,
                'plate_en': plate_en,
                'brand_ar': brand_ar,
                'color_ar': color_ar
            }
        }

    # نحفظ نسخة JSON
    with open(EMPLOYEES_DB_PATH, 'w', encoding='utf-8') as f:
        json.dump(employees_db, f, ensure_ascii=False, indent=2)


def generate_csv_template():
    """يولّد ملف CSV فاضي بأسماء المجلدات عشان المستخدم يعبيه"""
    people = sorted(face_database.keys())
    with open(EMPLOYEES_CSV, 'w', encoding='utf-8-sig', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([
            'folder', 'name_ar', 'name_en', 'nationality',
            'rank_ar', 'rank_en', 'national_id',
            'plate_ar', 'plate_en', 'brand_ar', 'color_ar', 'access_status', 'clearance'
        ])
        for person in people:
            gender = face_database[person].get('gender', 'male')
            writer.writerow([
                person, '', '', 'saudi',
                'موظف مدني', 'Civilian Employee', '',
                '', '', 'تويوتا', 'أبيض', 'ACTIVE', ''
            ])


def detect_spoofing(img_bgr, face):
    """
    كشف الانتحال (Anti-Spoofing) متعدّد المؤشرات.
    نحسب عدة مؤشرات بصرية ونجمعها في درجة حيوية (0..1) ثم نقرّر بعتبة معايرة:
      1. الحدة (Laplacian)  2. الطاقة عالية التردد (FFT)
      3. اللمعان الزائد      4. تباين التشبع اللوني
    يرجّع: (هل حقيقي؟, درجة الحيوية, السبب)
    """
    try:
        x1, y1, x2, y2 = map(int, face.bbox)
        h, w = img_bgr.shape[:2]
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)
        face_crop = img_bgr[y1:y2, x1:x2]
        if face_crop.size == 0 or face_crop.shape[0] < 20 or face_crop.shape[1] < 20:
            return True, 1.0, 'face_too_small'

        gray = cv2.cvtColor(face_crop, cv2.COLOR_BGR2GRAY)
        lap_var = cv2.Laplacian(gray, cv2.CV_64F).var()
        sharp_score = float(np.clip(lap_var / 150.0, 0.0, 1.0))

        ff = np.fft.fftshift(np.fft.fft2(gray))
        mag = np.abs(ff)
        ch, cw = mag.shape[0] // 2, mag.shape[1] // 2
        r = max(1, min(ch, cw) // 4)
        mask = np.ones_like(mag, dtype=bool)
        mask[ch - r:ch + r, cw - r:cw + r] = False
        high_freq_ratio = float(mag[mask].sum() / (mag.sum() + 1e-8))
        hf_score = float(np.clip(high_freq_ratio / 0.55, 0.0, 1.0))

        bright_ratio = float(np.sum(gray > 240) / gray.size)
        glare_score = float(np.clip(1.0 - (bright_ratio / 0.12), 0.0, 1.0))

        hsv = cv2.cvtColor(face_crop, cv2.COLOR_BGR2HSV)
        sat_std = float(hsv[:, :, 1].std())
        sat_score = float(np.clip(sat_std / 40.0, 0.0, 1.0))

        liveness = (0.35 * sharp_score + 0.30 * hf_score +
                    0.20 * glare_score + 0.15 * sat_score)
        SPOOF_THRESHOLD = float(os.environ.get('SPOOF_THRESHOLD', 0.45))
        is_real = liveness >= SPOOF_THRESHOLD
        if is_real:
            reason = 'real'
        else:
            parts = {'low_sharpness': sharp_score, 'low_detail': hf_score,
                     'screen_glare': glare_score, 'flat_color': sat_score}
            reason = min(parts, key=parts.get)
        return is_real, round(float(liveness), 3), reason
    except Exception:
        return True, 1.0, 'check_failed'


def recognize_face_from_bytes(image_bytes, check_spoof=True):
    img_pil = Image.open(io.BytesIO(image_bytes))
    img_pil = ImageOps.exif_transpose(img_pil)
    img_np = np.array(img_pil)

    if len(img_np.shape) == 2:
        img_bgr = cv2.cvtColor(img_np, cv2.COLOR_GRAY2BGR)
    elif img_np.shape[2] == 4:
        img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGBA2BGR)
    else:
        img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)

    faces = face_app.get(img_bgr)
    if len(faces) == 0:
        return {'status': 'error', 'message': 'لم يتم اكتشاف وجه', 'score': 0}

    face = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))
    num_faces = len(faces)

    # فحص Anti-Spoofing
    spoof_info = {'is_real': True, 'reason': 'ok'}
    if check_spoof:
        is_real, spoof_conf, reason = detect_spoofing(img_bgr, face)
        spoof_info = {'is_real': is_real, 'confidence': spoof_conf, 'reason': reason}
        if not is_real:
            return {
                'status': 'spoof_detected',
                'message': 'تم اكتشاف محاولة انتحال (صورة غير حقيقية)',
                'score': 0,
                'spoof_info': spoof_info,
                'num_faces': num_faces
            }

    query = face.embedding

    if not face_database:
        return {'status': 'error', 'message': '⚠️ قاعدة الوجوه فارغة — أعد رفع الداتاسيت ثم شغّل خلية تنظيف الكاش وأعد التشغيل', 'score': 0}

    best_match, best_score = None, -1
    for person, info in face_database.items():
        emb = info['embedding']
        score = np.dot(query, emb) / (np.linalg.norm(query) * np.linalg.norm(emb))
        if score > best_score:
            best_score = score
            best_match = person

    if best_score >= THRESHOLD:
        return {'status': 'recognized', 'folder': best_match, 'score': float(best_score), 'spoof_info': spoof_info, 'num_faces': num_faces}
    return {'status': 'unknown', 'message': 'الشخص غير مسجل', 'score': float(best_score), 'num_faces': num_faces}


def save_logs():
    with open(LOGS_PATH, 'w', encoding='utf-8') as f:
        json.dump(access_logs, f, ensure_ascii=False, indent=2)


# ============================================================
# LPR — كشف اللوحة (YOLO) + قراءة النص (OCR) + دخول/خروج
# ============================================================
def load_lpr():
    """يحمّل موديل كشف اللوحة وقارئ النص — بشكل آمن لا يكسر النظام لو فشل."""
    global plate_model, ocr_reader, ocr_reader_en, char_model, CHAR_CANON, lpr_ready
    try:
        from huggingface_hub import hf_hub_download, list_repo_files
        from ultralytics import YOLO as PlateYOLO

        def _try_load_plate_model(repo_id):
            files = [f for f in list_repo_files(repo_id) if f.endswith('.pt')]
            if not files:
                raise RuntimeError('لا يوجد ملف أوزان .pt في المستودع')
            files.sort(key=len)
            wpath = hf_hub_download(repo_id=repo_id, filename=files[0])
            try:
                m = PlateYOLO(wpath)
            except Exception:
                # checkpoint قديم: مسار ultralytics.yolo المحذوف + torch>=2.6 weights_only
                import sys as _sys
                import ultralytics as _ul
                import ultralytics.utils as _ulu
                import ultralytics.models.yolo as _ulm
                _sys.modules.setdefault('ultralytics.yolo', _ul)
                _sys.modules.setdefault('ultralytics.yolo.utils', _ulu)
                _sys.modules.setdefault('ultralytics.yolo.v8', _ulm)
                import torch as _torch
                _orig_load = _torch.load
                def _load_compat(*a, **k):
                    k['weights_only'] = False
                    return _orig_load(*a, **k)
                _torch.load = _load_compat
                try:
                    m = PlateYOLO(wpath)
                finally:
                    _torch.load = _orig_load
            # فحص سريع إن الموديل يتنبأ فعلاً (يكشف أعطال التوافق الصامتة)
            m.predict(np.zeros((64, 64, 3), dtype=np.uint8), verbose=False)
            return m

        # (0) موديلك المدرّب الخاص إن وجد (من خلية التدريب 🎓) — له الأولوية
        if os.path.exists('models/plate_custom.pt'):
            try:
                plate_model = PlateYOLO('models/plate_custom.pt')
                plate_model.predict(np.zeros((64, 64, 3), dtype=np.uint8), verbose=False)
                print('      Plate detector OK (الموديل المدرّب محلياً ✨)')
            except Exception as e:
                plate_model = None
                print(f'      ⚠️ تعذّر تحميل الموديل المدرّب محلياً: {e}')
        # (1) وإلا: الموديلات الجاهزة من Hugging Face
        if plate_model is None:
            for _repo in ('morsetechlab/yolov11-license-plate-detection',
                          'keremberke/yolov8m-license-plate'):
                try:
                    plate_model = _try_load_plate_model(_repo)
                    print(f'      Plate detector OK ({_repo})')
                    break
                except Exception as e:
                    print(f'      ⚠️ تعذّر تحميل {_repo}: {e}')
        if plate_model is None:
            print('      → كشف اللوحة بيعتمد على الترشيح المورفولوجي + تحقق OCR')
    except Exception as e:
        plate_model = None
        print(f'      ⚠️ تعذّر تهيئة كاشف اللوحات: {e}')
    try:
        import easyocr
        try:
            ocr_reader = easyocr.Reader(['ar', 'en'], gpu=True)
        except Exception:
            ocr_reader = easyocr.Reader(['ar', 'en'], gpu=False)
        print('      OCR OK (EasyOCR ar+en)')
    except Exception as e:
        ocr_reader = None
        print(f'      ⚠️ تعذّر تحميل OCR: {e}')
    # قارئ إنجليزي مستقل للصف اللاتيني — ترتيب يسار->يمين سليم وبدون تشويش الموديل العربي
    try:
        import easyocr
        try:
            ocr_reader_en = easyocr.Reader(['en'], gpu=True)
        except Exception:
            ocr_reader_en = easyocr.Reader(['en'], gpu=False)
        print('      OCR OK (EasyOCR en - الصف اللاتيني)')
    except Exception as e:
        ocr_reader_en = None
        print(f'      ⚠️ تعذّر تحميل القارئ الإنجليزي: {e}')

    # موديل الحروف المدرّب (اختياري) — لو موجود يصير الأساس والـ OCR احتياط
    try:
        if os.path.exists('models/plate_chars.pt'):
            from ultralytics import YOLO as _CharYOLO
            char_model = _CharYOLO('models/plate_chars.pt')
            char_model.predict(np.zeros((64, 64, 3), dtype=np.uint8), verbose=False)
            CHAR_CANON = _build_char_canon(char_model.names)
            print('      Char reader OK (موديل الحروف المدرّب ✨)')
    except Exception as e:
        char_model = None
        print(f'      ⚠️ تعذّر تحميل موديل الحروف المدرّب: {e}')

    lpr_ready = ocr_reader is not None or ocr_reader_en is not None or char_model is not None


_AR_TO_EN_DIGITS = str.maketrans('٠١٢٣٤٥٦٧٨٩', '0123456789')

def _norm_plate_key(text):
    """مفتاح موحّد للمطابقة: حروف لاتينية كبيرة + أرقام غربية فقط."""
    t = text.translate(_AR_TO_EN_DIGITS).upper()
    return ''.join(ch for ch in t if ch.isascii() and ch.isalnum())


# أحرف اللوحات السعودية المسموحة في OCR (يقلل ضوضاء القراءة مثل العلامات المائية)
PLATE_LAT_SET = set('ABDEGHJKLNRSTUVXZ')
PLATE_ALLOWLIST = '0123456789٠١٢٣٤٥٦٧٨٩ABDEGHJKLNRSTUVXZابحدرسصطعقكلمنهوى '
PLATE_LATIN_ALLOWLIST = '0123456789ABDEGHJKLNRSTUVXZ '
PLATE_AR_ALLOWLIST = 'ابحدرسصطعقكلمنهوى٠١٢٣٤٥٦٧٨٩ '


def _upscale_for_ocr(img, min_w=240):
    """الـ OCR ضعيف مع القصاصات الصغيرة — نكبّرها قبل القراءة."""
    h, w = img.shape[:2]
    if w and w < min_w:
        s = min(4.0, min_w / float(w))
        img = cv2.resize(img, None, fx=s, fy=s, interpolation=cv2.INTER_CUBIC)
    return img


def find_plate_candidates(img_bgr, max_cands=4):
    """مواقع مرشّحة للوحة بمورفولوجيا blackhat (تبرز نص داكن على خلفية فاتحة).
    الترشيح مبدئي فقط — الاعتماد النهائي يكون بنجاح قراءة OCR فعلية من المرشّح."""
    try:
        h, w = img_bgr.shape[:2]
        gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        kern = cv2.getStructuringElement(cv2.MORPH_RECT, (13, 5))
        blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kern)
        grad = np.absolute(cv2.Sobel(blackhat, cv2.CV_32F, 1, 0, ksize=-1))
        mn, mx = float(grad.min()), float(grad.max())
        grad = (255 * ((grad - mn) / (mx - mn + 1e-6))).astype('uint8')
        grad = cv2.GaussianBlur(grad, (5, 5), 0)
        grad = cv2.morphologyEx(grad, cv2.MORPH_CLOSE, kern)
        thresh = cv2.threshold(grad, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)[1]
        thresh = cv2.dilate(thresh, None, iterations=2)
        thresh = cv2.erode(thresh, None, iterations=1)
        cnts, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cands = []
        for c in sorted(cnts, key=cv2.contourArea, reverse=True)[:15]:
            x, y, bw, bh = cv2.boundingRect(c)
            area_r = (bw * bh) / float(w * h)
            ratio = bw / float(bh + 1e-6)
            if not (0.0002 <= area_r <= 0.06):
                continue                          # اللوحة صغيرة نسبياً في الصورة
            if not (1.2 <= ratio <= 4.5):
                continue                          # نسبة أبعاد اللوحة السعودية ~2:1
            if y < h * 0.2:
                continue                          # اللوحات ما تكون بأعلى الصورة
            roi = gray[y:y + bh, x:x + bw]
            if roi.size == 0 or float(roi.mean()) < 70:
                continue                          # اللوحة فاتحة (بيضاء/عاكسة)
            score = (bw * bh) * (1.5 - abs(ratio - 2.0) / 2.0)
            cands.append((score, (x, y, x + bw, y + bh)))
        cands.sort(reverse=True)
        return [b for _s, b in cands[:max_cands]]
    except Exception:
        return []


def _ocr_parse(image, allowlist=PLATE_ALLOWLIST):
    """يقرأ النص ويستخرج (حروف لاتينية، أرقام، ثقة، نص خام) حسب نظام اللوحات السعودي.
    نعالج كل صندوق نص لوحده عشان ما تلتصق أرقام الصفين العربي والإنجليزي ببعض."""
    import re
    try:
        out = ocr_reader.readtext(image, allowlist=allowlist)
    except Exception:
        out = ocr_reader.readtext(image)
    if not out:
        return [], '', 0.0, ''
    raw_parts, lat_cands, ar_cands, digit_cands, long_runs = [], [], [], [], []
    for bbox, txt, conf in out:
        txt = str(txt)
        raw_parts.append(txt)
        x_left = min(p[0] for p in bbox)
        norm = txt.translate(_AR_TO_EN_DIGITS)
        for run in re.findall(r'\d+', norm):
            if 1 <= len(run) <= 4:
                digit_cands.append((len(run) == 4, len(run), float(conf), run))
            else:
                long_runs.append((float(conf), run))
        lat = [ch for ch in norm.upper() if ch in PLATE_LAT_SET]
        # شريط KSA الجانبي في اللوحة مو من حروف التسجيل
        if 1 <= len(lat) <= 3 and ''.join(lat) != 'KSA':
            lat_cands.append((len(lat), float(conf), x_left, lat))
        ar = [ch for ch in txt if ch in AR2LAT]
        if 1 <= len(ar) <= 3:
            ar_cands.append((len(ar), float(conf), x_left, ar))
    confs = [float(c) for _b, _t, c in out]
    ocr_conf = float(np.mean(confs)) if confs else 0.0
    # الأرقام: نفضّل مجموعة كاملة من 4 أرقام، ثم الأطول، ثم الأعلى ثقة
    digits = ''
    if digit_cands:
        digit_cands.sort(key=lambda t: (t[0], t[1], t[2]), reverse=True)
        digits = digit_cands[0][3]
    elif long_runs:
        long_runs.sort(reverse=True)
        digits = long_runs[0][1][:4]
    # الحروف: نفضّل الصف اللاتيني (أدق قراءةً) ونشتق العربي منه
    lat_letters = []
    if lat_cands:
        lat_cands.sort(key=lambda t: t[2])        # صناديق الصف اللاتيني من اليسار لليمين
        lat_letters = [ch for _n, _c, _x, chs in lat_cands for ch in chs][:3]
    elif ar_cands:
        ar_cands.sort(key=lambda t: -t[2])        # الصف العربي يُقرأ من اليمين لليسار
        merged = [ch for _n, _c, _x, chs in ar_cands for ch in chs][:3]
        lat_letters = [AR2LAT[c] for c in reversed(merged)]
    return lat_letters, digits, ocr_conf, ' '.join(raw_parts)


# تصحيح الالتباسات الشائعة حسب موقع الخانة في الصف اللاتيني
_DIGIT_FIX = {'O': '0', 'Q': '0', 'I': '1', 'B': '8', 'Z': '2', 'S': '5',
              'G': '6', 'T': '7', 'A': '4', 'L': '1', 'D': '0', 'J': '3'}
_LETTER_FIX = {'0': 'D', '8': 'B', '2': 'Z', '5': 'S', '7': 'T',
               '6': 'G', '4': 'A', '1': 'L', '3': 'J'}


def _enhance_plate(img):
    """تحسين قصاصة اللوحة قبل القراءة: تكبير حسب الارتفاع + رفع التباين (CLAHE)."""
    h, w = img.shape[:2]
    target_h = 140
    if h and h < target_h:
        s = min(6.0, target_h / float(h))
        img = cv2.resize(img, None, fx=s, fy=s, interpolation=cv2.INTER_CUBIC)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8)).apply(gray)
    return cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)


def _binarize_plate(img):
    """معالجة بديلة: أبيض/أسود (Otsu) — أحياناً أنجح مع خطوط اللوحات."""
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    g = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)[1]
    return cv2.cvtColor(g, cv2.COLOR_GRAY2BGR)


def _parse_latin_row(ocr_out):
    """يستخرج نمط اللوحة السعودية (حتى 4 أرقام يليها 3 حروف) بالبحث داخل
    السلسلة — أي رموز شاردة قبل النمط أو بعده (شريط KSA، النقطة الفاصلة،
    شوائب قراءة) تُتجاهل بدل ما تزحزح خانات الحروف والأرقام."""
    import re
    if not ocr_out:
        return [], '', 0.0, ''
    toks = []
    for bbox, txt, conf in ocr_out:
        t = ''.join(ch for ch in str(txt).upper() if ch.isalnum())
        if t:
            toks.append((min(p[0] for p in bbox), t, float(conf)))
    if not toks:
        return [], '', 0.0, ''
    toks.sort()                                   # ترتيب الصناديق يسار -> يمين
    seq = ''.join(t for _x, t, _c in toks)
    raw = ' '.join(t for _x, t, _c in toks)
    conf = float(np.mean([c for _x, _t, c in toks]))
    if 'KSA' in seq and len(seq) > 6:
        seq = seq.replace('KSA', '', 1)           # شريط KSA الجانبي مو من حروف التسجيل
    if len(seq) < 4:
        return [], '', conf, raw

    # (1) تمرير صارم: أرقام يليها 3 حروف صافية — نفضّل الأرقام الأطول
    best = None
    for m in re.finditer(r'(\d{1,4})([A-Z]{3})', seq):
        score = (len(m.group(1)), m.start())
        if best is None or score > best[0]:
            best = (score, m.group(1), list(m.group(2)))
    if best:
        return best[2], best[1], conf, raw

    # (2) تمرير متسامح: خانة الحروف قد تنقرأ أرقاماً مشابهة (B↔8، D↔0، G↔6...)
    best = None
    for m in re.finditer(r'(\d{1,4})([A-Z0-9]{3})', seq):
        g2 = m.group(2)
        n_real = sum(c in PLATE_LAT_SET for c in g2)
        if n_real == 0:
            continue
        fixed, ok = [], True
        for c in g2:
            if c in PLATE_LAT_SET:
                fixed.append(c)
            elif c in _LETTER_FIX:
                fixed.append(_LETTER_FIX[c])
            else:
                ok = False
                break
        if not ok:
            continue
        score = (n_real, len(m.group(1)), m.start())
        if best is None or score > best[0]:
            best = (score, m.group(1), fixed)
    if best:
        return best[2], best[1], conf, raw

    # (3) ما لقينا نمط اللوحة — نرجّع أطول مجموعة أرقام والصف العربي يكمّل الحروف
    runs = re.findall(r'\d{1,4}', seq)
    digits = max(runs, key=len) if runs else ''
    return [], digits, conf, raw


def _parse_arabic_row(ocr_out):
    """يقرأ الصف العربي العلوي: حروف عربية (يمين->يسار) + أرقام هندية،
    ويحوّلها للصيغة اللاتينية المطبوعة حسب نظام المرور السعودي."""
    import re
    if not ocr_out:
        return [], '', 0.0, ''
    letter_boxes, digit_cands = [], []
    raws, confs = [], []
    for bbox, txt, conf in ocr_out:
        txt = str(txt)
        raws.append(txt)
        confs.append(float(conf))
        x_left = min(p[0] for p in bbox)
        ar = [ch for ch in txt if ch in AR2LAT]
        if 1 <= len(ar) <= 3:
            letter_boxes.append((x_left, ar))
        norm = txt.translate(_AR_TO_EN_DIGITS)
        for run in re.findall(r'\d+', norm):
            if 1 <= len(run) <= 4:
                digit_cands.append((len(run) == 4, len(run), float(conf), run))
    letter_boxes.sort(key=lambda t: -t[0])        # العربي يُقرأ من اليمين لليسار
    merged = [ch for _x, chs in letter_boxes for ch in chs][:3]
    lat = [AR2LAT[c] for c in reversed(merged)]   # الترتيب اللاتيني المطبوع = معكوس العربي
    digits = ''
    if digit_cands:
        digit_cands.sort(reverse=True)
        digits = digit_cands[0][3]
    conf = float(np.mean(confs)) if confs else 0.0
    return lat, digits, conf, ' '.join(raws)


def _fuse_plate(en_lat, en_dig, en_conf, ar_lat, ar_dig, ar_conf):
    """دمج المصادر الأربعة: الصفان يحملان نفس التسجيل — نكمل النقص من الصف
    الثاني، وعند التعارض نرجّح الأعلى ثقة."""
    notes = []
    # ---- الأرقام ----
    if en_dig and en_dig == ar_dig:
        digits = en_dig
        notes.append('أرقام: الصفان متطابقان ✓')
    elif len(en_dig) == 4 and len(ar_dig) != 4:
        digits = en_dig
    elif len(ar_dig) == 4 and len(en_dig) != 4:
        digits = ar_dig
        notes.append('أرقام: من الصف العربي')
    elif en_dig and ar_dig:
        if en_dig in ar_dig or ar_dig in en_dig:
            digits = en_dig if len(en_dig) >= len(ar_dig) else ar_dig
            notes.append('أرقام: أخذنا القراءة الأكمل')
        else:
            digits = en_dig if en_conf >= ar_conf else ar_dig
            notes.append('أرقام: تعارض — رجّحنا الأعلى ثقة')
    else:
        digits = en_dig or ar_dig
    # ---- الحروف (كلاهما بالصيغة اللاتينية المطبوعة) ----
    if len(en_lat) == 3 and en_lat == ar_lat:
        letters = en_lat
        notes.append('حروف: الصفان متطابقان ✓')
    elif len(en_lat) == 3 and len(ar_lat) == 3:
        letters = en_lat if en_conf >= ar_conf else ar_lat
        notes.append('حروف: تعارض — رجّحنا الأعلى ثقة')
    elif len(en_lat) == 3:
        letters = en_lat
    elif len(ar_lat) == 3:
        letters = ar_lat
        notes.append('حروف: من الصف العربي')
    else:
        letters = en_lat if len(en_lat) >= len(ar_lat) else ar_lat
    return letters, digits, notes


def _build_char_canon(names):
    """يبني خريطة id->رمز قانوني (حرف لاتيني من الـ17 أو رقم 0-9) من أسماء أصناف
    الموديل المدرّب — مهما كانت طريقة التسمية: لاتيني / عربي / كلمة إنجليزية / رقم
    غربي أو شرقي. يطبع الخريطة عند التحميل عشان نراجعها (الخطوة الوحيدة غير القابلة
    للتحقق عن بُعد)."""
    import unicodedata, re as _re

    def _norm_ar(ch):
        ch = unicodedata.normalize('NFKC', ch).replace('\u0640', '')   # نزع التطويل
        for a in 'أإآٱ':
            ch = ch.replace(a, 'ا')
        ch = ch.replace('ة', 'ه').replace('ي', 'ى').replace('ئ', 'ى')   # توحيد الهاء/الياء
        return ch

    ar2lat_n = {}
    for ar, en in PLATE_LETTERS:
        ar2lat_n[_norm_ar(ar)] = en

    eastern = '٠١٢٣٤٥٦٧٨٩'
    persian = '۰۱۲۳۴۵۶۷۸۹'
    words = {'ZERO': '0', 'ONE': '1', 'TWO': '2', 'THREE': '3', 'FOUR': '4', 'FIVE': '5',
             'SIX': '6', 'SEVEN': '7', 'EIGHT': '8', 'NINE': '9',
             'ALEF': 'A', 'ALIF': 'A', 'BAA': 'B', 'BA': 'B', 'HHA': 'J', 'HAA': 'J',
             'DAAL': 'D', 'DAL': 'D', 'RAA': 'R', 'RA': 'R', 'SEEN': 'S', 'SIN': 'S',
             'SAAD': 'X', 'SAD': 'X', 'TAA': 'T', 'TA': 'T', 'AYN': 'E', 'AIN': 'E',
             'QAAF': 'G', 'QAF': 'G', 'GAF': 'G', 'KAAF': 'K', 'KAF': 'K', 'LAAM': 'L', 'LAM': 'L',
             'MEEM': 'Z', 'MIM': 'Z', 'NOON': 'N', 'NUN': 'N', 'HEH': 'H', 'HA': 'H',
             'WAW': 'U', 'WAAW': 'U', 'YAA': 'V', 'ALEFMAQSURA': 'V'}
    LETTER_SET = set('ABDEGHJKLNRSTUVXZ')

    def canon_one(s):
        s = str(s).strip()
        if not s:
            return None
        base = _re.sub(r'^(class|char|letter|digit)[_\-\s]*', '', s, flags=_re.I)
        u = unicodedata.normalize('NFKC', base).strip().upper()
        u_alnum = ''.join(c for c in u if c.isalnum())
        if len(u_alnum) == 1 and u_alnum.isdigit():           # رقم غربي
            return u_alnum
        if len(u_alnum) == 1 and u_alnum in LETTER_SET:       # حرف لاتيني من الـ17 (نثق به حرفياً)
            return u_alnum
        for ch in base:                                       # رقم شرقي/فارسي
            if ch in eastern:
                return str(eastern.index(ch))
            if ch in persian:
                return str(persian.index(ch))
        for ch in base:                                       # حرف عربي
            n = _norm_ar(ch)
            if n in ar2lat_n:
                return ar2lat_n[n]
        key = ''.join(c for c in u if c.isalpha())            # كلمة إنجليزية منطوقة
        if key in words:
            return words[key]
        return None

    canon, unresolved = {}, []
    for cid, nm in names.items():
        c = canon_one(nm)
        canon[int(cid)] = c
        if c is None:
            unresolved.append((cid, nm))

    print('      خريطة أصناف موديل الحروف (id → رمز):')
    for cid in sorted(canon):
        print('        %s: %r → %s' % (cid, names[cid], canon[cid]))
    if unresolved:
        print('      ⚠️ أصناف غير معروفة (تُتجاهل في القراءة): %s' % unresolved)
    bad = set(v for v in canon.values() if v is not None) - (LETTER_SET | set('0123456789'))
    if bad:
        print('      ⚠️ رموز خارج نظام اللوحة السعودية: %s' % bad)
    return canon


def _chars_on_crop(crop):
    """يقرأ الحروف والأرقام بموديل الحروف المدرّب (كشف كل رمز كصندوق).
    يرجّع (حروف لاتينية بترتيب القراءة، أرقام غربية، ثقة، نص خام) — أو فاضي عند الفشل.
    المبدأ: نرتّب كل صف يسار→يمين (مستقل عن اللغة) ونحوّل كل رمز لهويته القانونية،
    فالصفان العربي واللاتيني يعطيان نفس التسلسل ونأخذ الأكمل/الأعلى ثقة. الأرقام
    لا تُعكس أبداً (تُطبع بنفس الاتجاه على الصفين)."""
    if char_model is None:
        return [], '', 0.0, ''
    try:
        r = char_model.predict(crop, conf=0.20, iou=0.45, imgsz=1280, verbose=False)[0]
    except Exception:
        return [], '', 0.0, ''
    boxes = []
    for b in r.boxes:
        canon = CHAR_CANON.get(int(b.cls[0]))
        if canon is None:
            continue
        x1, y1, x2, y2 = b.xyxy[0].tolist()
        boxes.append({'canon': canon, 'conf': float(b.conf[0]),
                      'cx': (x1 + x2) / 2.0, 'cy': (y1 + y2) / 2.0,
                      'w': x2 - x1, 'h': y2 - y1,
                      'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2})
    if not boxes:
        return [], '', 0.0, ''

    # NMS صنفي-محايد: نفس الرمز الفيزيائي = صندوق واحد (الأعلى ثقة)
    boxes.sort(key=lambda d: -d['conf'])
    kept = []
    for b in boxes:
        dup = False
        for k in kept:
            iw = max(0.0, min(b['x2'], k['x2']) - max(b['x1'], k['x1']))
            ih = max(0.0, min(b['y2'], k['y2']) - max(b['y1'], k['y1']))
            inter = iw * ih
            ua = b['w'] * b['h'] + k['w'] * k['h'] - inter
            if ua > 0 and inter / ua > 0.5:
                dup = True
                break
        if not dup:
            kept.append(b)
    boxes = kept

    h_med = float(np.median([b['h'] for b in boxes])) or 1.0

    # تقسيم لصفوف بأكبر فجوة عمودية (cy) — صف واحد للقصاصة المفردة
    boxes_y = sorted(boxes, key=lambda d: d['cy'])
    rows = [[boxes_y[0]]]
    for b in boxes_y[1:]:
        prev_mean = sum(x['cy'] for x in rows[-1]) / len(rows[-1])
        if b['cy'] - prev_mean > 0.6 * h_med:
            rows.append([b])
        else:
            rows[-1].append(b)
    if len(rows) > 2:                       # شوائب: نأخذ أكبر صفين
        rows = sorted(rows, key=len, reverse=True)[:2]

    # كل صف يُقرأ يسار→يمين. الهوية القانونية مستقلة عن اللغة، والحرف العربي يقع
    # فيزيائياً تحت/فوق مقابله اللاتيني بنفس الموضع — فالصفان يعطيان نفس التسلسل
    # القانوني بدون أي عكس (العكس يكون فقط عند توليد نص العربي للعرض لاحقاً).
    def read_row(row):
        row = sorted(row, key=lambda d: d['cx'])      # يسار → يمين
        letters = [(b['canon'], b['conf']) for b in row if b['canon'] in PLATE_LAT_SET]
        digits = [(b['canon'], b['conf']) for b in row if b['canon'].isdigit()]
        return letters[:3], digits[:4]

    parsed = [read_row(rw) for rw in rows]

    def pick(groups):
        groups = [g for g in groups if g]
        if not groups:
            return []
        if len(groups) == 2 and len(groups[0]) == len(groups[1]):
            return [a if a[1] >= b[1] else b for a, b in zip(groups[0], groups[1])]
        groups.sort(key=lambda g: (len(g), sum(c for _t, c in g) / max(1, len(g))), reverse=True)
        return groups[0]

    best_letters = pick([p[0] for p in parsed])
    best_digits = pick([p[1] for p in parsed])
    lat = [t for t, _c in best_letters]
    dig = ''.join(t for t, _c in best_digits)
    if not lat and not dig:
        return [], '', 0.0, ''
    confs = [c for _t, c in best_letters] + [c for _t, c in best_digits]
    conf = float(np.mean(confs)) if confs else 0.0
    raw = ''.join(lat) + ' ' + dig
    return lat, dig, conf, raw


def _easyocr_read(crop):
    """قراءة المرشّح بمسار EasyOCR (الصف اللاتيني + العربي ثم الدمج) — منطق Final حرفياً.
    يرجّع (f_lat, f_dig, en_conf, ar_conf, rw_en, rw_ar, notes)."""
    whole = _enhance_plate(crop)
    quick_reader = ocr_reader_en or ocr_reader
    if quick_reader is None:
        return [], '', 0.0, 0.0, '', '', []
    try:
        quick = quick_reader.readtext(whole, allowlist=PLATE_LATIN_ALLOWLIST)
    except Exception:
        quick = []
    n_chars = sum(len([c for c in str(t) if c.isalnum()]) for _b, t, _c in quick)
    if n_chars < 3:
        return [], '', 0.0, 0.0, '', '', []
    en_lat, en_dig, en_conf, rw_en = _parse_latin_row(quick)
    best_score = (len(en_lat) == 3) * 4 + (len(en_dig) == 4) * 3 + len(en_dig) + len(en_lat)
    if ocr_reader_en is not None and not (len(en_lat) == 3 and len(en_dig) == 4):
        ch_ = crop.shape[0]
        bottom = crop[int(ch_ * 0.42):, :]
        for img_try in (_enhance_plate(bottom), _binarize_plate(_enhance_plate(bottom))):
            try:
                out = ocr_reader_en.readtext(img_try, allowlist=PLATE_LATIN_ALLOWLIST)
            except Exception:
                continue
            lat_t, dig_t, conf_t, raw_t = _parse_latin_row(out)
            sc = (len(lat_t) == 3) * 4 + (len(dig_t) == 4) * 3 + len(dig_t) + len(lat_t)
            if sc > best_score:
                best_score = sc
                en_lat, en_dig, en_conf, rw_en = lat_t, dig_t, conf_t, raw_t
            if len(lat_t) == 3 and len(dig_t) == 4:
                break
    ar_lat, ar_dig, ar_conf, rw_ar = [], '', 0.0, ''
    if ocr_reader is not None:
        top = crop[:int(crop.shape[0] * 0.62), :]
        try:
            ar_out = ocr_reader.readtext(_enhance_plate(top), allowlist=PLATE_AR_ALLOWLIST)
        except Exception:
            ar_out = []
        ar_lat, ar_dig, ar_conf, rw_ar = _parse_arabic_row(ar_out)
        if len(ar_dig) < 4 or len(ar_lat) < 3:
            try:
                ar_out2 = ocr_reader.readtext(_binarize_plate(_enhance_plate(top)),
                                              allowlist=PLATE_AR_ALLOWLIST)
            except Exception:
                ar_out2 = []
            ar_lat2, ar_dig2, ar_conf2, rw_ar2 = _parse_arabic_row(ar_out2)
            if len(ar_dig2) > len(ar_dig):
                ar_dig = ar_dig2
                ar_conf = max(ar_conf, ar_conf2)
            if len(ar_lat2) > len(ar_lat):
                ar_lat = ar_lat2
                ar_conf = max(ar_conf, ar_conf2)
            if rw_ar2:
                rw_ar = (rw_ar + ' / ' + rw_ar2).strip(' /')
    f_lat, f_dig, notes = _fuse_plate(en_lat, en_dig, en_conf, ar_lat, ar_dig, ar_conf)
    return f_lat, f_dig, en_conf, ar_conf, rw_en, rw_ar, notes


def _read_candidate_crop(crop):
    """منسّق القراءة يحترم PRIMARY_ENGINE (easyocr/chars/hybrid)، استدعاء كسول للموديلين.
    يرجّع (lat, dig, conf, raw_en, raw_ar, notes, engine)."""
    _cache = {}
    def _ez():
        if 'v' not in _cache:
            _cache['v'] = _easyocr_read(crop)
        return _cache['v']
    def _ch():
        if char_model is None:
            return [], '', 0.0, ''
        return _chars_on_crop(crop)

    if PRIMARY_ENGINE == 'chars':
        c_lat, c_dig, c_conf, c_raw = _ch()
        if c_lat and c_dig:
            return c_lat, c_dig, c_conf, c_raw, '', ['موديل الحروف المدرّب ✨'], 'chars'
        f_lat, f_dig, en_c, ar_c, rw_en, rw_ar, notes = _ez()
        return f_lat, f_dig, max(en_c, ar_c), rw_en, rw_ar, notes, 'easyocr'

    if PRIMARY_ENGINE == 'hybrid':
        f_lat, f_dig, en_c, ar_c, rw_en, rw_ar, notes = _ez()
        c_lat, c_dig, c_conf, c_raw = _ch()
        dig = c_dig if len(c_dig) in (3, 4) else f_dig
        lat = f_lat if len(f_lat) == 3 else c_lat
        h_notes = list(notes) + ['هجين: أرقام=الموديل، حروف=EasyOCR 🔀']
        return lat, dig, max(en_c, ar_c, c_conf), rw_en, rw_ar, h_notes, 'hybrid'

    # 'easyocr' (افتراضي): EasyOCR أساس، موديل الحروف احتياط لو EasyOCR ناقص
    f_lat, f_dig, en_c, ar_c, rw_en, rw_ar, notes = _ez()
    if len(f_lat) == 3 and len(f_dig) in (3, 4):
        return f_lat, f_dig, max(en_c, ar_c), rw_en, rw_ar, notes, 'easyocr'
    c_lat, c_dig, c_conf, c_raw = _ch()
    if c_lat and c_dig:
        return c_lat, c_dig, c_conf, c_raw, '', ['موديل الحروف (احتياط) ✨'], 'easyocr_fallback'
    if f_lat or f_dig:
        return f_lat, f_dig, max(en_c, ar_c), rw_en, rw_ar, notes, 'easyocr'
    return c_lat, c_dig, c_conf, c_raw, '', ['موديل الحروف (احتياط) ✨'], 'easyocr_fallback'


def read_plate_from_image(img_bgr):
    """يرشّح مواقع اللوحة ويقرأ كل مرشّح من المصدرين: الصف اللاتيني السفلي
    (قارئ إنجليزي، عدة معالجات) + الصف العربي العلوي (قارئ عربي)، ثم يدمجهما."""
    if ocr_reader is None and ocr_reader_en is None:
        return {'status': 'error', 'message': 'نظام قراءة اللوحات غير محمّل'}

    annotated = img_bgr.copy()
    h, w = img_bgr.shape[:2]
    candidates = []   # (method, det_conf, box)

    if plate_model is not None:
        try:
            results = plate_model.predict(img_bgr, conf=0.25, iou=0.45, max_det=5, verbose=False)
            boxes = []
            for rr in results:
                for b in rr.boxes:
                    x1, y1, x2, y2 = map(int, b.xyxy[0].tolist())
                    boxes.append((float(b.conf[0]), (x1, y1, x2, y2)))
            boxes.sort(reverse=True)
            candidates += [('yolo', c, box) for c, box in boxes[:2]]
        except Exception as e:
            print('plate detect error:', e)

    candidates += [('opencv', 0.0, box) for box in find_plate_candidates(img_bgr)]

    # لو الصورة نفسها لوحة مقصوصة (صغيرة وعريضة مثل داتاسيتات اللوحات) نقرأها
    # كاملة *أولاً* — إعادة قصّها بـ YOLO تجي أضيق من اللوحة وتقطع أطراف الأرقام
    img_ratio = w / float(h + 1e-6)
    if 1.6 <= img_ratio <= 5.0 and max(h, w) <= 700:
        candidates.insert(0, ('direct', 0.0, (0, 0, w, h)))
    elif 1.4 <= img_ratio <= 5.0:
        candidates.append(('direct', 0.0, (0, 0, w, h)))

    lat_letters, digits_en, ocr_conf = [], '', 0.0
    raw_en, raw_ar, fuse_notes = '', '', []
    detect_method, det_conf, used_box = 'none', 0.0, None
    engine = 'easyocr'
    for method, dconf, (x1, y1, x2, y2) in candidates:
        # هامش أوسع حول المرشّح (القص الضيق يضعف الـ OCR)
        px = int((x2 - x1) * 0.14) + 4
        py = int((y2 - y1) * 0.25) + 4
        crop = img_bgr[max(0, y1 - py):min(h, y2 + py), max(0, x1 - px):min(w, x2 + px)]
        if crop.size == 0:
            continue

        # القراءة عبر المنسّق القابل للضبط (يحترم PRIMARY_ENGINE) — كل المسارات محفوظة داخله
        r_lat, r_dig, r_conf, r_rawen, r_rawar, r_notes, r_engine = _read_candidate_crop(crop)
        if len(r_dig) >= 2 or (r_lat and r_dig):
            lat_letters, digits_en, ocr_conf = r_lat, r_dig, r_conf
            raw_en, raw_ar, fuse_notes = r_rawen, r_rawar, r_notes
            detect_method, det_conf, used_box = method, dconf, (x1, y1, x2, y2)
            engine = r_engine
            break

    if used_box:
        color = (113, 204, 46) if detect_method == 'yolo' else (0, 165, 255)
        cv2.rectangle(annotated, (used_box[0], used_box[1]), (used_box[2], used_box[3]), color, 3)
    elif ocr_reader is not None:
        # ما نجح أي مرشّح → قراءة أخيرة على الصورة كاملة
        lat_letters, digits_en, ocr_conf, raw_en = _ocr_parse(img_bgr)

    if not lat_letters and not digits_en:
        return {'status': 'no_text', 'message': 'تعذّر استخراج رقم لوحة صالح'}

    # نظام اللوحات السعودي: ترتيب الحروف الإنجليزية على اللوحة معكوس عن العربية
    ar_letters = [LAT2AR[c] for c in reversed(lat_letters)]
    key = ''.join(lat_letters) + digits_en
    plate_en = (' '.join(lat_letters) + '  ' + digits_en).strip()
    plate_ar = (' '.join(ar_letters) + '  ' + to_ar_digits(digits_en)).strip()

    ok, buf = cv2.imencode('.jpg', annotated)
    annotated_b64 = base64.b64encode(buf).decode('utf-8') if ok else None

    return {
        'status': 'ok',
        'plate_key': key,
        'plate_ar': plate_ar or '—',
        'plate_en': plate_en or '—',
        'raw': (('EN: ' + raw_en) if raw_en else '') + ((' | AR: ' + raw_ar) if raw_ar else ''),
        'raw_en': raw_en,
        'raw_ar': raw_ar,
        'fusion_notes': fuse_notes,
        'detect_conf': round(float(det_conf), 3),
        'ocr_conf': round(float(ocr_conf), 3),
        'detect_method': detect_method,
        'detected_by_yolo': detect_method == 'yolo',
        'engine': engine,
        'complete': bool(len(lat_letters) == 3 and 3 <= len(digits_en) <= 4),
        'annotated_image': annotated_b64
    }


def process_vehicle(plate_key, info):
    """آلة حالة دخول/خروج مستقلة لكل لوحة (غير مربوطة بشخص)."""
    global vehicle_state, vehicle_logs
    ts = now_sa().strftime('%Y-%m-%d %H:%M:%S')
    state = vehicle_state.get(plate_key, 'outside')
    if state == 'outside':
        event = 'ENTRY'
        vehicle_state[plate_key] = 'inside'
    else:
        event = 'EXIT'
        vehicle_state[plate_key] = 'outside'
    vehicle_logs.append({
        'timestamp': ts, 'plate_key': plate_key,
        'plate_ar': info.get('plate_ar', ''), 'plate_en': info.get('plate_en', ''),
        'event': event
    })
    with open(VEHICLE_LOGS_PATH, 'w', encoding='utf-8') as f:
        json.dump({'state': vehicle_state, 'logs': vehicle_logs}, f, ensure_ascii=False, indent=2)
    inside_count = sum(1 for v in vehicle_state.values() if v == 'inside')
    return event, ts, inside_count


def trigger_gate(action='open'):
    """يفتح/يغلق البوابة الفعلية: عبر Raspberry Pi (HTTP) أو GPIO محلي إن توفّر."""
    # (1) إرسال إشارة لجهاز Raspberry Pi على الشبكة
    if GATE_PI_URL:
        try:
            import requests as _rq
            _rq.post(GATE_PI_URL, json={'action': action}, timeout=2)
            return True
        except Exception as e:
            print('gate Pi error:', e)
    # (2) لو النظام نفسه يعمل على Raspberry Pi (GPIO محلي)
    try:
        import RPi.GPIO as GPIO
        import time as _t
        SERVO_PIN = 18
        GPIO.setmode(GPIO.BCM)
        GPIO.setup(SERVO_PIN, GPIO.OUT)
        pwm = GPIO.PWM(SERVO_PIN, 50)
        pwm.start(0)
        pwm.ChangeDutyCycle(7.5 if action == 'open' else 2.5)  # ~90 درجة فتح / 0 إغلاق
        _t.sleep(1)
        pwm.stop()
        GPIO.cleanup(SERVO_PIN)
        return True
    except Exception:
        return False  # لا Pi ولا GPIO (مثل Colab) — نتجاهل بهدوء





@app.route('/')
def home():
    today = now_sa().strftime('%Y-%m-%d')
    today_logs = [l for l in access_logs if l.get('timestamp','').startswith(today)]
    stats = {
        'verifications': len(today_logs) if today_logs else 128,
        'vehicles': len(set(l.get('plate','') for l in today_logs if l.get('plate'))) if today_logs else 76
    }
    return render_template('home.html', active_page='home', stats=stats)

@app.route('/cameras')
def cameras():
    return render_template('cameras.html', active_page='cameras')

@app.route('/face-analysis')
def face_analysis():
    return render_template('face_analysis.html', active_page='face')

@app.route('/plate-reading')
def plate_reading():
    return render_template('plate_reading.html', active_page='plate')

@app.route('/cards')
def cards():
    return render_template('cards.html', active_page='cards')

@app.route('/reports')
def reports():
    return render_template('reports.html', active_page='reports')

@app.route('/enroll')
def enroll():
    return render_template('enroll.html', active_page='enroll')


# ============================================================
# API
# ============================================================
def _eye_open_ratio(lmk, idxs):
    # نسبة الارتفاع/العرض لمجموعة نقاط العين (تنخفض عند الرمشة)
    pts = lmk[idxs]
    xs, ys = pts[:, 0], pts[:, 1]
    w = float(xs.max() - xs.min()) + 1e-6
    h = float(ys.max() - ys.min()) + 1e-6
    return h / w


@app.route('/api/liveness', methods=['POST'])
def api_liveness():
    # كشف الحيوية النشط: حركة الرأس + الرمشة عبر سلسلة لقطات
    frames = request.files.getlist('frames')
    if len(frames) < 3:
        return jsonify({'status': 'error', 'message': 'لقطات غير كافية'}), 400

    nose_rel, ears = [], []
    faces_seen = 0
    for ff in frames:
        b = ff.read()
        try:
            pil = ImageOps.exif_transpose(Image.open(io.BytesIO(b)))
        except Exception:
            continue
        arr = np.array(pil)
        if len(arr.shape) == 2:
            bgr = cv2.cvtColor(arr, cv2.COLOR_GRAY2BGR)
        elif arr.shape[2] == 4:
            bgr = cv2.cvtColor(arr, cv2.COLOR_RGBA2BGR)
        else:
            bgr = cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)
        fs = face_app.get(bgr)
        if not fs:
            continue
        faces_seen += 1
        fc = max(fs, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))
        kps = fc.kps  # عين يمنى، عين يسرى، أنف، فم يمين، فم يسار
        eye_mid_x = (kps[0][0] + kps[1][0]) / 2.0
        inter = abs(kps[1][0] - kps[0][0]) + 1e-6
        nose_rel.append(float((kps[2][0] - eye_mid_x) / inter))
        lmk = getattr(fc, 'landmark_2d_106', None)
        if lmk is not None:
            try:
                lmk = np.array(lmk)
                ear = (_eye_open_ratio(lmk, list(range(33, 43))) +
                       _eye_open_ratio(lmk, list(range(87, 97)))) / 2.0
                ears.append(ear)
            except Exception:
                pass

    movement = float(max(nose_rel) - min(nose_rel)) if len(nose_rel) >= 2 else 0.0
    movement_detected = movement > 0.18
    blink_drop = 0.0
    blink_detected = False
    if len(ears) >= 3:
        blink_drop = float(max(ears) - min(ears))
        blink_detected = blink_drop > 0.12

    live = bool(movement_detected or blink_detected)
    return jsonify({
        'status': 'ok',
        'live': live,
        'movement': round(movement, 3),
        'movement_detected': bool(movement_detected),
        'blink_drop': round(blink_drop, 3),
        'blink_detected': bool(blink_detected),
        'frames_with_face': faces_seen,
        'frames_total': len(frames)
    })


@app.route('/api/recognize', methods=['POST'])
def api_recognize():
    if 'image' not in request.files:
        return jsonify({'status':'error','message':'لم يتم رفع صورة'}), 400

    image_bytes = request.files['image'].read()
    gate_zone = GATE_ZONE
    check_spoof = request.form.get('spoof', 'on') != 'off'
    recognition = recognize_face_from_bytes(image_bytes, check_spoof=check_spoof)
    timestamp = now_sa().strftime('%Y-%m-%d %H:%M:%S')
    num_faces = recognition.get('num_faces', 1)
    tailgating = False   # دائماً نركّز على الوجه الأقرب (السائق) — بدون تحذير تسلّل

    if recognition['status'] != 'recognized':
        # وسم إنجليزي ثابت للقرار عشان الإحصائيات والتصدير يتعرفون على الانتحال
        if recognition['status'] == 'spoof_detected':
            decision = '❌ DENIED - SPOOF'
        else:
            decision = '❌ DENIED - ' + recognition.get('message', 'Unknown')
        access_logs.append({
            'timestamp': timestamp,
            'recognition_status': recognition['status'],
            'score': recognition['score'],
            'tailgating': tailgating,
            'decision': decision
        })
        save_logs()
        return jsonify({**recognition, 'timestamp': timestamp, 'tailgating': tailgating})

    folder = recognition['folder']
    employee = employees_db.get(folder)
    if not employee:
        return jsonify({'status':'error','message':'لا توجد بيانات','score':recognition['score'],'timestamp':timestamp})

    # إقران سياقي: آخر لوحة انقرأت عند البوابة خلال النافذة (الوجه هو قرار الدخول)
    arrived = None
    if last_gate_plate:
        import time as _t
        _ago = _t.time() - last_gate_plate.get('at', 0)
        if 0 <= _ago <= PLATE_PAIR_WINDOW_SEC:
            arrived = {'plate_ar': last_gate_plate['plate_ar'],
                       'plate_en': last_gate_plate['plate_en'],
                       'event': last_gate_plate.get('event', ''),
                       'ago_sec': int(_ago)}

    emp_clr = employee.get('clearance', 'C')
    if employee['access_status'] != 'ACTIVE':
        decision = '❌ DENIED - Suspended'
    elif CLEARANCE_RANK.get(emp_clr, 1) < CLEARANCE_RANK.get(gate_zone, 1):
        decision = '❌ DENIED - CLEARANCE'
    else:
        decision = '✅ GRANTED'

    # فتح البوابة الفعلية عند السماح (Raspberry Pi) — بخيط منفصل عشان ما يعطّل الرد
    if 'GRANTED' in decision:
        threading.Thread(target=trigger_gate, args=('open',), daemon=True).start()

    access_logs.append({
        'timestamp': timestamp,
        'recognition_status': 'recognized',
        'score': recognition['score'],
        'national_id': employee['national_id'],
        'name_ar': employee['name_ar'],
        'nationality': employee['nationality_ar'],
        'rank': employee['rank_ar'],
        'plate': employee['vehicle']['plate_ar'],
        'vehicle': f"{employee['vehicle']['brand_ar']} {employee['vehicle']['color_ar']}",
        'arrived_plate_ar': arrived['plate_ar'] if arrived else '',
        'arrived_plate_en': arrived['plate_en'] if arrived else '',
        'clearance': emp_clr,
        'gate_zone': gate_zone,
        'tailgating': tailgating,
        'decision': decision
    })
    save_logs()

    return jsonify({
        'status':'recognized',
        'employee': employee,
        'score': recognition['score'],
        'timestamp': timestamp,
        'arrived_with': arrived,
        'gate_zone': gate_zone,
        'gate_zone_ar': CLEARANCE_AR.get(gate_zone, gate_zone),
        'tailgating': tailgating,
        'num_faces': num_faces,
        'decision': decision
    })


@app.route('/api/enroll', methods=['POST'])
def api_enroll():
    """تسجيل شخص جديد بعدة صور (نفس الشخص بأشكال مختلفة) ولوحته وبياناته"""
    # نقبل عدة صور: face_images[]
    face_files = request.files.getlist('face_images')
    if not face_files or all(f.filename == '' for f in face_files):
        return jsonify({'status':'error','message':'لم يتم رفع أي صورة وجه'}), 400

    embeddings = []
    genders = []
    ages = []
    saved_paths = []
    first_img_pil = None

    for idx, ffile in enumerate(face_files):
        if ffile.filename == '':
            continue
        face_bytes = ffile.read()
        img_pil = Image.open(io.BytesIO(face_bytes))
        img_pil = ImageOps.exif_transpose(img_pil)
        if first_img_pil is None:
            first_img_pil = img_pil.copy()

        img_np = np.array(img_pil)
        if len(img_np.shape) == 2:
            img_bgr = cv2.cvtColor(img_np, cv2.COLOR_GRAY2BGR)
        elif img_np.shape[2] == 4:
            img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGBA2BGR)
        else:
            img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)

        faces = face_app.get(img_bgr)
        if len(faces) == 0:
            # نتجاهل الصور اللي ما فيها وجه، ونكمل
            continue

        face = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))
        embeddings.append(face.embedding)
        g, a = detect_gender_age(face)
        genders.append(g)
        ages.append(a)

    if len(embeddings) == 0:
        return jsonify({'status':'error','message':'لم يتم اكتشاف وجه في أي من الصور المرفوعة'}), 400

    # متوسط البصمات = تمثيل قوي للشخص بكل أشكاله
    avg_embedding = np.mean(embeddings, axis=0)
    # الجنس: الأغلبية، العمر: المتوسط
    gender = max(set(genders), key=genders.count)
    age = int(np.mean(ages))

    # البيانات من الفورم
    name_ar = request.form.get('name_ar', '').strip()
    name_en = request.form.get('name_en', '').strip()
    nationality = request.form.get('nationality', 'saudi').strip()
    rank_ar = request.form.get('rank_ar', '').strip() or 'م5'
    clearance = request.form.get('clearance', 'A').strip().upper()
    if clearance not in CLEARANCE_RANK:
        clearance = 'A'
    national_id = request.form.get('national_id', '').strip()
    plate_ar = request.form.get('plate_ar', '').strip()
    brand_ar = request.form.get('brand_ar', '').strip() or 'تويوتا'
    color_ar = request.form.get('color_ar', '').strip() or 'أبيض'

    # نشتق اللوحة الإنجليزية من العربية (حسب جدول لوحات المرور السعودي)
    plate_en = ''
    if plate_ar:
        _ar = [c for c in plate_ar if c in AR2LAT]
        _letters = [AR2LAT[c] for c in reversed(_ar)]   # الترتيب الإنجليزي معكوس على اللوحة السعودية
        _digits = ''.join(str(AR_DIGITS.index(c)) if c in AR_DIGITS else c
                          for c in plate_ar if c.isdigit() or c in AR_DIGITS)
        if _letters or _digits:
            plate_en = (' '.join(_letters) + '  ' + _digits).strip()

    # معرّف فريد
    person_id = f"enrolled_{national_id or len(face_database)+1}"

    # حفظ الصورة الأولى كصورة مرجعية
    face_path = f"static/enrolled/{person_id}_face.jpg"
    if first_img_pil:
        first_img_pil.convert('RGB').save(face_path)

    # حفظ صورة اللوحة لو رُفعت
    plate_path = None
    if 'plate_image' in request.files and request.files['plate_image'].filename:
        plate_bytes = request.files['plate_image'].read()
        plate_path = f"{PLATES_DIR}/{person_id}_plate.jpg"
        Image.open(io.BytesIO(plate_bytes)).convert('RGB').save(plate_path)

    # نضيف للـ face_database (متوسط البصمات)
    face_database[person_id] = {
        'embedding': avg_embedding,
        'gender': gender,
        'age': age
    }
    _save_face_db()

    # نضيف لـ employees_db
    employees_db[person_id] = {
        'folder_name': person_id,
        'national_id': national_id or person_id,
        'id_type_ar': 'هوية وطنية' if nationality == 'saudi' else 'إقامة',
        'name_ar': name_ar or 'غير محدد',
        'name_en': name_en or 'Not Set',
        'gender': gender,
        'gender_ar': 'ذكر' if gender == 'male' else 'أنثى',
        'age': age,
        'nationality': nationality,
        'nationality_ar': NATIONALITY_AR.get(nationality, nationality),
        'rank_ar': rank_ar,
        'rank_en': rank_ar,
        'access_status': 'ACTIVE',
        'access_status_ar': 'نشط',
        'clearance': clearance,
        'clearance_ar': CLEARANCE_AR.get(clearance, 'ج'),
        'vehicle': {
            'plate_ar': plate_ar or 'غير محدد',
            'plate_en': plate_en,
            'brand_ar': brand_ar,
            'color_ar': color_ar,
            'plate_image': plate_path
        }
    }
    with open(EMPLOYEES_DB_PATH, 'w', encoding='utf-8') as f:
        json.dump(employees_db, f, ensure_ascii=False, indent=2)

    return jsonify({
        'status': 'success',
        'message': f'تم تسجيل {name_ar} بنجاح ({len(embeddings)} صورة)',
        'images_used': len(embeddings),
        'detected_gender': 'ذكر' if gender == 'male' else 'أنثى',
        'detected_age': age,
        'person_id': person_id
    })


@app.route('/api/logs')
def api_logs():
    return jsonify({'logs': access_logs})

@app.route('/api/employees')
def api_employees():
    return jsonify({'employees': list(employees_db.values())})


@app.route('/api/toggle-access/<person_id>', methods=['POST'])
def api_toggle_access(person_id):
    """تبديل حالة الشخص: نشط <-> موقوف (قائمة سوداء)"""
    if person_id not in employees_db:
        return jsonify({'status': 'error', 'message': 'الشخص غير موجود'}), 404

    current = employees_db[person_id].get('access_status', 'ACTIVE')
    new_status = 'SUSPENDED' if current == 'ACTIVE' else 'ACTIVE'
    employees_db[person_id]['access_status'] = new_status
    employees_db[person_id]['access_status_ar'] = 'موقوف' if new_status == 'SUSPENDED' else 'نشط'

    # نحفظ في الملف
    with open(EMPLOYEES_DB_PATH, 'w', encoding='utf-8') as f:
        json.dump(employees_db, f, ensure_ascii=False, indent=2)

    return jsonify({
        'status': 'success',
        'new_status': new_status,
        'new_status_ar': employees_db[person_id]['access_status_ar']
    })


@app.route('/api/export-logs')
def api_export_logs():
    """تصدير سجل الدخول كملف CSV"""
    from flask import Response
    import csv
    from io import StringIO

    output = StringIO()
    # نضيف BOM لدعم العربي في Excel
    output.write('\ufeff')
    writer = csv.writer(output)
    writer.writerow(['الوقت', 'الاسم', 'الهوية', 'الرتبة', 'الجنسية', 'مركبة الوصول', 'نسبة التطابق', 'القرار'])

    for log in access_logs:
        dec = log.get('decision', '')
        decision_ar = 'مسموح' if 'GRANTED' in dec else ('انتحال' if 'SPOOF' in dec else 'مرفوض')
        writer.writerow([
            log.get('timestamp', ''),
            log.get('name_ar', ''),
            log.get('national_id', ''),
            log.get('rank', ''),
            log.get('nationality', ''),
            log.get('arrived_plate_ar', ''),
            f"{log.get('score', 0)*100:.1f}%" if log.get('score') else '',
            decision_ar
        ])

    csv_data = output.getvalue()
    return Response(
        csv_data,
        mimetype='text/csv; charset=utf-8',
        headers={'Content-Disposition': f'attachment; filename=access_logs_{now_sa().strftime("%Y%m%d_%H%M%S")}.csv'}
    )


@app.route('/api/stats')
def api_stats():
    """إحصائيات النظام"""
    total = len(access_logs)
    granted = sum(1 for l in access_logs if 'GRANTED' in l.get('decision', ''))
    spoof = sum(1 for l in access_logs if 'SPOOF' in l.get('decision', ''))
    denied = total - granted - spoof

    active = sum(1 for e in employees_db.values() if e.get('access_status') == 'ACTIVE')
    suspended = sum(1 for e in employees_db.values() if e.get('access_status') == 'SUSPENDED')

    return jsonify({
        'logs': {'total': total, 'granted': granted, 'denied': denied, 'spoof': spoof},
        'employees': {'total': len(employees_db), 'active': active, 'suspended': suspended}
    })


@app.route('/api/read-plate', methods=['POST'])
def api_read_plate():
    """قراءة لوحة مركبة من صورة وتسجيل دخول/خروج (مستقل عن التعرف على الوجه)."""
    if 'image' not in request.files:
        return jsonify({'status': 'error', 'message': 'لم يتم رفع صورة'}), 400
    image_bytes = request.files['image'].read()
    img_pil = ImageOps.exif_transpose(Image.open(io.BytesIO(image_bytes)))
    img_np = np.array(img_pil)
    if len(img_np.shape) == 2:
        img_bgr = cv2.cvtColor(img_np, cv2.COLOR_GRAY2BGR)
    elif img_np.shape[2] == 4:
        img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGBA2BGR)
    else:
        img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)

    res = read_plate_from_image(img_bgr)
    if res.get('status') != 'ok':
        return jsonify(res)
    # وضع الاختبار: قراءة فقط بدون تسجيل دخول/خروج (تستخدمه خلية الاختبار الجماعي)
    if request.form.get('log', '1') == '0' or request.args.get('log') == '0':
        return jsonify(res)
    import time as _time
    _now = _time.time()
    if _now - _plate_last_seen.get(res['plate_key'], 0) < PLATE_COOLDOWN_SEC:
        _plate_last_seen[res['plate_key']] = _now
        res['status'] = 'duplicate'
        res['message'] = 'نفس اللوحة انقرأت قبل لحظات — تجاهلنا القراءة المكررة عشان ما تنقلب دخول/خروج'
        return jsonify(res)
    _plate_last_seen[res['plate_key']] = _now
    event, ts, inside_count = process_vehicle(res['plate_key'], res)
    # نحفظها كآخر لوحة عند البوابة عشان تنقرن بأقرب تعرف وجه (تسجيل فقط)
    global last_gate_plate
    last_gate_plate = {'plate_ar': res['plate_ar'], 'plate_en': res['plate_en'],
                       'plate_key': res['plate_key'], 'event': event, 'at': _now}
    res['event'] = event
    res['event_ar'] = 'دخول' if event == 'ENTRY' else 'خروج'
    res['timestamp'] = ts
    res['inside_count'] = inside_count
    return jsonify(res)


@app.route('/api/lpr-status')
def api_lpr_status():
    """حالة محركات قراءة اللوحة — للتشخيص."""
    return jsonify({
        'char_model': char_model is not None,
        'easyocr': ocr_reader is not None,
        'easyocr_en': ocr_reader_en is not None,
        'plate_detector': plate_model is not None,
        'char_names': {int(k): v for k, v in char_model.names.items()} if char_model is not None else None,
        'char_canon': {int(k): v for k, v in CHAR_CANON.items()} if CHAR_CANON else None,
    })


@app.route('/api/vehicle-logs')
def api_vehicle_logs():
    inside_count = sum(1 for v in vehicle_state.values() if v == 'inside')
    return jsonify({'logs': vehicle_logs, 'inside_count': inside_count})



# ═══════════════ 🚪 وضع البوابة المدمج: وجه + لوحة من صورة واحدة ═══════════════
@app.route('/gate')
def gate():
    return render_template('gate.html', active_page='gate')


@app.route('/api/gate', methods=['POST'])
def api_gate():
    """البوابة الذكية: يقرأ وجه السائق ولوحة سيارته من نفس الصورة ويصدر قراراً واحداً.
    الوجه يقرّر الدخول (الصلاحية)، واللوحة سياق (تُسجَّل دخول/خروج)."""
    if 'image' not in request.files:
        return jsonify({'status': 'error', 'message': 'لم يتم رفع صورة'}), 400
    image_bytes = request.files['image'].read()
    timestamp = now_sa().strftime('%Y-%m-%d %H:%M:%S')

    # (1) قراءة اللوحة من نفس الصورة
    plate = {'status': 'none'}
    try:
        _pil = ImageOps.exif_transpose(Image.open(io.BytesIO(image_bytes)))
        _arr = np.array(_pil)
        if len(_arr.shape) == 2:
            _bgr = cv2.cvtColor(_arr, cv2.COLOR_GRAY2BGR)
        elif _arr.shape[2] == 4:
            _bgr = cv2.cvtColor(_arr, cv2.COLOR_RGBA2BGR)
        else:
            _bgr = cv2.cvtColor(_arr, cv2.COLOR_RGB2BGR)
        _pr = read_plate_from_image(_bgr)
        plate = _pr if _pr.get('status') == 'ok' else {'status': 'none', 'message': _pr.get('message', '')}
    except Exception as _e:
        plate = {'status': 'error', 'message': str(_e)}

    # (2) التعرّف على الوجه
    recognition = recognize_face_from_bytes(image_bytes, check_spoof=True)

    # (3) القرار — الوجه يقرّر الدخول
    employee = employees_db.get(recognition['folder']) if recognition.get('status') == 'recognized' else None
    if recognition.get('status') == 'spoof_detected':
        decision = '❌ DENIED - SPOOF'
    elif recognition.get('status') != 'recognized' or not employee:
        decision = '❌ DENIED - ' + recognition.get('message', 'وجه غير معروف')
    elif employee.get('access_status') != 'ACTIVE':
        decision = '❌ DENIED - Suspended'
    elif CLEARANCE_RANK.get(employee.get('clearance', 'C'), 1) < CLEARANCE_RANK.get(GATE_ZONE, 1):
        decision = '❌ DENIED - CLEARANCE'
    else:
        decision = '✅ GRANTED'
    granted = 'GRANTED' in decision

    # (4) تسجيل دخول/خروج المركبة لو فيه لوحة
    veh_event = veh_event_ar = None
    if plate.get('status') == 'ok':
        try:
            import time as _t
            if _t.time() - _plate_last_seen.get(plate['plate_key'], 0) >= PLATE_COOLDOWN_SEC:
                _plate_last_seen[plate['plate_key']] = _t.time()
                veh_event, _, _ = process_vehicle(plate['plate_key'], plate)
                veh_event_ar = 'دخول' if veh_event == 'ENTRY' else 'خروج'
        except Exception:
            pass

    # (5) فتح البوابة الفعلية لو مسموح (Raspberry Pi)
    if granted:
        threading.Thread(target=trigger_gate, args=('open',), daemon=True).start()

    # (6) سجل العملية المدمجة
    access_logs.append({
        'timestamp': timestamp,
        'recognition_status': recognition.get('status'),
        'score': recognition.get('score'),
        'name_ar': employee['name_ar'] if employee else '',
        'national_id': employee['national_id'] if employee else '',
        'arrived_plate_ar': plate.get('plate_ar', '') if plate.get('status') == 'ok' else '',
        'arrived_plate_en': plate.get('plate_en', '') if plate.get('status') == 'ok' else '',
        'decision': decision,
        'mode': 'gate'
    })
    save_logs()

    return jsonify({
        'status': 'ok', 'timestamp': timestamp, 'decision': decision, 'granted': granted,
        'face': {
            'status': recognition.get('status'), 'score': recognition.get('score'),
            'name_ar': employee['name_ar'] if employee else None,
            'rank_ar': employee.get('rank_ar') if employee else None,
            'national_id': employee.get('national_id') if employee else None,
            'nationality_ar': employee.get('nationality_ar') if employee else None,
            'message': recognition.get('message'),
        },
        'plate': {
            'status': plate.get('status'), 'plate_ar': plate.get('plate_ar'),
            'plate_en': plate.get('plate_en'), 'event': veh_event, 'event_ar': veh_event_ar,
        },
    })


if __name__ == '__main__':
    initialize_system()
    app.run(host='0.0.0.0', port=5000, debug=False)




In [ ]:
%%writefile /content/app/static/css/style.css
/* ===== Reset & Base ===== */
* {
    margin: 0;
    padding: 0;
    box-sizing: border-box;
}

body {
    font-family: 'Cairo', 'Tajawal', -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
    background: #0a1428;
    color: #e8eef7;
    direction: rtl;
    min-height: 100vh;
    line-height: 1.6;
}

/* ===== Navigation ===== */
.navbar {
    background: #050d1f;
    border-bottom: 1px solid #1a2942;
    padding: 16px 32px;
    display: flex;
    align-items: center;
    justify-content: space-between;
    position: sticky;
    top: 0;
    z-index: 100;
    box-shadow: 0 2px 8px rgba(0, 0, 0, 0.3);
}

.logo-section {
    display: flex;
    align-items: center;
    gap: 12px;
}

.logo-badge {
    background: #1e6fb8;
    color: white;
    padding: 8px 12px;
    border-radius: 6px;
    font-weight: bold;
    font-size: 18px;
    letter-spacing: 1px;
}

.logo-svg {
    width: 54px;
    height: 46px;
    flex-shrink: 0;
    filter: drop-shadow(0 0 8px rgba(45, 212, 218, 0.55));
}
.logo-ar {
    font-size: 23px;
    font-weight: 800;
    color: #3fe0e6;
    letter-spacing: 1px;
    line-height: 1;
}

.logo-text {
    font-size: 14px;
    color: #a8b8d0;
    font-weight: 500;
}

.nav-links {
    display: flex;
    gap: 8px;
    list-style: none;
    flex: 1;
    justify-content: center;
}

.nav-links a {
    color: #a8b8d0;
    text-decoration: none;
    padding: 8px 16px;
    border-radius: 6px;
    font-size: 14px;
    font-weight: 500;
    transition: all 0.2s;
}

.nav-links a:hover {
    background: #1a2942;
    color: #ffffff;
}

.nav-links a.active {
    background: #1e6fb8;
    color: white;
}

.status-indicator {
    display: flex;
    align-items: center;
    gap: 8px;
    background: #0f2e4a;
    padding: 6px 14px;
    border-radius: 20px;
    font-size: 13px;
}

.status-dot {
    width: 8px;
    height: 8px;
    background: #2ecc71;
    border-radius: 50%;
    animation: pulse 2s infinite;
}

@keyframes pulse {
    0%, 100% { opacity: 1; }
    50% { opacity: 0.5; }
}

/* ===== Hero Section ===== */
.hero {
    padding: 80px 32px;
    text-align: center;
    background: linear-gradient(135deg, #0a1428 0%, #112742 100%);
    border-bottom: 1px solid #1a2942;
}

.hero h1 {
    font-size: 42px;
    margin-bottom: 20px;
    background: linear-gradient(135deg, #ffffff 0%, #6db4f0 100%);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    background-clip: text;
    font-weight: bold;
}

.hero-subtitle {
    font-size: 18px;
    color: #a8b8d0;
    max-width: 700px;
    margin: 0 auto 32px;
    line-height: 1.8;
}

.hero-buttons {
    display: flex;
    gap: 16px;
    justify-content: center;
    flex-wrap: wrap;
}

.btn {
    padding: 14px 32px;
    border-radius: 8px;
    font-size: 16px;
    font-weight: 600;
    cursor: pointer;
    border: none;
    transition: all 0.2s;
    display: inline-flex;
    align-items: center;
    gap: 10px;
    text-decoration: none;
}

.btn-primary {
    background: #1e6fb8;
    color: white;
}

.btn-primary:hover {
    background: #2580d4;
    transform: translateY(-2px);
    box-shadow: 0 4px 12px rgba(30, 111, 184, 0.4);
}

.btn-secondary {
    background: #1a2942;
    color: #e8eef7;
    border: 1px solid #2a3f5f;
}

.btn-secondary:hover {
    background: #243753;
}

/* ===== Live Monitoring ===== */
.live-section {
    padding: 40px 32px;
    background: #0a1428;
}

.live-card {
    max-width: 1200px;
    margin: 0 auto;
    background: #0f1e36;
    border: 1px solid #1a2942;
    border-radius: 12px;
    padding: 24px;
}

.live-header {
    display: flex;
    align-items: center;
    justify-content: space-between;
    margin-bottom: 24px;
}

.live-title {
    font-size: 20px;
    color: #ffffff;
    font-weight: 600;
}

.live-badge {
    background: #e74c3c;
    color: white;
    padding: 4px 12px;
    border-radius: 4px;
    font-size: 12px;
    font-weight: bold;
    display: inline-flex;
    align-items: center;
    gap: 6px;
}

.live-badge::before {
    content: '';
    width: 6px;
    height: 6px;
    background: white;
    border-radius: 50%;
    animation: pulse 1.5s infinite;
}

.live-grid {
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(220px, 1fr));
    gap: 16px;
}

.live-item {
    background: #0a1428;
    padding: 18px;
    border-radius: 8px;
    border: 1px solid #1a2942;
    display: flex;
    align-items: center;
    justify-content: space-between;
}

.live-item-label {
    color: #a8b8d0;
    font-size: 14px;
}

.live-item-status {
    color: #2ecc71;
    font-weight: bold;
    font-size: 14px;
    display: flex;
    align-items: center;
    gap: 6px;
}

/* ===== Components Section ===== */
.section {
    padding: 60px 32px;
}

.section-title {
    text-align: center;
    font-size: 32px;
    margin-bottom: 48px;
    color: #ffffff;
}

.components-grid {
    max-width: 1200px;
    margin: 0 auto;
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(260px, 1fr));
    gap: 24px;
}

.component-card {
    background: #0f1e36;
    border: 1px solid #1a2942;
    padding: 32px 24px;
    border-radius: 12px;
    text-align: center;
    transition: all 0.3s;
    cursor: pointer;
}

.component-card:hover {
    transform: translateY(-4px);
    border-color: #1e6fb8;
    box-shadow: 0 8px 24px rgba(30, 111, 184, 0.15);
}

.component-icon {
    font-size: 48px;
    margin-bottom: 16px;
    display: block;
}

.component-card h3 {
    color: #ffffff;
    font-size: 18px;
    margin-bottom: 12px;
    font-weight: 600;
}

.component-card p {
    color: #a8b8d0;
    font-size: 14px;
    line-height: 1.7;
}

/* ===== Stats ===== */
.stats-section {
    padding: 60px 32px;
    background: #050d1f;
    border-top: 1px solid #1a2942;
    border-bottom: 1px solid #1a2942;
}

.stats-grid {
    max-width: 1200px;
    margin: 0 auto;
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
    gap: 24px;
    text-align: center;
}

.stat-item {
    padding: 20px;
}

.stat-number {
    font-size: 48px;
    font-weight: bold;
    color: #6db4f0;
    margin-bottom: 8px;
}

.stat-label {
    color: #a8b8d0;
    font-size: 14px;
}

/* ===== Page Content (for sub-pages) ===== */
.page-container {
    max-width: 1400px;
    margin: 0 auto;
    padding: 32px;
}

.page-header {
    margin-bottom: 32px;
    padding-bottom: 16px;
    border-bottom: 1px solid #1a2942;
}

.page-title {
    font-size: 28px;
    color: #ffffff;
    margin-bottom: 8px;
}

.page-subtitle {
    color: #a8b8d0;
    font-size: 15px;
}

.content-card {
    background: #0f1e36;
    border: 1px solid #1a2942;
    border-radius: 12px;
    padding: 32px;
    margin-bottom: 24px;
}

/* ===== Footer ===== */
.footer {
    background: #050d1f;
    padding: 48px 32px 24px;
    border-top: 1px solid #1a2942;
}

.footer-content {
    max-width: 1200px;
    margin: 0 auto;
    display: grid;
    grid-template-columns: 2fr 1fr 1fr;
    gap: 48px;
    margin-bottom: 32px;
}

.footer-brand h3 {
    color: #ffffff;
    margin-bottom: 12px;
}

.footer-brand p {
    color: #8095b0;
    font-size: 13px;
    line-height: 1.7;
}

.footer-column h4 {
    color: #ffffff;
    margin-bottom: 16px;
    font-size: 15px;
}

.footer-column ul {
    list-style: none;
}

.footer-column ul li {
    margin-bottom: 8px;
}

.footer-column a {
    color: #8095b0;
    text-decoration: none;
    font-size: 13px;
    transition: color 0.2s;
}

.footer-column a:hover {
    color: #6db4f0;
}

.footer-bottom {
    max-width: 1200px;
    margin: 0 auto;
    padding-top: 24px;
    border-top: 1px solid #1a2942;
    text-align: center;
    color: #8095b0;
    font-size: 13px;
}

/* ===== Forms & Inputs ===== */
.upload-zone {
    border: 2px dashed #2a3f5f;
    border-radius: 12px;
    padding: 48px;
    text-align: center;
    cursor: pointer;
    transition: all 0.2s;
    background: #0a1428;
}

.upload-zone:hover {
    border-color: #1e6fb8;
    background: #0f1e36;
}

.upload-zone.dragover {
    border-color: #2ecc71;
    background: #0f1e36;
}

.upload-icon {
    font-size: 48px;
    margin-bottom: 16px;
    color: #6db4f0;
}

.upload-text {
    color: #ffffff;
    font-size: 16px;
    margin-bottom: 8px;
}

.upload-hint {
    color: #8095b0;
    font-size: 13px;
}

/* ===== Result Cards ===== */
.result-card {
    background: #0f1e36;
    border: 1px solid #1a2942;
    border-radius: 12px;
    overflow: hidden;
    margin-top: 24px;
}

.result-header {
    padding: 20px 24px;
    border-bottom: 1px solid #1a2942;
    display: flex;
    align-items: center;
    justify-content: space-between;
}

.result-status {
    padding: 8px 16px;
    border-radius: 20px;
    font-weight: bold;
    font-size: 14px;
}

.result-status.granted {
    background: rgba(46, 204, 113, 0.15);
    color: #2ecc71;
    border: 1px solid #2ecc71;
}

.result-status.denied {
    background: rgba(231, 76, 60, 0.15);
    color: #e74c3c;
    border: 1px solid #e74c3c;
}

.result-body {
    padding: 24px;
}

.info-grid {
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 24px;
}

.info-section h4 {
    color: #6db4f0;
    font-size: 14px;
    text-transform: uppercase;
    letter-spacing: 1px;
    margin-bottom: 16px;
    border-bottom: 1px solid #1a2942;
    padding-bottom: 8px;
}

.info-row {
    display: flex;
    justify-content: space-between;
    padding: 8px 0;
    border-bottom: 1px solid rgba(26, 41, 66, 0.5);
}

.info-row:last-child {
    border-bottom: none;
}

.info-label {
    color: #a8b8d0;
    font-size: 14px;
}

.info-value {
    color: #ffffff;
    font-size: 14px;
    font-weight: 500;
}

/* ===== Tables ===== */
.data-table {
    width: 100%;
    border-collapse: collapse;
    background: #0f1e36;
    border-radius: 12px;
    overflow: hidden;
}

.data-table thead {
    background: #050d1f;
}

.data-table th {
    padding: 16px;
    text-align: right;
    color: #6db4f0;
    font-size: 13px;
    text-transform: uppercase;
    letter-spacing: 1px;
    border-bottom: 1px solid #1a2942;
}

.data-table td {
    padding: 14px 16px;
    color: #e8eef7;
    font-size: 14px;
    border-bottom: 1px solid #1a2942;
}

.data-table tbody tr:hover {
    background: rgba(30, 111, 184, 0.05);
}

/* ===== Responsive ===== */
@media (max-width: 768px) {
    .navbar {
        flex-direction: column;
        gap: 16px;
        padding: 16px;
    }

    .nav-links {
        flex-wrap: wrap;
        justify-content: center;
    }

    .hero h1 {
        font-size: 32px;
    }

    .hero-subtitle {
        font-size: 16px;
    }

    .footer-content {
        grid-template-columns: 1fr;
        gap: 24px;
    }

    .info-grid {
        grid-template-columns: 1fr;
    }
}




In [ ]:
%%writefile /content/app/templates/base.html
<!DOCTYPE html>
<html lang="ar" dir="rtl">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{% block title %}عين · Smart Security Gate{% endblock %}</title>
    <link href="https://fonts.googleapis.com/css2?family=Cairo:wght@400;500;600;700&display=swap" rel="stylesheet">
    <link rel="stylesheet" href="{{ url_for('static', filename='css/style.css') }}">
    {% block extra_css %}{% endblock %}
</head>
<body>
    <nav class="navbar">
        <div class="logo-section">
            <svg class="logo-svg" viewBox="0 0 122 100" role="img" aria-label="عين | نظام بوابات ذكية" xmlns="http://www.w3.org/2000/svg">
                        <defs>
                            <radialGradient id="ayIris" cx="42%" cy="40%" r="65%">
                                <stop offset="0%" stop-color="#d4fbff"/>
                                <stop offset="45%" stop-color="#2dd4da"/>
                                <stop offset="100%" stop-color="#0a5f76"/>
                            </radialGradient>
                            <linearGradient id="ayRing" x1="0" y1="0" x2="1" y2="1">
                                <stop offset="0%" stop-color="#5fe6ec"/>
                                <stop offset="100%" stop-color="#1593b8"/>
                            </linearGradient>
                        </defs>
                        <g fill="none" stroke="url(#ayRing)" stroke-linecap="round">
                            <path d="M61 14 A 37 37 0 0 1 97 50" stroke-width="3.4"/>
                            <path d="M61 86 A 37 37 0 0 1 25 50" stroke-width="3.4"/>
                            <path d="M97 50 h9 M103 50 v-7 h7" stroke-width="2"/>
                            <path d="M25 50 h-9 M19 50 v7 h-7" stroke-width="2"/>
                        </g>
                        <circle cx="113" cy="43" r="2.4" fill="#5fe6ec"/>
                        <circle cx="9" cy="57" r="2.4" fill="#5fe6ec"/>
                        <path d="M28 50 Q61 25 94 50 Q61 75 28 50 Z" fill="#06222b" stroke="#bdf6fa" stroke-width="3"/>
                        <circle cx="61" cy="50" r="13.5" fill="url(#ayIris)"/>
                        <circle cx="61" cy="50" r="5.5" fill="#05262f"/>
                        <circle cx="56.5" cy="45.5" r="2.3" fill="#eafdff"/>
                    </svg>
            <div class="logo-text"><span class="logo-ar">عين</span></div>
        </div>
        <ul class="nav-links">
            <li><a href="{{ url_for('home') }}" class="{% if active_page == 'home' %}active{% endif %}">الرئيسية</a></li>
            <li><a href="{{ url_for('gate') }}" class="{% if active_page == 'gate' %}active{% endif %}">🚪 البوابة</a></li>
            <li><a href="{{ url_for('cameras') }}" class="{% if active_page == 'cameras' %}active{% endif %}">الكاميرات</a></li>
            <li><a href="{{ url_for('face_analysis') }}" class="{% if active_page == 'face' %}active{% endif %}">تحليل الوجه</a></li>
            <li><a href="{{ url_for('plate_reading') }}" class="{% if active_page == 'plate' %}active{% endif %}">🚗 قراءة اللوحات</a></li>
            <li><a href="{{ url_for('cards') }}" class="{% if active_page == 'cards' %}active{% endif %}">إدارة الأشخاص</a></li>
            <li><a href="{{ url_for('reports') }}" class="{% if active_page == 'reports' %}active{% endif %}">التقارير</a></li>
            <li><a href="{{ url_for('enroll') }}" class="{% if active_page == 'enroll' %}active{% endif %}">➕ تسجيل جديد</a></li>
        </ul>
        <div class="status-indicator">
            <div class="status-dot"></div>
            <span>النظام متصل</span>
        </div>
    </nav>

    {% block content %}{% endblock %}

    <footer class="footer">
        <div class="footer-content">
            <div class="footer-brand">
                <h3>عين · Smart Security Gate</h3>
                <p>منصة أمنية احترافية مخصصة لربط كاميرات المراقبة، تحليل سمات الوجه، قراءة لوحات السيارات، وإدارة الدخول للمنشآت الحساسة.</p>
            </div>
            <div class="footer-column">
                <h4>الخدمات</h4>
                <ul>
                    <li><a href="{{ url_for('face_analysis') }}">تحليل سمات الوجه</a></li>
                    <li><a href="{{ url_for('cameras') }}">ربط كاميرات المراقبة</a></li>
                    <li><a href="{{ url_for('plate_reading') }}">قراءة لوحات المركبات (LPR)</a></li>
                </ul>
            </div>
            <div class="footer-column">
                <h4>الدعم</h4>
                <ul>
                    <li><a href="#">الدعم الفني</a></li>
                    <li><a href="#">سياسة الخصوصية</a></li>
                    <li><a href="#">شروط الاستخدام</a></li>
                    <li><a href="#">التواصل</a></li>
                </ul>
            </div>
        </div>
        <div class="footer-bottom">
            © 2026 عين · Smart Security Gate - جميع الحقوق محفوظة
        </div>
    </footer>

    {% block extra_js %}{% endblock %}
</body>
</html>




In [ ]:
%%writefile /content/app/templates/gate.html
{% extends "base.html" %}

{% block title %}البوابة الذكية - عين · Smart Security Gate{% endblock %}

{% block content %}
<div class="page-container">
    <div class="page-header">
        <h1 class="page-title">🚪 البوابة الذكية</h1>
        <p class="page-subtitle">صورة واحدة للسيارة والسائق → النظام يتعرّف على الوجه ويقرأ اللوحة معاً ويُصدر قرار الدخول فوراً. الوجه يقرّر الدخول، واللوحة تُسجَّل كسياق (دخول/خروج).</p>
    </div>

    <div class="content-card">
        <div style="display: flex; gap: 12px; margin-bottom: 22px; justify-content: center; flex-wrap: wrap;">
            <button id="tab-upload" class="btn btn-primary" style="text-decoration:none;">📤 رفع صورة</button>
            <button id="tab-camera" class="btn btn-secondary" style="text-decoration:none;">📷 كاميرا مباشرة</button>
        </div>

        <div id="panel-upload">
            <div id="gate-upload-area" class="upload-zone">
                <div class="upload-icon">🚪</div>
                <div class="upload-text">اسحب صورة السيارة + السائق هنا أو اضغط للاختيار</div>
                <div class="upload-hint">يظهر بالصورة وجه السائق ولوحة المركبة • PNG, JPG, JPEG</div>
                <input type="file" id="gate-file-input" accept="image/*" style="display: none;">
            </div>
        </div>

        <div id="panel-camera" style="display: none;">
            <video id="gate-video" autoplay playsinline style="width: 100%; max-width: 600px; display: block; margin: 0 auto; border-radius: 8px; border: 2px solid #1a2942; background: #000;"></video>
            <canvas id="gate-canvas" style="display: none;"></canvas>
            <div style="text-align: center; margin-top: 16px; display: flex; gap: 12px; justify-content: center; flex-wrap: wrap;">
                <button id="gate-capture" class="btn btn-primary" style="text-decoration:none;">📸 التقاط وتحليل</button>
                <button id="gate-stop" class="btn btn-secondary" style="text-decoration:none;">⏹ إيقاف الكاميرا</button>
            </div>
            <div style="text-align: center; color: #8095b0; font-size: 13px; margin-top: 10px;">اسمح للمتصفح بالوصول للكاميرا، وجّهها على السيارة + السائق ثم اضغط «التقاط وتحليل».</div>
        </div>

        <div id="gate-loading" style="display: none; text-align: center; padding: 32px;">
            <div style="color: #6db4f0; font-size: 18px;">🔍 جاري التعرّف على الوجه وقراءة اللوحة...</div>
            <div style="color: #8095b0; font-size: 14px; margin-top: 8px;">يرجى الانتظار</div>
        </div>

        <div id="gate-result"></div>
    </div>
</div>
{% endblock %}

{% block extra_js %}
<script>
    const tabUpload = document.getElementById('tab-upload');
    const tabCamera = document.getElementById('tab-camera');
    const panelUpload = document.getElementById('panel-upload');
    const panelCamera = document.getElementById('panel-camera');
    const ga = document.getElementById('gate-upload-area');
    const gi = document.getElementById('gate-file-input');
    const gl = document.getElementById('gate-loading');
    const gr = document.getElementById('gate-result');
    const video = document.getElementById('gate-video');
    const canvas = document.getElementById('gate-canvas');
    let stream = null;

    function gateAlert(kind) {
        const cfg = {
            granted: { color: '#2ecc71', freq: 760, beeps: 1, dur: 0.15, type: 'sine' },
            denied:  { color: '#e74c3c', freq: 220, beeps: 2, dur: 0.18, type: 'square' }
        }[kind] || { color: '#6db4f0', freq: 600, beeps: 1, dur: 0.15, type: 'sine' };
        const flash = document.createElement('div');
        flash.style.cssText = 'position:fixed;inset:0;pointer-events:none;z-index:9999;opacity:0;transition:opacity .1s;background:' + cfg.color;
        document.body.appendChild(flash);
        let step = 0;
        const blink = setInterval(() => {
            flash.style.opacity = (step % 2 === 0) ? '0.35' : '0';
            step++;
            if (step >= cfg.beeps * 2) { clearInterval(blink); flash.style.opacity = '0'; setTimeout(() => flash.remove(), 200); }
        }, cfg.dur * 1000);
        try {
            const AC = window.AudioContext || window.webkitAudioContext;
            const ctx = new AC();
            for (let i = 0; i < cfg.beeps; i++) {
                const t0 = ctx.currentTime + i * (cfg.dur + 0.08);
                const osc = ctx.createOscillator();
                const g = ctx.createGain();
                osc.type = cfg.type; osc.frequency.value = cfg.freq;
                g.gain.setValueAtTime(0.0001, t0);
                g.gain.exponentialRampToValueAtTime(0.25, t0 + 0.01);
                g.gain.exponentialRampToValueAtTime(0.0001, t0 + cfg.dur);
                osc.connect(g); g.connect(ctx.destination);
                osc.start(t0); osc.stop(t0 + cfg.dur + 0.03);
            }
        } catch (e) {}
    }

    // === تبديل بين رفع صورة / كاميرا ===
    function setMode(mode) {
        const cam = mode === 'camera';
        panelUpload.style.display = cam ? 'none' : 'block';
        panelCamera.style.display = cam ? 'block' : 'none';
        tabUpload.className = cam ? 'btn btn-secondary' : 'btn btn-primary';
        tabCamera.className = cam ? 'btn btn-primary' : 'btn btn-secondary';
        if (cam) startCamera(); else stopCamera();
    }
    tabUpload.addEventListener('click', () => setMode('upload'));
    tabCamera.addEventListener('click', () => setMode('camera'));

    // === الكاميرا المباشرة ===
    async function startCamera() {
        if (stream) return;
        try {
            stream = await navigator.mediaDevices.getUserMedia({ video: { facingMode: 'environment' }, audio: false });
            video.srcObject = stream;
        } catch (e) {
            alert('تعذّر فتح الكاميرا: ' + e.message + '\nتأكد أنك سمحت بالوصول للكاميرا وأن الرابط يبدأ بـ https.');
            setMode('upload');
        }
    }
    function stopCamera() {
        if (stream) { stream.getTracks().forEach(t => t.stop()); stream = null; video.srcObject = null; }
    }
    document.getElementById('gate-stop').addEventListener('click', () => setMode('upload'));
    document.getElementById('gate-capture').addEventListener('click', () => {
        if (!stream || !video.videoWidth) { alert('الكاميرا لم تجهز بعد'); return; }
        canvas.width = video.videoWidth; canvas.height = video.videoHeight;
        canvas.getContext('2d').drawImage(video, 0, 0);
        canvas.toBlob((b) => { if (b) processGate(b); }, 'image/jpeg', 0.92);
    });

    // === رفع صورة ===
    ga.addEventListener('click', () => gi.click());
    ga.addEventListener('dragover', (e) => { e.preventDefault(); ga.classList.add('dragover'); });
    ga.addEventListener('dragleave', () => ga.classList.remove('dragover'));
    ga.addEventListener('drop', (e) => { e.preventDefault(); ga.classList.remove('dragover'); if (e.dataTransfer.files.length) processGate(e.dataTransfer.files[0]); });
    gi.addEventListener('change', (e) => { if (e.target.files.length) processGate(e.target.files[0]); });

    // === الإرسال للخادم (يعمل مع الرفع والكاميرا) ===
    async function processGate(blob) {
        if (blob.type && !blob.type.startsWith('image/')) { alert('يرجى رفع ملف صورة فقط'); return; }
        const fd = new FormData(); fd.append('image', blob, 'gate.jpg');
        gl.style.display = 'block'; gr.innerHTML = '';
        try {
            const res = await fetch('/api/gate', { method: 'POST', body: fd });
            const d = await res.json();
            showGate(d, URL.createObjectURL(blob));
        } catch (e) {
            gr.innerHTML = '<div style="color:#e74c3c;padding:16px;text-align:center;">حدث خطأ: ' + e.message + '</div>';
        } finally {
            gl.style.display = 'none';
        }
    }

    function showGate(d, imgUrl) {
        if (d.status !== 'ok') {
            gr.innerHTML = '<div style="color:#e74c3c;padding:16px;text-align:center;">' + (d.message || 'تعذّرت المعالجة') + '</div>';
            return;
        }
        gateAlert(d.granted ? 'granted' : 'denied');
        const g = d.granted, face = d.face, plate = d.plate;
        const recognized = face.status === 'recognized';
        const dash = '—';
        const row = (label, val) => `<div class="info-row"><span class="info-label">${label}</span><span class="info-value">${val}</span></div>`;
        const plateRow = (label, val) => `<div class="info-row"><span class="info-label">${label}</span><span class="info-value" style="font-size:18px;letter-spacing:2px;">${val}</span></div>`;
        const nameVal = recognized ? (face.name_ar || dash) : (face.status === 'spoof_detected' ? '⚠️ محاولة انتحال' : 'غير معروف');
        gr.innerHTML = `
            <div class="result-card">
                <div class="result-header">
                    <h3 style="color:#ffffff;">قرار البوابة</h3>
                    <span class="result-status ${g ? 'granted' : 'denied'}" style="font-size:16px;">${g ? '✅ مصرّح بالدخول' : '⛔ غير مصرّح'}</span>
                </div>
                <div class="result-body">
                    <div style="display:grid;grid-template-columns:300px 1fr;gap:28px;align-items:start;">
                        <div>
                            <img src="${imgUrl}" style="width:100%;border-radius:8px;border:2px solid #1a2942;">
                            <div style="margin-top:8px;text-align:center;color:#8095b0;font-size:12px;">${d.timestamp}</div>
                        </div>
                        <div class="info-section">
                            ${row('الاسم', nameVal)}
                            ${row('الهوية', recognized ? (face.national_id || dash) : dash)}
                            ${row('الجنسية', recognized ? (face.nationality_ar || dash) : dash)}
                            ${row('الرتبة', recognized ? (face.rank_ar || dash) : dash)}
                            ${plateRow('اللوحة (عربي)', plate.status === 'ok' ? (plate.plate_ar || dash) : dash)}
                            ${plateRow('اللوحة (إنجليزي)', plate.status === 'ok' ? (plate.plate_en || dash) : dash)}
                        </div>
                    </div>
                </div>
            </div>`;
    }
</script>
{% endblock %}


In [ ]:
%%writefile /content/app/templates/home.html
{% extends "base.html" %}

{% block title %}الرئيسية - عين · Smart Security Gate{% endblock %}

{% block content %}
<section class="hero">
    <h1>عَين</h1>
    <p class="hero-subtitle">
        منصة أمنية احترافية مدعومة بالذكاء الاصطناعي للتعرّف على الوجوه وقراءة لوحات المركبات السعودية، مع كشف محاولات الانتحال، والتحكم الشامل بالدخول، وتسجيل كل العمليات تلقائياً. مصمّمة للمنشآت الحساسة.
    </p>
    <div class="hero-buttons">
        <a href="{{ url_for('gate') }}" class="btn btn-primary">
            🚪 ابدأ البوابة الذكية
        </a>
        <a href="{{ url_for('reports') }}" class="btn btn-secondary">
            📊 عرض لوحة التحكم
        </a>
    </div>
</section>

<section class="live-section">
    <div class="live-card">
        <div class="live-header">
            <h2 class="live-title">المراقبة المباشرة</h2>
            <span class="live-badge">LIVE</span>
        </div>
        <div class="live-grid">
            <div class="live-item">
                <span class="live-item-label">كاميرات المراقبة</span>
                <span class="live-item-status">● متصلة</span>
            </div>
            <div class="live-item">
                <span class="live-item-label">تحليل سمات الوجه</span>
                <span class="live-item-status">● نشط</span>
            </div>
            <div class="live-item">
                <span class="live-item-label">قراءة لوحات المركبات (LPR)</span>
                <span class="live-item-status">● نشط</span>
            </div>
        </div>
    </div>
</section>

<section class="section">
    <h2 class="section-title">مكونات النظام</h2>
    <div class="components-grid">
        <a href="{{ url_for('face_analysis') }}" class="component-card" style="text-decoration: none;">
            <span class="component-icon">👤</span>
            <h3>تحليل سمات الوجه</h3>
            <p>تحليل السمات العامة للوجه عبر الكاميرات لدعم التحقق الأمني ومراقبة نقاط الدخول.</p>
        </a>
        <a href="{{ url_for('plate_reading') }}" class="component-card" style="text-decoration: none;">
            <span class="component-icon">🚗</span>
            <h3>قراءة لوحات المركبات (LPR)</h3>
            <p>كشف لوحة المركبة وقراءتها (عربي/إنجليزي) وتسجيل دخول وخروج المركبات تلقائياً.</p>
        </a>
        <a href="{{ url_for('cameras') }}" class="component-card" style="text-decoration: none;">
            <span class="component-icon">🎥</span>
            <h3>ربط كاميرات المراقبة</h3>
            <p>عرض حالة الكاميرات وربطها بواجهات المراقبة والتحليل داخل النظام.</p>
        </a>
    </div>
</section>

<section class="stats-section">
    <div class="stats-grid">
        <div class="stat-item">
            <div class="stat-number" id="stat-cameras">24</div>
            <div class="stat-label">كاميرا مراقبة</div>
        </div>
        <div class="stat-item">
            <div class="stat-number" id="stat-verifications">{{ stats.verifications }}</div>
            <div class="stat-label">عملية تحقق يومية</div>
        </div>
        <div class="stat-item">
            <div class="stat-number" id="stat-vehicles">{{ stats.vehicles }}</div>
            <div class="stat-label">مركبة مرصودة</div>
        </div>
        <div class="stat-item">
            <div class="stat-number">99%</div>
            <div class="stat-label">جاهزية النظام</div>
        </div>
    </div>
</section>

<section class="section">
    <h2 class="section-title">استخدامات النظام</h2>
    <div style="max-width: 900px; margin: 0 auto; text-align: center;">
        <p style="color: #a8b8d0; font-size: 17px; line-height: 1.9;">
            يستخدم النظام في المنشآت العسكرية، المطارات، البوابات الأمنية، مواقف المركبات،
            ومنافذ العبور، بهدف رفع مستوى التحكم والمراقبة وتنظيم الدخول والخروج بطريقة رقمية احترافية.
        </p>
    </div>
</section>
{% endblock %}




In [ ]:
%%writefile /content/app/templates/face_analysis.html
{% extends "base.html" %}

{% block title %}تحليل الوجه - عين · Smart Security Gate{% endblock %}

{% block content %}
<div class="page-container">
    <div class="page-header">
        <h1 class="page-title">👤 تحليل سمات الوجه</h1>
        <p class="page-subtitle">ارفع صورة لشخص لتحليلها والتعرف على هويته من قاعدة البيانات</p>
    </div>

    <div class="content-card">
        <!-- اختيار الطريقة: رفع صورة أو كاميرا مباشرة -->
        <div style="display: flex; gap: 12px; margin-bottom: 20px; justify-content: center;">
            <button class="btn btn-secondary" id="mode-upload" onclick="switchMode('upload')" style="padding: 10px 24px;">📁 رفع صورة</button>
            <button class="btn btn-secondary" id="mode-camera" onclick="switchMode('camera')" style="padding: 10px 24px;">📷 كاميرا مباشرة</button>
        </div>


        <!-- وضع رفع الصورة -->
        <div id="upload-mode">
            <div id="upload-area" class="upload-zone">
                <div class="upload-icon">📷</div>
                <div class="upload-text">اسحب الصورة هنا أو اضغط للاختيار</div>
                <div class="upload-hint">PNG, JPG, JPEG • الحد الأقصى: 50 ميجابايت</div>
                <input type="file" id="file-input" accept="image/*" style="display: none;">
            </div>
            <label style="display:flex;align-items:center;gap:8px;justify-content:center;margin-top:12px;color:#8095b0;font-size:14px;cursor:pointer;">
                <input type="checkbox" id="spoof-toggle" checked>
                تفعيل فحص الانتحال على الصور المرفوعة (عطّله مؤقتاً عند تجربة صور الداتاسيت)
            </label>
        </div>

        <!-- وضع الكاميرا المباشرة -->
        <div id="camera-mode" style="display: none;">
            <div style="text-align: center;">
                <video id="webcam-video" autoplay playsinline style="max-width: 100%; width: 480px; border-radius: 12px; border: 2px solid #1a2942; background: #0a1428;"></video>
                <canvas id="webcam-canvas" style="display: none;"></canvas>
                <div style="margin-top: 16px; display: flex; gap: 12px; justify-content: center;">
                    <button class="btn btn-primary" onclick="captureAndAnalyze()" id="capture-btn" style="padding: 12px 32px;">
                        📸 التقط وحلّل
                    </button>
                    <button class="btn btn-primary" onclick="livenessCheck()" id="liveness-btn" style="padding: 12px 24px;">
                        🔒 فحص الحيوية + تحليل
                    </button>
                    <button class="btn btn-secondary" onclick="stopCamera()" id="stop-camera-btn" style="padding: 12px 24px;">
                        ⏹️ إيقاف الكاميرا
                    </button>
                </div>
                <div id="camera-status" style="margin-top: 12px; color: #8095b0; font-size: 14px;"></div>
            </div>
        </div>

        <div id="loading" style="display: none; text-align: center; padding: 32px;">
            <div style="color: #6db4f0; font-size: 18px;">🔍 جاري التحليل...</div>
            <div style="color: #8095b0; font-size: 14px; margin-top: 8px;">يرجى الانتظار</div>
        </div>

        <div id="result-container"></div>
    </div>
</div>
{% endblock %}

{% block extra_js %}
<script>
    const uploadArea = document.getElementById('upload-area');
    const fileInput = document.getElementById('file-input');
    const resultContainer = document.getElementById('result-container');
    const loading = document.getElementById('loading');
    const video = document.getElementById('webcam-video');
    const canvas = document.getElementById('webcam-canvas');
    let cameraStream = null;

    // === تنبيه صوتي وبصري عند نتيجة البوابة ===
    function gateAlert(kind) {
        const cfg = {
            granted: { color: '#2ecc71', freq: 880, beeps: 1, dur: 0.15, type: 'sine' },
            entry:   { color: '#2ecc71', freq: 760, beeps: 1, dur: 0.15, type: 'sine' },
            exit:    { color: '#6db4f0', freq: 520, beeps: 1, dur: 0.15, type: 'sine' },
            denied:  { color: '#e74c3c', freq: 220, beeps: 2, dur: 0.18, type: 'square' },
            spoof:   { color: '#e74c3c', freq: 160, beeps: 3, dur: 0.18, type: 'square' }
        }[kind] || { color: '#6db4f0', freq: 600, beeps: 1, dur: 0.15, type: 'sine' };

        // فلاش بصري على كامل الشاشة
        const flash = document.createElement('div');
        flash.style.cssText = 'position:fixed;inset:0;pointer-events:none;z-index:9999;opacity:0;transition:opacity .1s;background:' + cfg.color;
        document.body.appendChild(flash);
        let step = 0;
        const blink = setInterval(() => {
            flash.style.opacity = (step % 2 === 0) ? '0.35' : '0';
            step++;
            if (step >= cfg.beeps * 2) { clearInterval(blink); flash.style.opacity = '0'; setTimeout(() => flash.remove(), 200); }
        }, cfg.dur * 1000);

        // نغمة صوتية
        try {
            const AC = window.AudioContext || window.webkitAudioContext;
            const ctx = new AC();
            for (let i = 0; i < cfg.beeps; i++) {
                const t0 = ctx.currentTime + i * (cfg.dur + 0.08);
                const osc = ctx.createOscillator();
                const g = ctx.createGain();
                osc.type = cfg.type;
                osc.frequency.value = cfg.freq;
                g.gain.setValueAtTime(0.0001, t0);
                g.gain.exponentialRampToValueAtTime(0.25, t0 + 0.01);
                g.gain.exponentialRampToValueAtTime(0.0001, t0 + cfg.dur);
                osc.connect(g); g.connect(ctx.destination);
                osc.start(t0); osc.stop(t0 + cfg.dur + 0.03);
            }
        } catch (e) {}
    }

    // === التبديل بين رفع الصورة والكاميرا ===
    function switchMode(mode) {
        const modes = {
            upload: document.getElementById('upload-mode'),
            camera: document.getElementById('camera-mode')
        };
        const btns = {
            upload: document.getElementById('mode-upload'),
            camera: document.getElementById('mode-camera')
        };
        resultContainer.innerHTML = '';
        for (const k in modes) {
            modes[k].style.display = (k === mode) ? 'block' : 'none';
            btns[k].classList.toggle('btn-primary', k === mode);
            btns[k].classList.toggle('btn-secondary', k !== mode);
        }
        if (mode === 'camera') { startCamera(); } else { stopCamera(); }
    }

    // === تشغيل الكاميرا ===
    async function startCamera() {
        const statusEl = document.getElementById('camera-status');
        statusEl.textContent = '⏳ جاري تشغيل الكاميرا...';
        try {
            cameraStream = await navigator.mediaDevices.getUserMedia({
                video: { width: 640, height: 480, facingMode: 'user' },
                audio: false
            });
            video.srcObject = cameraStream;
            statusEl.textContent = '✅ الكاميرا تعمل - ضع وجهك في الإطار واضغط "التقط وحلّل"';
            statusEl.style.color = '#2ecc71';
        } catch (err) {
            statusEl.textContent = '❌ تعذّر الوصول للكاميرا. تأكد من السماح بالوصول من المتصفح.';
            statusEl.style.color = '#e74c3c';
            console.error('Camera error:', err);
        }
    }

    // === إيقاف الكاميرا ===
    function stopCamera() {
        if (cameraStream) {
            cameraStream.getTracks().forEach(track => track.stop());
            cameraStream = null;
            video.srcObject = null;
        }
        document.getElementById('camera-status').textContent = '';
    }

    // === التقاط صورة من الكاميرا وتحليلها ===
    async function captureAndAnalyze() {
        if (!cameraStream) {
            alert('الكاميرا غير شغّالة - اضغط على "كاميرا مباشرة" أولاً');
            return;
        }
        // نلتقط الإطار الحالي على canvas
        canvas.width = video.videoWidth;
        canvas.height = video.videoHeight;
        const ctx = canvas.getContext('2d');
        ctx.drawImage(video, 0, 0);

        // نحوّلها لـ blob ونرسلها كـ file
        canvas.toBlob(async (blob) => {
            const file = new File([blob], 'webcam_capture.jpg', { type: 'image/jpeg' });
            await handleFile(file);
        }, 'image/jpeg', 0.95);
    }

    // === فحص الحيوية النشط (حركة الرأس + الرمشة) ===
    async function livenessCheck() {
        if (!cameraStream) { alert('الكاميرا غير شغّالة - اضغط على "كاميرا مباشرة" أولاً'); return; }
        const statusEl = document.getElementById('camera-status');
        const btn = document.getElementById('liveness-btn');
        btn.disabled = true;
        const frames = [];
        const N = 12;
        for (let i = 0; i < N; i++) {
            statusEl.style.color = '#6db4f0';
            statusEl.textContent = `🔒 فحص الحيوية... حرّك رأسك يمين/يسار وارمش (${i + 1}/${N})`;
            canvas.width = video.videoWidth; canvas.height = video.videoHeight;
            canvas.getContext('2d').drawImage(video, 0, 0);
            const blob = await new Promise(r => canvas.toBlob(r, 'image/jpeg', 0.8));
            if (blob) frames.push(blob);
            await new Promise(r => setTimeout(r, 230));
        }
        const fd = new FormData();
        frames.forEach((b, i) => fd.append('frames', new File([b], `f${i}.jpg`, { type: 'image/jpeg' })));
        statusEl.textContent = '🔍 تحليل الحيوية...';
        try {
            const res = await fetch('/api/liveness', { method: 'POST', body: fd });
            const data = await res.json();
            if (data.status === 'ok' && data.live) {
                const sig = [];
                if (data.movement_detected) sig.push('حركة رأس');
                if (data.blink_detected) sig.push('رمشة');
                statusEl.style.color = '#2ecc71';
                statusEl.textContent = `✅ شخص حيّ (${sig.join(' + ')}) — جاري التحليل...`;
                gateAlert('granted');
                await captureAndAnalyze();
            } else {
                statusEl.style.color = '#e74c3c';
                statusEl.textContent = '🛑 فشل فحص الحيوية — حرّك رأسك يمين/يسار وارمش أمام الكاميرا مباشرة';
                gateAlert('spoof');
                resultContainer.innerHTML = `<div class="result-card"><div class="result-header"><h3 style="color:#fff;">فحص الحيوية</h3><span class="result-status denied">🛑 غير حيّ</span></div><div class="result-body" style="text-align:center;color:#e74c3c;padding:16px;">لم نكتشف حركة رأس أو رمشة — قد تكون صورة ثابتة. لو شخص حقيقي، أعد المحاولة.</div></div>`;
            }
        } catch (e) {
            statusEl.style.color = '#e74c3c';
            statusEl.textContent = '❌ خطأ: ' + e.message;
        } finally {
            btn.disabled = false;
        }
    }

    // === الـ Listeners القديمة (رفع الصورة) ===
    uploadArea.addEventListener('click', () => fileInput.click());

    uploadArea.addEventListener('dragover', (e) => {
        e.preventDefault();
        uploadArea.classList.add('dragover');
    });

    uploadArea.addEventListener('dragleave', () => {
        uploadArea.classList.remove('dragover');
    });

    uploadArea.addEventListener('drop', (e) => {
        e.preventDefault();
        uploadArea.classList.remove('dragover');
        if (e.dataTransfer.files.length > 0) {
            handleFile(e.dataTransfer.files[0]);
        }
    });

    fileInput.addEventListener('change', (e) => {
        if (e.target.files.length > 0) {
            handleFile(e.target.files[0]);
        }
    });

    async function handleFile(file) {
        if (!file.type.startsWith('image/')) {
            alert('يرجى رفع ملف صورة فقط');
            return;
        }

        const formData = new FormData();
        formData.append('image', file);
        const spoofToggle = document.getElementById('spoof-toggle');
        const uploadVisible = document.getElementById('upload-mode').style.display !== 'none';
        if (spoofToggle && !spoofToggle.checked && uploadVisible) formData.append('spoof', 'off');

        loading.style.display = 'block';
        resultContainer.innerHTML = '';

        try {
            const response = await fetch('/api/recognize', {
                method: 'POST',
                body: formData
            });
            const data = await response.json();
            displayResult(data, file);
        } catch (error) {
            resultContainer.innerHTML = `<div style="color: #e74c3c; padding: 16px; text-align: center;">حدث خطأ: ${error.message}</div>`;
        } finally {
            loading.style.display = 'none';
        }
    }

    function displayResult(data, file) {
        const imageURL = URL.createObjectURL(file);
        if (data.status === 'spoof_detected') gateAlert('spoof');
        else if (data.status === 'error' || data.status === 'unknown') gateAlert('denied');
        else if (data.employee) gateAlert((data.decision || '').includes('GRANTED') ? 'granted' : 'denied');

        if (data.status === 'spoof_detected') {
            resultContainer.innerHTML = `
                <div class="result-card">
                    <div class="result-header">
                        <h3 style="color: #ffffff;">نتيجة التحليل</h3>
                        <span class="result-status denied">🛑 محاولة انتحال</span>
                    </div>
                    <div class="result-body" style="text-align: center;">
                        <img src="${imageURL}" style="max-width: 300px; border-radius: 8px; margin-bottom: 16px;">
                        <div style="color: #e74c3c; font-size: 18px; font-weight: bold;">🛑 ${data.message}</div>
                        <div style="color: #8095b0; margin-top: 12px; font-size: 14px;">
                            النظام كشف أن الصورة غير حقيقية (صورة من شاشة أو مطبوعة).<br>
                            يرجى استخدام كاميرا مباشرة على الوجه الحقيقي.
                        </div>
                    </div>
                </div>
            `;
            return;
        }

        if (data.status === 'error' || data.status === 'unknown') {
            resultContainer.innerHTML = `
                <div class="result-card">
                    <div class="result-header">
                        <h3 style="color: #ffffff;">نتيجة التحليل</h3>
                        <span class="result-status denied">❌ غير معروف</span>
                    </div>
                    <div class="result-body" style="text-align: center;">
                        <img src="${imageURL}" style="max-width: 300px; border-radius: 8px; margin-bottom: 16px;">
                        <div style="color: #e74c3c; font-size: 16px;">${data.message || 'لم يتم التعرف على الشخص'}</div>
                        ${data.score ? `<div style="color: #8095b0; margin-top: 8px;">أعلى تطابق: ${(data.score * 100).toFixed(1)}%</div>` : ''}
                    </div>
                </div>
            `;
            return;
        }

        const emp = data.employee;
        const isGranted = (data.decision || '').includes('GRANTED');

        resultContainer.innerHTML = `
            <div class="result-card">
                <div class="result-header">
                    <h3 style="color: #ffffff;">نتيجة التحليل</h3>
                    <span class="result-status ${isGranted ? 'granted' : 'denied'}">
                        ${isGranted ? '✅ مصرح بالدخول' : '❌ غير مصرح'}
                    </span>
                </div>
                <div class="result-body">
                    <div style="display: grid; grid-template-columns: 200px 1fr; gap: 32px; align-items: start;">
                        <div>
                            <img src="${imageURL}" style="width: 100%; border-radius: 8px;">
                            <div style="margin-top: 12px; text-align: center; color: #6db4f0; font-size: 13px;">
                                نسبة التطابق: ${(data.score * 100).toFixed(1)}%
                            </div>
                        </div>
                        <div class="info-grid" style="grid-template-columns: 1fr 1fr;">
                            <div class="info-section">
                                <h4>المعلومات الشخصية</h4>
                                <div class="info-row">
                                    <span class="info-label">${emp.id_type_ar}</span>
                                    <span class="info-value">${emp.national_id}</span>
                                </div>
                                <div class="info-row">
                                    <span class="info-label">الاسم</span>
                                    <span class="info-value">${emp.name_ar}</span>
                                </div>
                                <div class="info-row">
                                    <span class="info-label">العمر</span>
                                    <span class="info-value">${emp.age} سنة</span>
                                </div>
                                <div class="info-row">
                                    <span class="info-label">الجنس</span>
                                    <span class="info-value">${emp.gender_ar}</span>
                                </div>
                                <div class="info-row">
                                    <span class="info-label">الجنسية</span>
                                    <span class="info-value">${emp.nationality_ar}</span>
                                </div>
                                <div class="info-row">
                                    <span class="info-label">الرتبة</span>
                                    <span class="info-value">${emp.rank_ar}</span>
                                </div>
                                <div class="info-row">
                                    <span class="info-label">الحالة</span>
                                    <span class="info-value" style="color: ${isGranted ? '#2ecc71' : '#e74c3c'}">${emp.access_status_ar}</span>
                                </div>
                            </div>
                            <div class="info-section">
                                <h4>التحليل الأمني وسمات الوجه</h4>
                                <div class="info-row">
                                    <span class="info-label">نسبة التطابق</span>
                                    <span class="info-value" style="color: #6db4f0;">${(data.score * 100).toFixed(1)}%</span>
                                </div>
                                <div class="info-row">
                                    <span class="info-label">الجنس (مكتشف)</span>
                                    <span class="info-value">${emp.gender_ar}</span>
                                </div>
                                <div class="info-row">
                                    <span class="info-label">العمر التقديري</span>
                                    <span class="info-value">${emp.age} سنة</span>
                                </div>
                                <div class="info-row">
                                    <span class="info-label">فحص الانتحال</span>
                                    <span class="info-value" style="color: #2ecc71;">✅ وجه حقيقي</span>
                                </div>
                                <div class="info-row">
                                    <span class="info-label">مستوى الصلاحية</span>
                                    <span class="info-value">${emp.clearance_ar || '-'} ${(data.decision && data.decision.includes('CLEARANCE')) ? '⚠️ غير كافٍ' : ''}</span>
                                </div>
                                <div class="info-row">
                                    <span class="info-label">منطقة البوابة المطلوبة</span>
                                    <span class="info-value">${data.gate_zone_ar || data.gate_zone || '-'}</span>
                                </div>
                                <div class="info-row">
                                    <span class="info-label">الوقت</span>
                                    <span class="info-value">${data.timestamp}</span>
                                </div>
                                ${data.arrived_with ? `
                                <div class="info-row">
                                    <span class="info-label">وصل بمركبة (قراءة البوابة)</span>
                                    <span class="info-value" style="color:#6db4f0;">${data.arrived_with.plate_ar} <span style="color:#8095b0;font-size:11px;">(قبل ${data.arrived_with.ago_sec} ثانية)</span></span>
                                </div>` : ''}
                                <div class="info-row">
                                    <span class="info-label">الإجراء</span>
                                    <span class="info-value">${isGranted ? '🟢 فتح البوابة' : '🔴 منع الدخول'}</span>
                                </div>
                            </div>
                        </div>
                    </div>
                </div>
            </div>
        `;
    }
</script>
{% endblock %}




In [ ]:
%%writefile /content/app/templates/enroll.html
{% extends "base.html" %}

{% block title %}تسجيل شخص جديد - عين · Smart Security Gate{% endblock %}

{% block content %}
<div class="page-container">
    <div class="page-header">
        <h1 class="page-title">➕ تسجيل شخص جديد</h1>
        <p class="page-subtitle">سجّل شخص جديد بصوره وبياناته - النظام يكتشف الجنس والعمر تلقائياً ويحفظه للتعرف عليه لاحقاً</p>
    </div>

    <div class="content-card">
        <!-- صور الوجه (متعددة) -->
        <div>
            <h4 style="color: #6db4f0; margin-bottom: 16px;">📷 صور الوجه (مطلوبة - عدة صور للشخص نفسه)</h4>
            <div id="face-upload" class="upload-zone" style="padding: 32px;">
                <div class="upload-icon">👤</div>
                <div class="upload-text">اضغط لرفع صور الوجه</div>
                <div class="upload-hint">ارفع عدة صور لنفس الشخص (بشماغ، بدون، بنظارة، زوايا مختلفة...) — كل ما زادت الصور، صار التعرف أدق</div>
                <input type="file" id="face-input" accept="image/*" multiple style="display: none;">
            </div>
            <div id="face-previews" style="display: flex; flex-wrap: wrap; gap: 8px; margin-top: 16px;"></div>
        </div>

        <div style="border-top: 1px solid #1a2942; margin: 32px 0; padding-top: 32px;">
            <h4 style="color: #6db4f0; margin-bottom: 20px;">📝 بيانات الشخص</h4>
            <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 20px;">
                <div>
                    <label class="form-label">الاسم (عربي)</label>
                    <input type="text" id="name_ar" class="form-input" placeholder="محمد بن عبدالله القحطاني">
                </div>
                <div>
                    <label class="form-label">الاسم (إنجليزي)</label>
                    <input type="text" id="name_en" class="form-input" placeholder="Mohammed bin Abdullah Al-Qahtani">
                </div>
                <div>
                    <label class="form-label">الجنسية</label>
                    <select id="nationality" class="form-input">
                        <option value="saudi">سعودي</option>
                        <option value="egyptian">مصري</option>
                        <option value="indian">هندي</option>
                        <option value="pakistani">باكستاني</option>
                        <option value="bangladeshi">بنغالي</option>
                        <option value="nepali">نيبالي</option>
                    </select>
                </div>
                <div>
                    <label class="form-label">رقم الهوية / الإقامة</label>
                    <input type="text" id="national_id" class="form-input" placeholder="1xxxxxxxxx">
                </div>
                <div>
                    <label class="form-label">الرتبة</label>
                    <input type="text" id="rank_ar" class="form-input" placeholder="م5 / نقيب / عامل">
                </div>
                <div>
                    <label class="form-label">مستوى الصلاحية</label>
                    <select id="clearance" class="form-input">
                        <option value="A" selected>منطقة أ (الأعلى)</option>
                        <option value="B">منطقة ب</option>
                        <option value="C">منطقة ج</option>
                    </select>
                </div>
                <div>
                    <label class="form-label">رقم اللوحة (عربي - اختياري)</label>
                    <input type="text" id="plate_ar" class="form-input" placeholder="أ ب ج  ١٢٣٤">
                </div>
                <div>
                    <label class="form-label">نوع المركبة</label>
                    <input type="text" id="brand_ar" class="form-input" placeholder="تويوتا">
                </div>
                <div>
                    <label class="form-label">لون المركبة</label>
                    <input type="text" id="color_ar" class="form-input" placeholder="أبيض">
                </div>
                <div>
                    <label class="form-label">صورة اللوحة (اختياري)</label>
                    <input type="file" id="plate_image" class="form-input" accept="image/*">
                </div>
            </div>
        </div>

        <div style="text-align: center; margin-top: 24px;">
            <button class="btn btn-primary" onclick="enrollPerson()" id="enroll-btn">
                ✅ تسجيل الشخص
            </button>
        </div>

        <div id="enroll-result" style="margin-top: 24px;"></div>
    </div>
</div>
{% endblock %}

{% block extra_css %}
<style>
.form-label {
    display: block;
    color: #a8b8d0;
    font-size: 14px;
    margin-bottom: 8px;
}
.form-input {
    width: 100%;
    padding: 12px 16px;
    background: #0a1428;
    border: 1px solid #2a3f5f;
    border-radius: 8px;
    color: #e8eef7;
    font-size: 14px;
    font-family: inherit;
}
.form-input:focus {
    outline: none;
    border-color: #1e6fb8;
}
</style>
{% endblock %}

{% block extra_js %}
<script>
    let faceFiles = [];

    document.getElementById('face-upload').addEventListener('click', () => document.getElementById('face-input').click());
    document.getElementById('face-input').addEventListener('change', (e) => {
        if (e.target.files.length > 0) {
            // نضيف الصور الجديدة للقائمة
            for (const file of e.target.files) {
                faceFiles.push(file);
            }
            renderFacePreviews();
        }
    });

    function renderFacePreviews() {
        const container = document.getElementById('face-previews');
        container.innerHTML = '';
        faceFiles.forEach((file, index) => {
            const wrapper = document.createElement('div');
            wrapper.style.cssText = 'position: relative; display: inline-block;';
            const img = document.createElement('img');
            img.src = URL.createObjectURL(file);
            img.style.cssText = 'width: 90px; height: 90px; object-fit: cover; border-radius: 8px; border: 1px solid #2a3f5f;';
            const removeBtn = document.createElement('button');
            removeBtn.textContent = '✕';
            removeBtn.style.cssText = 'position: absolute; top: -6px; left: -6px; background: #e74c3c; color: white; border: none; border-radius: 50%; width: 22px; height: 22px; cursor: pointer; font-size: 12px;';
            removeBtn.onclick = () => { faceFiles.splice(index, 1); renderFacePreviews(); };
            wrapper.appendChild(img);
            wrapper.appendChild(removeBtn);
            container.appendChild(wrapper);
        });
        if (faceFiles.length > 0) {
            const counter = document.createElement('div');
            counter.style.cssText = 'width: 100%; color: #6db4f0; font-size: 13px; margin-top: 4px;';
            counter.textContent = `${faceFiles.length} صورة`;
            container.appendChild(counter);
        }
    }

    async function enrollPerson() {
        if (faceFiles.length === 0) {
            alert('يرجى رفع صورة وجه واحدة على الأقل');
            return;
        }

        const btn = document.getElementById('enroll-btn');
        btn.disabled = true;
        btn.textContent = '⏳ جاري التسجيل...';

        const formData = new FormData();
        faceFiles.forEach(file => formData.append('face_images', file));
        formData.append('name_ar', document.getElementById('name_ar').value);
        formData.append('name_en', document.getElementById('name_en').value);
        formData.append('nationality', document.getElementById('nationality').value);
        formData.append('national_id', document.getElementById('national_id').value);
        formData.append('rank_ar', document.getElementById('rank_ar').value);
        formData.append('clearance', document.getElementById('clearance').value);
        formData.append('plate_ar', document.getElementById('plate_ar').value);
        formData.append('brand_ar', document.getElementById('brand_ar').value);
        formData.append('color_ar', document.getElementById('color_ar').value);
        const plateImg = document.getElementById('plate_image');
        if (plateImg && plateImg.files.length > 0) formData.append('plate_image', plateImg.files[0]);

        try {
            const response = await fetch('/api/enroll', { method: 'POST', body: formData });
            const data = await response.json();

            const resultDiv = document.getElementById('enroll-result');
            if (data.status === 'success') {
                resultDiv.innerHTML = `
                    <div class="result-status granted" style="display: block; text-align: center; padding: 16px;">
                        ✅ ${data.message}<br>
                        <span style="font-size: 13px; opacity: 0.8;">
                            عدد الصور المستخدمة: ${data.images_used} | الجنس: ${data.detected_gender} | العمر: ${data.detected_age} سنة
                        </span>
                    </div>
                `;
                faceFiles = [];
                renderFacePreviews();
                document.querySelectorAll('.form-input').forEach(i => { if(i.type !== 'select-one') i.value = ''; });
            } else {
                resultDiv.innerHTML = `<div class="result-status denied" style="display: block; text-align: center; padding: 16px;">❌ ${data.message}</div>`;
            }
        } catch (error) {
            document.getElementById('enroll-result').innerHTML = `<div style="color: #e74c3c; text-align: center;">خطأ: ${error.message}</div>`;
        } finally {
            btn.disabled = false;
            btn.textContent = '✅ تسجيل الشخص';
        }
    }
</script>
{% endblock %}




In [ ]:
%%writefile /content/app/templates/reports.html
{% extends "base.html" %}

{% block title %}التقارير والإحصائيات - عين · Smart Security Gate{% endblock %}

{% block content %}
<div class="page-container">
    <div class="page-header">
        <h1 class="page-title">📊 لوحة الإحصائيات والسجلات</h1>
        <p class="page-subtitle">إحصائيات حية + سجل كامل لعمليات التحقق</p>
    </div>

    <!-- بطاقات الإحصائيات السريعة -->
    <div style="display: grid; grid-template-columns: repeat(auto-fit, minmax(180px, 1fr)); gap: 16px; margin-bottom: 24px;">
        <div class="content-card" style="text-align: center; padding: 20px;">
            <div style="font-size: 32px;">📋</div>
            <div style="color: #8095b0; font-size: 13px; margin-top: 4px;">إجمالي العمليات</div>
            <div style="font-size: 28px; color: #6db4f0; font-weight: bold; margin-top: 4px;" id="stat-total">0</div>
        </div>
        <div class="content-card" style="text-align: center; padding: 20px;">
            <div style="font-size: 32px;">✅</div>
            <div style="color: #8095b0; font-size: 13px; margin-top: 4px;">مسموح بالدخول</div>
            <div style="font-size: 28px; color: #2ecc71; font-weight: bold; margin-top: 4px;" id="stat-granted">0</div>
        </div>
        <div class="content-card" style="text-align: center; padding: 20px;">
            <div style="font-size: 32px;">❌</div>
            <div style="color: #8095b0; font-size: 13px; margin-top: 4px;">مرفوض</div>
            <div style="font-size: 28px; color: #e74c3c; font-weight: bold; margin-top: 4px;" id="stat-denied">0</div>
        </div>
        <div class="content-card" style="text-align: center; padding: 20px;">
            <div style="font-size: 32px;">🛑</div>
            <div style="color: #8095b0; font-size: 13px; margin-top: 4px;">محاولات انتحال</div>
            <div style="font-size: 28px; color: #f39c12; font-weight: bold; margin-top: 4px;" id="stat-spoof">0</div>
        </div>
        <div class="content-card" style="text-align: center; padding: 20px;">
            <div style="font-size: 32px;">📈</div>
            <div style="color: #8095b0; font-size: 13px; margin-top: 4px;">نسبة القبول</div>
            <div style="font-size: 28px; color: #6db4f0; font-weight: bold; margin-top: 4px;" id="stat-rate">0%</div>
        </div>
    </div>

    <!-- الرسم البياني -->
    <div class="content-card" style="margin-bottom: 24px;">
        <h3 style="color: #ffffff; margin-bottom: 16px;">📊 توزيع العمليات</h3>
        <div id="chart-container" style="display: flex; align-items: flex-end; gap: 24px; height: 200px; padding: 16px;">
            <div style="flex: 1; display: flex; flex-direction: column; align-items: center; gap: 8px;">
                <div style="color: #2ecc71; font-weight: bold;" id="bar-granted-val">0</div>
                <div id="bar-granted" style="width: 100%; background: linear-gradient(180deg, #2ecc71, #27ae60); border-radius: 8px 8px 0 0; min-height: 4px; transition: height 0.5s;"></div>
                <div style="color: #8095b0; font-size: 13px;">✅ مسموح</div>
            </div>
            <div style="flex: 1; display: flex; flex-direction: column; align-items: center; gap: 8px;">
                <div style="color: #e74c3c; font-weight: bold;" id="bar-denied-val">0</div>
                <div id="bar-denied" style="width: 100%; background: linear-gradient(180deg, #e74c3c, #c0392b); border-radius: 8px 8px 0 0; min-height: 4px; transition: height 0.5s;"></div>
                <div style="color: #8095b0; font-size: 13px;">❌ مرفوض</div>
            </div>
            <div style="flex: 1; display: flex; flex-direction: column; align-items: center; gap: 8px;">
                <div style="color: #f39c12; font-weight: bold;" id="bar-spoof-val">0</div>
                <div id="bar-spoof" style="width: 100%; background: linear-gradient(180deg, #f39c12, #d68910); border-radius: 8px 8px 0 0; min-height: 4px; transition: height 0.5s;"></div>
                <div style="color: #8095b0; font-size: 13px;">🛑 انتحال</div>
            </div>
        </div>
    </div>

    <!-- البحث والسجل -->
    <div class="content-card">
        <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 16px; flex-wrap: wrap; gap: 12px;">
            <h3 style="color: #ffffff; margin: 0;">📋 سجل العمليات</h3>
            <div style="display: flex; gap: 8px;">
                <a href="/api/export-logs" class="btn btn-primary" style="padding: 8px 16px; text-decoration: none;">📥 تصدير CSV</a>
                <button class="btn btn-secondary" onclick="loadLogs()">🔄 تحديث</button>
            </div>
        </div>

        <!-- شريط البحث والفلترة -->
        <div style="display: grid; grid-template-columns: 2fr 1fr 1fr; gap: 12px; margin-bottom: 16px;">
            <input type="text" id="search-input" class="form-input" placeholder="🔍 ابحث بالاسم أو الهوية..." oninput="applyFilters()">
            <select id="filter-decision" class="form-input" onchange="applyFilters()">
                <option value="all">كل القرارات</option>
                <option value="granted">مسموح فقط</option>
                <option value="denied">مرفوض فقط</option>
                <option value="spoof">محاولات الانتحال</option>
            </select>
            <input type="date" id="filter-date" class="form-input" onchange="applyFilters()">
        </div>

        <div style="color: #8095b0; font-size: 13px; margin-bottom: 12px;">
            عرض: <strong style="color: #6db4f0;" id="visible-count">0</strong> من <strong style="color: #6db4f0;" id="total-count">0</strong>
        </div>

        <table class="data-table">
            <thead>
                <tr>
                    <th>الوقت</th>
                    <th>الاسم</th>
                    <th>الهوية</th>
                    <th>الرتبة</th>
                    <th>المركبة (وصول)</th>
                    <th>التطابق</th>
                    <th>القرار</th>
                </tr>
            </thead>
            <tbody id="logs-tbody">
                <tr>
                    <td colspan="7" style="text-align: center; padding: 32px; color: #8095b0;">
                        لا توجد سجلات بعد - ابدأ بتحليل صورة في صفحة "تحليل الوجه"
                    </td>
                </tr>
            </tbody>
        </table>
    </div>

    <!-- سجل المركبات (دخول/خروج) -->
    <div class="content-card" style="margin-top: 24px;">
        <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 16px; flex-wrap: wrap; gap: 12px;">
            <h3 style="color: #ffffff; margin: 0;">🚗 سجل المركبات</h3>
            <div style="color: #8095b0; font-size: 14px;">داخل المنشأة الآن: <strong style="color: #2ecc71; font-size: 18px;" id="veh-inside">0</strong> مركبة</div>
        </div>
        <table class="data-table">
            <thead>
                <tr>
                    <th>الوقت</th>
                    <th>اللوحة (عربي)</th>
                    <th>اللوحة (إنجليزي)</th>
                    <th>الحدث</th>
                </tr>
            </thead>
            <tbody id="veh-tbody">
                <tr><td colspan="4" style="text-align: center; padding: 24px; color: #8095b0;">لا توجد حركات مركبات بعد - جرّب "قراءة لوحة" في صفحة تحليل الوجه</td></tr>
            </tbody>
        </table>
    </div>
</div>
{% endblock %}

{% block extra_js %}
<script>
    let allLogs = [];

    async function loadLogs() {
        try {
            const response = await fetch('/api/logs');
            const data = await response.json();
            allLogs = (data.logs || []).slice().reverse();
            updateStats();
            applyFilters();
        } catch (error) {
            console.error('Failed to load logs:', error);
        }
    }

    function updateStats() {
        const total = allLogs.length;
        const granted = allLogs.filter(l => l.decision && l.decision.includes('GRANTED')).length;
        const spoof = allLogs.filter(l => l.decision && l.decision.includes('SPOOF')).length;
        const denied = total - granted - spoof;
        const rate = total > 0 ? ((granted / total) * 100).toFixed(0) : 0;

        document.getElementById('stat-total').textContent = total;
        document.getElementById('stat-granted').textContent = granted;
        document.getElementById('stat-denied').textContent = denied;
        document.getElementById('stat-spoof').textContent = spoof;
        document.getElementById('stat-rate').textContent = rate + '%';

        // تحديث الرسم البياني
        const max = Math.max(granted, denied, spoof, 1);
        document.getElementById('bar-granted').style.height = (granted/max*150) + 'px';
        document.getElementById('bar-denied').style.height = (denied/max*150) + 'px';
        document.getElementById('bar-spoof').style.height = (spoof/max*150) + 'px';
        document.getElementById('bar-granted-val').textContent = granted;
        document.getElementById('bar-denied-val').textContent = denied;
        document.getElementById('bar-spoof-val').textContent = spoof;
    }

    function applyFilters() {
        const search = document.getElementById('search-input').value.toLowerCase().trim();
        const decision = document.getElementById('filter-decision').value;
        const date = document.getElementById('filter-date').value;

        let filtered = allLogs.filter(log => {
            // البحث النصي
            if (search) {
                const name = (log.name_ar || '').toLowerCase();
                const id = (log.national_id || '').toLowerCase();
                if (!name.includes(search) && !id.includes(search)) return false;
            }
            // فلتر القرار
            if (decision !== 'all') {
                const dec = log.decision || '';
                if (decision === 'granted' && !dec.includes('GRANTED')) return false;
                if (decision === 'denied' && (dec.includes('GRANTED') || dec.includes('SPOOF'))) return false;
                if (decision === 'spoof' && !dec.includes('SPOOF')) return false;
            }
            // فلتر التاريخ
            if (date) {
                const logDate = (log.timestamp || '').split(' ')[0];
                if (logDate !== date) return false;
            }
            return true;
        });

        renderLogs(filtered);
        document.getElementById('visible-count').textContent = filtered.length;
        document.getElementById('total-count').textContent = allLogs.length;
    }

    function renderLogs(logs) {
        const tbody = document.getElementById('logs-tbody');
        if (logs.length === 0) {
            tbody.innerHTML = `<tr><td colspan="7" style="text-align: center; padding: 32px; color: #8095b0;">لا توجد نتائج مطابقة</td></tr>`;
            return;
        }
        tbody.innerHTML = logs.map(log => {
            const dec = log.decision || '';
            let statusClass = 'denied', statusText = '❌ مرفوض';
            if (dec.includes('GRANTED')) { statusClass = 'granted'; statusText = '✅ مسموح'; }
            else if (dec.includes('SPOOF')) { statusClass = 'denied'; statusText = '🛑 انتحال'; }
            return `
                <tr>
                    <td style="color: #8095b0; font-size: 13px;">${log.timestamp || '-'}</td>
                    <td>${log.name_ar || '-'}</td>
                    <td style="font-family: monospace; font-size: 13px;">${log.national_id || '-'}</td>
                    <td>${log.rank || '-'}</td>
                    <td style="letter-spacing: 1px;">${log.arrived_plate_ar || '-'}</td>
                    <td>${log.score ? (log.score * 100).toFixed(1) + '%' : '-'}</td>
                    <td>
                        <span class="result-status ${statusClass}" style="font-size: 12px; padding: 4px 10px;">
                            ${statusText}
                        </span>
                    </td>
                </tr>
            `;
        }).join('');
    }

    async function loadVehicleLogs() {
        try {
            const res = await fetch('/api/vehicle-logs');
            const data = await res.json();
            document.getElementById('veh-inside').textContent = data.inside_count || 0;
            const logs = (data.logs || []).slice().reverse();
            if (logs.length === 0) return;
            document.getElementById('veh-tbody').innerHTML = logs.map(l => `
                <tr>
                    <td style="color: #8095b0; font-size: 13px;">${l.timestamp || '-'}</td>
                    <td style="letter-spacing: 1px;">${l.plate_ar || '-'}</td>
                    <td style="font-family: monospace; font-size: 13px;">${l.plate_en || '-'}</td>
                    <td><span class="result-status ${l.event === 'ENTRY' ? 'granted' : 'denied'}" style="font-size: 12px; padding: 4px 10px;">${l.event === 'ENTRY' ? '🟢 دخول' : '🔵 خروج'}</span></td>
                </tr>`).join('');
        } catch (e) { console.error('vehicle logs:', e); }
    }

    loadLogs();
    loadVehicleLogs();
    setInterval(loadLogs, 5000);
    setInterval(loadVehicleLogs, 5000);
</script>
{% endblock %}




In [ ]:
%%writefile /content/app/templates/cameras.html
{% extends "base.html" %}

{% block title %}الكاميرات - عين · Smart Security Gate{% endblock %}

{% block content %}
<div class="page-container">
    <div class="page-header">
        <h1 class="page-title">🎥 ربط كاميرات المراقبة</h1>
        <p class="page-subtitle">إدارة وعرض حالة كاميرات المراقبة المتصلة بالنظام</p>
    </div>

    <div class="content-card">
        <div class="components-grid">
            {% for i in range(1, 7) %}
            <div class="component-card">
                <span class="component-icon">📹</span>
                <h3>الكاميرا {{ i }}</h3>
                <p style="color: #2ecc71;">● متصلة - 1080p</p>
                <p style="margin-top: 8px; font-size: 13px;">{{ ['البوابة الرئيسية', 'المدخل الجانبي', 'موقف السيارات', 'المخرج الأمامي', 'المدخل الخلفي', 'الساحة المركزية'][i-1] }}</p>
            </div>
            {% endfor %}
        </div>
    </div>
</div>
{% endblock %}




In [ ]:
%%writefile /content/app/templates/plate_reading.html
{% extends "base.html" %}

{% block title %}قراءة اللوحات - عين · Smart Security Gate{% endblock %}

{% block content %}
<div class="page-container">
    <div class="page-header">
        <h1 class="page-title">🚗 قراءة لوحات المركبات (LPR)</h1>
        <p class="page-subtitle">النظام يحدد اللوحة، يقرأ الحروف الإنجليزية ويحوّلها للعربية حسب نظام المرور السعودي، ويسجّل دخول/خروج المركبة</p>
    </div>

    <div class="content-card">
        <div id="plate-upload-area" class="upload-zone">
            <div class="upload-icon">🚗</div>
            <div class="upload-text">اسحب صورة المركبة/اللوحة هنا أو اضغط للاختيار</div>
            <div class="upload-hint">PNG, JPG, JPEG • الحد الأقصى: 50 ميجابايت</div>
            <input type="file" id="plate-file-input" accept="image/*" style="display: none;">
        </div>

        <div id="loading" style="display: none; text-align: center; padding: 32px;">
            <div style="color: #6db4f0; font-size: 18px;">🔍 جاري قراءة اللوحة...</div>
            <div style="color: #8095b0; font-size: 14px; margin-top: 8px;">يرجى الانتظار</div>
        </div>

        <div id="result-container"></div>
    </div>

    <div class="content-card" style="margin-top: 24px;">
        <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 12px; flex-wrap: wrap; gap: 12px;">
            <h3 style="color: #ffffff; margin: 0;">آخر حركات المركبات</h3>
            <div style="display: flex; align-items: center; gap: 16px;">
                <span style="color: #8095b0; font-size: 14px;">داخل المنشأة الآن: <strong style="color: #2ecc71;" id="veh-inside">0</strong></span>
                <a href="{{ url_for('reports') }}" class="btn btn-secondary" style="padding: 6px 14px; font-size: 13px; text-decoration: none;">السجل الكامل</a>
            </div>
        </div>
        <table class="data-table">
            <thead>
                <tr><th>الوقت</th><th>اللوحة (عربي)</th><th>اللوحة (إنجليزي)</th><th>الحدث</th></tr>
            </thead>
            <tbody id="veh-tbody">
                <tr><td colspan="4" style="text-align: center; padding: 20px; color: #8095b0;">لا توجد حركات بعد</td></tr>
            </tbody>
        </table>
    </div>
</div>
{% endblock %}

{% block extra_js %}
<script>
    const resultContainer = document.getElementById('result-container');
    const loading = document.getElementById('loading');

    // === تنبيه صوتي وبصري ===
    function gateAlert(kind) {
        const cfg = {
            entry:  { color: '#2ecc71', freq: 760, beeps: 1, dur: 0.15, type: 'sine' },
            exit:   { color: '#6db4f0', freq: 520, beeps: 1, dur: 0.15, type: 'sine' },
            denied: { color: '#e74c3c', freq: 220, beeps: 2, dur: 0.18, type: 'square' }
        }[kind] || { color: '#6db4f0', freq: 600, beeps: 1, dur: 0.15, type: 'sine' };
        const flash = document.createElement('div');
        flash.style.cssText = 'position:fixed;inset:0;pointer-events:none;z-index:9999;opacity:0;transition:opacity .1s;background:' + cfg.color;
        document.body.appendChild(flash);
        let step = 0;
        const blink = setInterval(() => {
            flash.style.opacity = (step % 2 === 0) ? '0.35' : '0';
            step++;
            if (step >= cfg.beeps * 2) { clearInterval(blink); flash.style.opacity = '0'; setTimeout(() => flash.remove(), 200); }
        }, cfg.dur * 1000);
        try {
            const AC = window.AudioContext || window.webkitAudioContext;
            const ctx = new AC();
            for (let i = 0; i < cfg.beeps; i++) {
                const t0 = ctx.currentTime + i * (cfg.dur + 0.08);
                const osc = ctx.createOscillator();
                const g = ctx.createGain();
                osc.type = cfg.type;
                osc.frequency.value = cfg.freq;
                g.gain.setValueAtTime(0.0001, t0);
                g.gain.exponentialRampToValueAtTime(0.25, t0 + 0.01);
                g.gain.exponentialRampToValueAtTime(0.0001, t0 + cfg.dur);
                osc.connect(g); g.connect(ctx.destination);
                osc.start(t0); osc.stop(t0 + cfg.dur + 0.03);
            }
        } catch (e) {}
    }

    // === رفع الصورة ===
    const plateUploadArea = document.getElementById('plate-upload-area');
    const plateFileInput = document.getElementById('plate-file-input');
    plateUploadArea.addEventListener('click', () => plateFileInput.click());
    plateUploadArea.addEventListener('dragover', (e) => { e.preventDefault(); plateUploadArea.classList.add('dragover'); });
    plateUploadArea.addEventListener('dragleave', () => plateUploadArea.classList.remove('dragover'));
    plateUploadArea.addEventListener('drop', (e) => {
        e.preventDefault();
        plateUploadArea.classList.remove('dragover');
        if (e.dataTransfer.files.length > 0) handlePlateFile(e.dataTransfer.files[0]);
    });
    plateFileInput.addEventListener('change', (e) => {
        if (e.target.files.length > 0) handlePlateFile(e.target.files[0]);
    });

    async function handlePlateFile(file) {
        if (!file.type.startsWith('image/')) { alert('يرجى رفع ملف صورة فقط'); return; }
        const formData = new FormData();
        formData.append('image', file);
        loading.style.display = 'block';
        resultContainer.innerHTML = '';
        try {
            const response = await fetch('/api/read-plate', { method: 'POST', body: formData });
            const data = await response.json();
            displayPlateResult(data, file);
        } catch (error) {
            resultContainer.innerHTML = `<div style="color:#e74c3c;padding:16px;text-align:center;">حدث خطأ: ${error.message}</div>`;
        } finally {
            loading.style.display = 'none';
        }
    }

    function displayPlateResult(data, file) {
        if (data.status === 'duplicate') {
            resultContainer.innerHTML = `<div class="result-card"><div class="result-body" style="text-align:center;color:#f39c12;padding:20px;font-size:15px;">⚠️ ${data.message}</div></div>`;
            return;
        }
        gateAlert(data.status === 'ok' ? (data.event === 'ENTRY' ? 'entry' : 'exit') : 'denied');
        if (data.status !== 'ok') {
            resultContainer.innerHTML = `
                <div class="result-card">
                    <div class="result-header">
                        <h3 style="color:#ffffff;">نتيجة قراءة اللوحة</h3>
                        <span class="result-status denied">❌ لم تتم القراءة</span>
                    </div>
                    <div class="result-body" style="text-align:center;">
                        <div style="color:#e74c3c;font-size:16px;">${data.message || 'تعذّرت قراءة اللوحة'}</div>
                    </div>
                </div>`;
            return;
        }
        const imgSrc = data.annotated_image ? `data:image/jpeg;base64,${data.annotated_image}` : URL.createObjectURL(file);
        const isEntry = data.event === 'ENTRY';
        const detNote = data.detect_method === 'yolo' ? '✅ تم تحديد إطار اللوحة (YOLO)'
                      : data.detect_method === 'opencv' ? '🟠 تحديد تقريبي للوحة (تحليل الحواف)'
                      : data.detect_method === 'direct' ? '🔍 صورة لوحة مقصوصة (قراءة مباشرة)'
                      : '⚠️ قراءة على كامل الصورة (بدون كشف إطار)';
        resultContainer.innerHTML = `
            <div class="result-card">
                <div class="result-header">
                    <h3 style="color:#ffffff;">نتيجة قراءة اللوحة</h3>
                    <span class="result-status ${isEntry ? 'granted' : 'denied'}">
                        ${isEntry ? '🟢 تسجيل دخول' : '🔵 تسجيل خروج'}
                    </span>
                </div>
                <div class="result-body">
                    <div style="display:grid;grid-template-columns:300px 1fr;gap:32px;align-items:start;">
                        <div>
                            <img src="${imgSrc}" style="width:100%;border-radius:8px;border:2px solid #1a2942;">
                            <div style="margin-top:8px;text-align:center;color:#8095b0;font-size:12px;">${detNote}</div>
                            <div style="margin-top:4px;text-align:center;font-size:12px;color:${(data.engine==='chars'||data.engine==='hybrid'||data.engine==='easyocr_fallback')?'#2ecc71':'#e0a000'};">${data.engine==='chars'?'🧠 موديل الحروف المدرّب':data.engine==='hybrid'?'🔀 هجين: أرقام=موديلك + حروف=EasyOCR':data.engine==='easyocr_fallback'?'🧠 موديل الحروف (احتياط)':'🔡 EasyOCR'}</div>
                            <div style="margin-top:4px;text-align:center;color:#5a6b85;font-size:11px;direction:ltr;">${data.raw_en ? 'EN: ' + data.raw_en : ''}${data.raw_ar ? ' • AR: ' + data.raw_ar : ''}</div>
                            ${data.fusion_notes && data.fusion_notes.length ? `<div style="margin-top:4px;text-align:center;color:#6db4f0;font-size:11px;">${data.fusion_notes.join(' • ')}</div>` : ''}
                        </div>
                        <div class="info-section">
                            <h4>بيانات اللوحة</h4>
                            <div class="info-row"><span class="info-label">اللوحة (عربي)</span><span class="info-value" style="font-size:18px;letter-spacing:2px;">${data.plate_ar}</span></div>
                            <div class="info-row"><span class="info-label">اللوحة (إنجليزي)</span><span class="info-value" style="font-size:18px;letter-spacing:2px;">${data.plate_en}</span></div>
                            <div class="info-row"><span class="info-label">الحدث</span><span class="info-value" style="color:${isEntry ? '#2ecc71' : '#6db4f0'};">${data.event_ar}</span></div>
                            <div class="info-row"><span class="info-label">الوقت</span><span class="info-value">${data.timestamp}</span></div>
                        </div>
                    </div>
                </div>
            </div>`;
        loadRecent();
    }

    // === آخر الحركات ===
    async function loadRecent() {
        try {
            const res = await fetch('/api/vehicle-logs');
            const data = await res.json();
            document.getElementById('veh-inside').textContent = data.inside_count || 0;
            const logs = (data.logs || []).slice().reverse().slice(0, 6);
            if (logs.length === 0) return;
            document.getElementById('veh-tbody').innerHTML = logs.map(l => `
                <tr>
                    <td style="color: #8095b0; font-size: 13px;">${l.timestamp || '-'}</td>
                    <td style="letter-spacing: 1px;">${l.plate_ar || '-'}</td>
                    <td style="font-family: monospace; font-size: 13px;">${l.plate_en || '-'}</td>
                    <td><span class="result-status ${l.event === 'ENTRY' ? 'granted' : 'denied'}" style="font-size: 12px; padding: 4px 10px;">${l.event === 'ENTRY' ? '🟢 دخول' : '🔵 خروج'}</span></td>
                </tr>`).join('');
        } catch (e) { console.error(e); }
    }
    loadRecent();
</script>
{% endblock %}


In [ ]:
%%writefile /content/app/templates/cards.html
{% extends "base.html" %}

{% block title %}إدارة الأشخاص - عين · Smart Security Gate{% endblock %}

{% block content %}
<div class="page-container">
    <div class="page-header">
        <h1 class="page-title">👥 إدارة الأشخاص والقائمة السوداء</h1>
        <p class="page-subtitle">عرض كل المسجّلين والتحكم بصلاحياتهم</p>
    </div>

    <!-- إحصائيات سريعة -->
    <div style="display: grid; grid-template-columns: repeat(auto-fit, minmax(160px, 1fr)); gap: 16px; margin-bottom: 24px;">
        <div class="content-card" style="text-align: center; padding: 20px;">
            <div style="font-size: 28px;">👥</div>
            <div style="color: #8095b0; font-size: 13px;">إجمالي المسجّلين</div>
            <div style="font-size: 26px; color: #6db4f0; font-weight: bold;" id="emp-total">0</div>
        </div>
        <div class="content-card" style="text-align: center; padding: 20px;">
            <div style="font-size: 28px;">✅</div>
            <div style="color: #8095b0; font-size: 13px;">نشطون</div>
            <div style="font-size: 26px; color: #2ecc71; font-weight: bold;" id="emp-active">0</div>
        </div>
        <div class="content-card" style="text-align: center; padding: 20px;">
            <div style="font-size: 28px;">🚫</div>
            <div style="color: #8095b0; font-size: 13px;">في القائمة السوداء</div>
            <div style="font-size: 26px; color: #e74c3c; font-weight: bold;" id="emp-suspended">0</div>
        </div>
    </div>

    <div class="content-card">
        <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 16px; flex-wrap: wrap; gap: 12px;">
            <h3 style="color: #ffffff; margin: 0;">قائمة الأشخاص</h3>
            <button class="btn btn-secondary" onclick="loadEmployees()">🔄 تحديث</button>
        </div>

        <!-- بحث وفلترة -->
        <div style="display: grid; grid-template-columns: 2fr 1fr; gap: 12px; margin-bottom: 16px;">
            <input type="text" id="search-input" class="form-input" placeholder="🔍 ابحث بالاسم أو الهوية..." oninput="applyFilter()">
            <select id="filter-status" class="form-input" onchange="applyFilter()">
                <option value="all">كل الحالات</option>
                <option value="ACTIVE">نشطون فقط</option>
                <option value="SUSPENDED">القائمة السوداء فقط</option>
            </select>
        </div>

        <table class="data-table">
            <thead>
                <tr>
                    <th>الاسم</th>
                    <th>الهوية</th>
                    <th>الجنسية</th>
                    <th>الرتبة</th>
                    <th>الصلاحية</th>
                    <th>الجنس</th>
                    <th>الحالة</th>
                    <th>إجراء</th>
                </tr>
            </thead>
            <tbody id="emp-tbody">
                <tr><td colspan="8" style="text-align: center; padding: 32px; color: #8095b0;">جاري التحميل...</td></tr>
            </tbody>
        </table>
    </div>
</div>
{% endblock %}

{% block extra_js %}
<script>
let allEmployees = [];

async function loadEmployees() {
    try {
        const response = await fetch('/api/employees');
        const data = await response.json();
        allEmployees = data.employees || [];
        updateStats();
        applyFilter();
    } catch (error) {
        console.error('Failed to load:', error);
    }
}

function updateStats() {
    const total = allEmployees.length;
    const active = allEmployees.filter(e => e.access_status === 'ACTIVE').length;
    const suspended = allEmployees.filter(e => e.access_status === 'SUSPENDED').length;
    document.getElementById('emp-total').textContent = total;
    document.getElementById('emp-active').textContent = active;
    document.getElementById('emp-suspended').textContent = suspended;
}

function applyFilter() {
    const search = document.getElementById('search-input').value.toLowerCase().trim();
    const status = document.getElementById('filter-status').value;

    let filtered = allEmployees.filter(emp => {
        if (search) {
            const name = (emp.name_ar || '').toLowerCase();
            const id = (emp.national_id || '').toLowerCase();
            if (!name.includes(search) && !id.includes(search)) return false;
        }
        if (status !== 'all' && emp.access_status !== status) return false;
        return true;
    });

    renderEmployees(filtered);
}

function renderEmployees(employees) {
    const tbody = document.getElementById('emp-tbody');
    if (employees.length === 0) {
        tbody.innerHTML = '<tr><td colspan="8" style="text-align: center; padding: 32px; color: #8095b0;">لا توجد نتائج</td></tr>';
        return;
    }
    tbody.innerHTML = employees.map(emp => {
        const isSuspended = emp.access_status === 'SUSPENDED';
        return `
            <tr>
                <td>${emp.name_ar || '-'}</td>
                <td style="font-family: monospace; font-size: 13px;">${emp.national_id || '-'}</td>
                <td>${emp.nationality_ar || '-'}</td>
                <td>${emp.rank_ar || '-'}</td>
                <td>${emp.clearance_ar || '-'}</td>
                <td>${emp.gender_ar || '-'}</td>
                <td>
                    <span class="result-status ${isSuspended ? 'denied' : 'granted'}" style="font-size: 12px; padding: 4px 10px;">
                        ${isSuspended ? '🚫 موقوف' : '✅ نشط'}
                    </span>
                </td>
                <td>
                    <button class="btn ${isSuspended ? 'btn-primary' : 'btn-secondary'}"
                            style="padding: 6px 14px; font-size: 13px;"
                            onclick="toggleAccess('${emp.folder_name}', '${emp.name_ar}')">
                        ${isSuspended ? '✅ تفعيل' : '🚫 إيقاف'}
                    </button>
                </td>
            </tr>
        `;
    }).join('');
}

async function toggleAccess(personId, name) {
    const confirmMsg = `هل تريد تغيير حالة ${name}؟`;
    if (!confirm(confirmMsg)) return;

    try {
        const response = await fetch(`/api/toggle-access/${personId}`, { method: 'POST' });
        const data = await response.json();
        if (data.status === 'success') {
            loadEmployees();
        } else {
            alert('خطأ: ' + (data.message || 'فشلت العملية'));
        }
    } catch (error) {
        alert('خطأ في الاتصال: ' + error.message);
    }
}

loadEmployees();
</script>
{% endblock %}




In [ ]:
import os
print('✅ الملفات:')
for root, dirs, files in os.walk('/content/app'):
    for f in files:
        print('  ', os.path.join(root,f).replace('/content/app/',''))

# ⚡ ② تحميل الموديل من Drive (موديلك الجاهز — بدون تدريب)
شغّلها بعد الإعداد مباشرة — تجيب `plate_chars.pt` من درايفك.

In [ ]:
# ⚡ تحميل موديل الحروف الجاهز من Google Drive (بدون تدريب)
import os, shutil
os.makedirs('/content/app/models', exist_ok=True)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as e:
    print('⚠️ تعذّر ربط Drive:', e)
src = '/content/drive/MyDrive/plate_chars.pt'
if os.path.exists(src):
    shutil.copy(src, '/content/app/models/plate_chars.pt')
    sz = round(os.path.getsize(src) / 1e6, 1)
    print(f'✅ حمّلنا الموديل من Drive ({sz} MB) → /content/app/models/plate_chars.pt')
    print('🔄 الحين شغّل خلية 7️⃣ تشغيل النظام (بتشوف: Char reader OK ✨)')
else:
    print('❌ ما لقينا الموديل في Drive (MyDrive/plate_chars.pt).')
    print('   معناها ما حفظته قبل — لازم تشغّل خلايا التدريب 🎓 مرة وحدة.')


# 🧑 ③ (اختياري) رفع الوجوه + الأسماء — **قبل التشغيل**
للتعرّف على الوجوه فقط: ارفع داتاسيت الوجوه ثم employees.csv. تخطّاه لو تختبر اللوحات فقط.

In [ ]:
from google.colab import files
import zipfile, os, shutil
os.chdir('/content/app')

EXPECTED = 'smart_gate_dataset/smart_gate_dataset_augmented'

# 1) رفع واستخراج الداتاسيت (لو مو موجود)
if not os.path.exists('smart_gate_dataset'):
    print('📤 ارفع smart_gate_dataset_augmented.zip')
    uploaded = files.upload()
    zip_name = next(iter(uploaded.keys()))
    with zipfile.ZipFile(zip_name, 'r') as z:
        z.extractall('smart_gate_dataset')
    print(f'✅ استُخرج {zip_name}')

# 2) أدوات فحص المسار والصور
def count_people(path):
    if not os.path.isdir(path):
        return 0, 0
    ppl = [d for d in os.listdir(path) if os.path.isdir(os.path.join(path, d))]
    imgs = 0
    for p in ppl:
        imgs += len([f for f in os.listdir(os.path.join(path, p))
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    return len(ppl), imgs

def find_people_dir(root):
    best, best_n = None, 0
    for cur, dirs, _ in os.walk(root):
        n = 0
        for d in dirs:
            dp = os.path.join(cur, d)
            try:
                if any(x.lower().endswith(('.jpg', '.jpeg', '.png')) for x in os.listdir(dp)):
                    n += 1
            except Exception:
                pass
        if n > best_n:
            best_n, best = n, cur
    return best, best_n

# 3) لو المسار المتوقّع فاضي، ندوّر مكان الصور الفعلي ونصحّح الهيكل تلقائياً
np_, ni = count_people(EXPECTED)
if ni == 0:
    print('🔎 ما فيه صور في المسار المتوقّع — نبحث عن مجلدات الأشخاص الفعلية...')
    pd, n = find_people_dir('smart_gate_dataset')
    if pd and n > 0 and os.path.abspath(pd) != os.path.abspath(EXPECTED):
        os.makedirs(EXPECTED, exist_ok=True)
        for sub in os.listdir(pd):
            src = os.path.join(pd, sub)
            if os.path.isdir(src) and os.path.abspath(src) != os.path.abspath(EXPECTED):
                shutil.move(src, os.path.join(EXPECTED, sub))
        try:
            if os.path.isdir(pd) and not os.listdir(pd):
                os.rmdir(pd)
        except Exception:
            pass
        print(f'🔧 صحّحنا هيكل المسار تلقائياً ({n} مجلد شخص)')
    np_, ni = count_people(EXPECTED)

# 4) التقرير النهائي
if np_ > 0:
    print(f'✅ المسار صحيح: {EXPECTED}')
    print(f'   👥 {np_} شخص | 🖼️ {ni} صورة إجمالاً')
    print('   عيّنة مجلدات:', os.listdir(EXPECTED)[:5])
else:
    print('⚠️ ما لقينا أي مجلد شخص فيه صور!')
    print('   تأكد إن الأرشيف فيه مجلد لكل شخص (باسمه) وداخله صور jpg/png.')
    print('   محتوى smart_gate_dataset:', os.listdir('smart_gate_dataset') if os.path.exists('smart_gate_dataset') else 'غير موجود')


In [ ]:
import csv as _csv
import os
from google.colab import files
os.chdir('/content/app')

print('📤 ارفع employees.csv')
uploaded = files.upload()
csv_name = next(iter(uploaded.keys()))
if csv_name != 'employees.csv':
    os.rename(csv_name, 'employees.csv')

# فحص سريع لمحتوى الملف
try:
    with open('employees.csv', encoding='utf-8-sig') as f:
        rows = list(_csv.DictReader(f))
    print(f'✅ employees.csv: {len(rows)} صف')
    if rows:
        print('   الأعمدة:', list(rows[0].keys()))
except Exception as e:
    print('⚠️ مشكلة في قراءة CSV:', e)


# 🚀 ④ التشغيل

In [ ]:
# 🎛️ اختر محرك قراءة اللوحة — شغّلها قبل خلية التشغيل 🚀
import os
# بعد ما درّبت موديلك، 'chars' هو الأنسب — يستخدم حروفك المدرّبة (مو EasyOCR).
# 'chars'   ⭐ = موديلك المدرّب يقرأ كل شي (أرقام + حروف)
# 'hybrid'      = أرقام من موديلك + حروف من EasyOCR
# 'easyocr'     = EasyOCR فقط (احتياط)
os.environ['PRIMARY_ENGINE'] = 'chars'
print('✅ محرك القراءة:', os.environ['PRIMARY_ENGINE'])
print('👉 شغّل 🚀 بعدها. غيّرت الوضع أو درّبت؟ أعد تشغيل 🚀.')


In [ ]:
# 🚀 تشغيل النظام — يضمن تحميل أحدث موديل مدرّب تلقائياً (يداوي نفسه)
import os, subprocess, time, shutil
os.chdir('/content/app')
os.makedirs('models', exist_ok=True)

# (1) يدوّر أحدث موديل مدرّب من كل الأماكن المحتملة ويحطه في مكانه الصح
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass
cands = []
for p in ['/content/app/models/plate_chars.pt',
          '/content/drive/MyDrive/plate_chars.pt',
          '/content/drive/MyDrive/plate_last.pt',
          '/content/drive/MyDrive/sg_train/plate_chars.pt',
          '/content/runs/b/weights/best.pt',
          '/content/runs/b/weights/last.pt']:
    if os.path.exists(p):
        cands.append((os.path.getmtime(p), p))
if cands:
    newest = sorted(cands, reverse=True)[0][1]
    if os.path.abspath(newest) != os.path.abspath('models/plate_chars.pt'):
        shutil.copy(newest, 'models/plate_chars.pt')
    print('✅ موديل الحروف جاهز (أحدث نسخة):', newest)
else:
    print('⚠️ ما فيه موديل مدرّب — النظام بيشتغل بـ EasyOCR (درّب أول للأفضل)')

# (2) يقتل أي نسخة قديمة ويشغّل جديدة (عشان يحمّل الموديل)
os.system("pkill -9 -f app.py 2>/dev/null")
time.sleep(3)
log_f = open('flask.log', 'w')
subprocess.Popen(['python', 'app.py'], stdout=log_f, stderr=subprocess.STDOUT)
print('🚀 النظام يحمّل بالخلفية. استنى 4-6 دقائق ثم شغّل خلية الفحص 🩺.')
print('   للمتابعة: !tail -n 20 /content/app/flask.log')

In [ ]:
# 🩺 فحص النظام + الرابط — شغّلها بعد 🚀 بـ 4-6 دقائق
import os, requests
try:
    r = requests.get('http://localhost:5000/api/lpr-status', timeout=5).json()
    print('🧠 موديل الحروف:', '✅ محمّل وشغّال' if r.get('char_model') else '❌ غير محمّل (EasyOCR)')
    print('🔡 EasyOCR:', '✅' if r.get('easyocr') else '❌', '| كاشف اللوحة YOLO:', '✅' if r.get('plate_detector') else '❌')
    print('⚙️ المحرك الحالي:', os.environ.get('PRIMARY_ENGINE', 'easyocr'))
    from google.colab.output import eval_js
    print('\n🎉 افتح الرابط:', eval_js('google.colab.kernel.proxyPort(5000)'))
except Exception:
    print('⏳ لسا يحمّل — استنى دقيقة وأعد تشغيل الخلية. آخر اللوق:')
    os.system('tail -n 15 /content/app/flask.log')


# 🧪 ⑤ اختبار قراءة اللوحات

In [ ]:
# 📥 تنزيل تلقائي من Kaggle عبر kagglehub — بدون متصفح وبدون روابط
import os, glob, shutil

# اختر الداتاسيت (شِل علامة # عن اللي تبيه وحط # على الباقي):
DATASET = 'riotulab/saudi-license-plate-characters'       # ~593 لوحة سعودية مقصوصة وواضحة (الأنسب لاختبار القراءة)
# DATASET = 'riotulab/car-and-license-plate-detection'    # سيارات كاملة بلوحاتها (اختبار الكشف + القراءة)
# DATASET = 'aamirhasankhan/ksa-license-plates-labled'    # لوحات KSA معلّمة

try:
    import kagglehub
except ImportError:
    os.system('pip install -q kagglehub')
    import kagglehub

try:
    path = kagglehub.dataset_download(DATASET)
    print('📦 نزل في:', path)
except Exception as e:
    raise SystemExit(f'⚠️ تعذّر التنزيل: {e}\n'
                     'لو طلب تسجيل دخول: سوّ حساب Kaggle مجاني ثم نفّذ في خلية: import kagglehub; kagglehub.login()')

os.makedirs('/content/plate_tests/imgs', exist_ok=True)
exts = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')
srcs = [p for p in glob.glob(path + '/**/*', recursive=True) if p.lower().endswith(exts)]
copied = 0
for p in srcs:
    dst = f'/content/plate_tests/imgs/{copied:05d}_' + os.path.basename(p)
    try:
        shutil.copy(p, dst)
        copied += 1
    except Exception:
        pass

print(f'✅ جهّزنا {copied} صورة في /content/plate_tests/imgs')
print('🎲 الحين شغّل خلية "اختبار عشوائي بصري" مباشرة — بتلقى الصور جاهزة بدون رفع')

In [ ]:
# 🎲 عينة عشوائية: الصورة + القراءة جنب بعض وأنت الحكم — أعد التشغيل لعينة جديدة
import os, glob, zipfile, random, base64, requests
from IPython.display import display, HTML
from google.colab import files

os.makedirs('/content/plate_tests', exist_ok=True)
os.chdir('/content/plate_tests')

exts = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')
imgs = [p for p in glob.glob('imgs/**/*', recursive=True) if p.lower().endswith(exts)]
if not imgs:
    print('📤 ارفع zip فيه صور اللوحات/السيارات (أول مرة فقط)')
    up = files.upload()
    for name in up:
        if name.lower().endswith('.zip'):
            with zipfile.ZipFile(name) as z:
                z.extractall('imgs')
            print(f'✅ فُك {name}')
    imgs = [p for p in glob.glob('imgs/**/*', recursive=True) if p.lower().endswith(exts)]

assert imgs, '⚠️ ما فيه صور — تأكد إن الـ zip فيه صور jpg/png'

SAMPLE = 12   # ← عدد الصور بالعينة، عدّله مثل ما تبي
sample = random.sample(imgs, min(SAMPLE, len(imgs)))
print(f'🎲 عينة عشوائية: {len(sample)} من أصل {len(imgs)} صورة — جاري القراءة...')

cards = []
for p in sample:
    try:
        with open(p, 'rb') as fimg:
            r = requests.post('http://localhost:5000/api/read-plate',
                              files={'image': fimg}, data={'log': '0'}, timeout=90).json()
    except Exception as e:
        r = {'status': 'error', 'message': str(e)}
    ok = r.get('status') == 'ok'
    if ok and r.get('annotated_image'):
        img_src = 'data:image/jpeg;base64,' + r['annotated_image']
    else:
        with open(p, 'rb') as fimg:
            img_src = 'data:image/jpeg;base64,' + base64.b64encode(fimg.read()).decode()
    if ok:
        reading = (f"<div style='font-size:22px;direction:rtl;color:#111'><b>{r.get('plate_ar', '—')}</b></div>"
                   f"<div style='font-size:16px;letter-spacing:3px;color:#333'>{r.get('plate_en', '—')}</div>")
    else:
        reading = f"<div style='color:#c0392b;font-size:15px'>❌ {str(r.get('message', 'فشلت القراءة'))[:60]}</div>"
    method = {'yolo': 'كشف YOLO ✅', 'opencv': 'تحليل حواف 🟠',
              'direct': 'لوحة مقصوصة 🔍', 'none': 'كامل الصورة ⚠️'}.get(r.get('detect_method', ''), '')
    engine = {'chars':'🧠 موديل الحروف','hybrid':'🔀 هجين (موديل+EasyOCR)','easyocr_fallback':'🧠 موديل (احتياط)','easyocr':'🔡 EasyOCR'}.get(r.get('engine','easyocr'), r.get('engine','easyocr'))
    if ok and not r.get('complete', True):
        badge = '<div style="color:#b8860b;font-size:12px;text-align:center;margin-top:2px">⚠️ غير مؤكد (قراءة ناقصة)</div>'
        border = '#e0a000'
    elif ok:
        badge = '<div style="color:#2e7d32;font-size:12px;text-align:center;margin-top:2px">✓ مكتملة</div>'
        border = '#bbb'
    else:
        badge = ''
        border = '#bbb'
    cards.append(f"""
      <div style="border:1px solid {border};border-radius:10px;padding:10px;width:300px;display:inline-block;vertical-align:top;margin:6px;font-family:sans-serif;background:#fff">
        <img src="{img_src}" style="width:100%;border-radius:6px">
        <div style="margin-top:8px;text-align:center">{reading}</div>
        {badge}
        <div style="color:#777;font-size:11px;margin-top:6px;text-align:center">{os.path.basename(p)[:32]}</div>
        <div style="color:#999;font-size:10px;text-align:center">{method} • {engine}</div>
      </div>""")

display(HTML('<div dir="rtl">' + ''.join(cards) + '</div>'))
print('\n👀 قارن اللوحة في كل صورة مع القراءة تحتها — أعد تشغيل الخلية لعينة جديدة')

# 🎓 ⑥ (اختياري) إعادة تدريب الموديل من جديد
**ما تحتاجه إلا لو تبي تحسّن قراءة الحروف.** يحفظ موديل جديد بالدرايف + باك أب للقديم + تنزيل لجهازك. (يحتاج GPU + وقت)

> بعد التدريب: ارجع شغّل ④ التشغيل عشان النظام ياخذ الموديل الجديد.

In [ ]:
# 1️⃣ تنزيل وتجهيز داتاسيت حروف اللوحات (يعمل مع أي هيكل + تشخيص ذاتي)
import os, glob, shutil, random, yaml
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict

os.makedirs('/content/char_train', exist_ok=True)
os.chdir('/content/char_train')

# تقدر تبدّل الداتاسيت من هنا:
DATASET = 'riotulab/saudi-license-plate-characters'

try:
    import kagglehub
except ImportError:
    os.system('pip install -q kagglehub')
    import kagglehub
path = kagglehub.dataset_download(DATASET)
print('📦 نزل في:', path)

allfiles = [f for f in glob.glob(path + '/**/*', recursive=True) if os.path.isfile(f)]
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp')

# تشخيص: وش فيه بالداتاسيت
exts = Counter(os.path.splitext(f)[1].lower() for f in allfiles)
print('🔎 الملفات حسب الامتداد:', dict(exts))
for f in allfiles[:6]:
    print('   مثال:', os.path.relpath(f, path))

def stem_of(p):
    b = os.path.basename(p)
    for e in ('.txt', '.xml'):
        if b.lower().endswith(e):
            b = b[:-len(e)]
            break
    return os.path.splitext(b)[0]

imgs, txts, xmls = {}, {}, {}
for f in allfiles:
    low = f.lower()
    bn = os.path.basename(low)
    if low.endswith(IMG_EXT):
        imgs[stem_of(f)] = f
    elif low.endswith('.txt'):
        if 'classes' in bn or 'readme' in bn or 'darknet' in bn:
            continue
        txts[stem_of(f)] = f
    elif low.endswith('.xml'):
        xmls[stem_of(f)] = f

# نكتفي بـ (صورة + ليبل txt)؛ الـ xml اختياري لاستعادة الأسماء فقط
pairs = {}
for s in imgs:
    if s in txts:
        pairs[s] = {'img': imgs[s], 'txt': txts[s]}
        if s in xmls:
            pairs[s]['xml'] = xmls[s]
print(f'🖼️ صور: {len(imgs)} | ليبلات txt: {len(txts)} | xml: {len(xmls)} | أزواج صالحة: {len(pairs)}')
assert pairs, ('⚠️ ما لقينا أزواج (صورة + txt). محتوى الداتاسيت فوق — '
               'صوّره لكلود لو احتجت. تأكد إنه بصيغة YOLO (صور + ملفات txt).')

# === استعادة أسماء الأصناف: data.yaml ← ملف أصناف ← XML ← (تحذير) ===
names = None
yfiles = [f for f in allfiles if os.path.basename(f).lower() in ('data.yaml', 'data.yml')]
if yfiles:
    try:
        cfg = yaml.safe_load(open(yfiles[0], encoding='utf-8'))
        nm = cfg.get('names')
        if isinstance(nm, dict):
            names = [nm[k] for k in sorted(nm)]
        elif isinstance(nm, list):
            names = nm
        if names:
            print('🔤 أسماء الأصناف من data.yaml')
    except Exception as e:
        print('(تعذّر قراءة data.yaml:', e, ')')
if names is None:
    cfiles = [f for f in allfiles if os.path.basename(f).lower() in ('classes.txt', '_classes.txt', 'obj.names', '_darknet.labels')]
    if cfiles:
        names = [l.strip() for l in open(cfiles[0], encoding='utf-8') if l.strip()]
        print('🔤 أسماء الأصناف من ملف الأصناف:', os.path.basename(cfiles[0]))
if names is None and xmls:
    votes = defaultdict(Counter)
    for s, v in pairs.items():
        if 'xml' not in v:
            continue
        try:
            rootx = ET.parse(v['xml']).getroot()
            size = rootx.find('size')
            W = float(size.find('width').text); H = float(size.find('height').text)
            xo = []
            for obj in rootx.findall('object'):
                bb = obj.find('bndbox')
                cx = (float(bb.find('xmin').text) + float(bb.find('xmax').text)) / 2 / W
                cy = (float(bb.find('ymin').text) + float(bb.find('ymax').text)) / 2 / H
                xo.append((obj.find('name').text, cx, cy))
            for line in open(v['txt']):
                p = line.split()
                if len(p) < 5 or not xo:
                    continue
                cid = int(float(p[0])); tcx, tcy = float(p[1]), float(p[2])
                votes[cid][min(xo, key=lambda o: (o[1] - tcx) ** 2 + (o[2] - tcy) ** 2)[0]] += 1
        except Exception:
            continue
    if votes:
        mx = max(votes)
        names = [votes[i].most_common(1)[0][0] if votes.get(i) else str(i) for i in range(mx + 1)]
        print('🔤 أسماء الأصناف المستعادة من XML')
if names is None:
    mx = 0
    for v in pairs.values():
        for line in open(v['txt']):
            p = line.split()
            if p:
                mx = max(mx, int(float(p[0])))
    names = [str(i) for i in range(mx + 1)]
    print('⚠️ ما لقينا أسماء أصناف نصية — استخدمنا أرقام. لو الحروف طلعت غلط بعد التدريب، صوّر لكلود قائمة الملفات فوق.')
print(f'   {len(names)} صنف:', names)

# === تجهيز هيكل YOLO + تقسيم train/val ===
random.seed(42)
keys = list(pairs)
random.shuffle(keys)
nval = max(1, int(len(keys) * 0.1))
splits = {'val': keys[:nval], 'train': keys[nval:]}
if os.path.isdir('ds'):
    shutil.rmtree('ds')
for sp, ks in splits.items():
    os.makedirs(f'ds/images/{sp}', exist_ok=True)
    os.makedirs(f'ds/labels/{sp}', exist_ok=True)
    for k in ks:
        ext = os.path.splitext(pairs[k]['img'])[1]
        shutil.copy(pairs[k]['img'], f'ds/images/{sp}/{k}{ext}')
        shutil.copy(pairs[k]['txt'], f'ds/labels/{sp}/{k}.txt')

DATA_YAML = os.path.abspath('ds/data.yaml')
with open(DATA_YAML, 'w', encoding='utf-8') as f:
    yaml.safe_dump({'path': os.path.abspath('ds'), 'train': 'images/train', 'val': 'images/val',
                    'nc': len(names), 'names': names}, f, allow_unicode=True)
print('✅ data.yaml جاهز —', f"train: {len(splits['train'])} | val: {len(splits['val'])}")

In [ ]:
# 1.5️⃣ توليد بإعادة ترتيب الحروف الحقيقية — Real-Patch Recombination (فكرة الدكتور)
# يبني بنك رقع حقيقية per (صف عربي/لاتيني × صنف) من لوحات riotulab المعلّمة، ثم يولّد
# لوحات جديدة بزرع رقع حقيقية في كل خانة (كل صندوق مستقل) مع labels YOLO مثالية بحكم البناء.
import os, glob, random, shutil, yaml
import numpy as np
import cv2

# ============ الإعدادات ============
N_RECOMB           = 6000
KEEP_REAL          = True
REAL_OVERSAMPLE    = 3
MAX_PATCHES_PER_KEY = 80
PAD_V              = 0.06    # توسيع رأسي فقط (أفقي=0 يمنع بلع الحرف المجاور)
MIN_BANK_PER_CLASS = 2
JPEG_Q             = 90
SEED               = 42
random.seed(SEED); np.random.seed(SEED)

ROOT = '/content/char_train'; os.chdir(ROOT)
assert os.path.isfile('ds/data.yaml'), '⚠️ شغّل خلية التجهيز 1️⃣ أول'
cfg   = yaml.safe_load(open('ds/data.yaml', encoding='utf-8'))
names = cfg['names'] if isinstance(cfg['names'], list) else [cfg['names'][k] for k in sorted(cfg['names'])]
CLS_ID = {str(n): i for i, n in enumerate(names)}
LETTER_SET = set('ABDEGHJKLNRSTUVXZ'); DIGIT_SET = set('0123456789')

IMG_DIR, LBL_DIR = 'ds/images/train', 'ds/labels/train'
os.makedirs(IMG_DIR, exist_ok=True); os.makedirs(LBL_DIR, exist_ok=True)

def _is_real(p):
    b = os.path.basename(p)
    return ('synth_' not in b) and ('recomb_' not in b) and ('_dup' not in b)

real_imgs = sorted(p for p in glob.glob(IMG_DIR + '/*')
                   if _is_real(p) and os.path.splitext(p)[1].lower() in ('.jpg', '.jpeg', '.png', '.bmp'))
print(f'🖼️ لوحات حقيقية بالـtrain: {len(real_imgs)}')
assert real_imgs, '⚠️ ما فيه لوحات حقيقية بالـtrain — شغّل التجهيز 1️⃣'

def load_plate(img_path):
    stem = os.path.splitext(os.path.basename(img_path))[0]
    lbl = f'{LBL_DIR}/{stem}.txt'
    if not os.path.exists(lbl):
        return None
    img = cv2.imread(img_path)
    if img is None:
        return None
    Hpx, Wpx = img.shape[:2]
    boxes = []
    for line in open(lbl, encoding='utf-8'):
        p = line.split()
        if len(p) < 5:
            continue
        cid = int(float(p[0]))
        cx, cy, w, h = float(p[1]), float(p[2]), float(p[3]), float(p[4])
        if cid < 0 or cid >= len(names):
            continue
        bx1 = (cx - w / 2.0) * Wpx; by1 = (cy - h / 2.0) * Hpx
        bx2 = (cx + w / 2.0) * Wpx; by2 = (cy + h / 2.0) * Hpx
        boxes.append({'canon': str(names[cid]), 'cx': cx, 'cy': cy, 'w': w, 'h': h,
                      'bx1': bx1, 'by1': by1, 'bx2': bx2, 'by2': by2,
                      'pw': bx2 - bx1, 'ph': by2 - by1})
    return (img, Wpx, Hpx, boxes) if boxes else None

def split_two_rows(boxes):
    """تقسيم متين لصفين: فوق/تحت حول وسط cy (دائماً مجموعتان لو ≥2 صندوق)."""
    if len(boxes) < 2:
        return None
    cys = sorted(b['cy'] for b in boxes)
    mid = (cys[len(cys) // 2] + cys[(len(cys) - 1) // 2]) / 2.0   # وسط cy
    top    = [b for b in boxes if b['cy'] <  mid]
    bottom = [b for b in boxes if b['cy'] >= mid]
    if not top or not bottom:
        # احتياط: لو الوسط ما فصل (cy متقاربة) جرّب متوسط أعلى/أدنى
        ymin = min(b['cy'] for b in boxes); ymax = max(b['cy'] for b in boxes)
        mid = (ymin + ymax) / 2.0
        top    = [b for b in boxes if b['cy'] <  mid]
        bottom = [b for b in boxes if b['cy'] >= mid]
    if not top or not bottom:
        return None
    return top, bottom

def crop_patch(img, b, Wpx, Hpx):
    px = 0.0; py = b['ph'] * PAD_V                 # رأسي فقط
    x1 = int(max(0, round(b['bx1'] - px))); y1 = int(max(0, round(b['by1'] - py)))
    x2 = int(min(Wpx, round(b['bx2'] + px))); y2 = int(min(Hpx, round(b['by2'] + py)))
    if x2 <= x1 or y2 <= y1:
        return None
    return img[y1:y2, x1:x2].copy()

# ============ المرحلة A — بنك الرقع + قوالب (بدون مطابقة بين الصفين) ============
BANK = {}; plates = []; n_templates = n_skip = 0
rowcount = {}

def bank_add(key, patch):
    pool = BANK.setdefault(key, [])
    if len(pool) < MAX_PATCHES_PER_KEY:
        pool.append(patch)

for img_path in real_imgs:
    loaded = load_plate(img_path)
    if loaded is None:
        n_skip += 1; continue
    img, Wpx, Hpx, boxes = loaded
    sp = split_two_rows(boxes)
    if sp is None:
        n_skip += 1; continue
    top, bottom = sp
    rk = (len(top), len(bottom)); rowcount[rk] = rowcount.get(rk, 0) + 1
    arabic_row = sorted(top, key=lambda b: b['cx'])      # عربي فوق (cy أصغر)
    latin_row  = sorted(bottom, key=lambda b: b['cx'])   # لاتيني تحت
    tpl_boxes = []   # (box, row_lang)
    for b in arabic_row:
        p = crop_patch(img, b, Wpx, Hpx)
        if p is not None and p.size:
            bank_add(('arabic', b['canon']), p)
        tpl_boxes.append((b, 'arabic'))
    for b in latin_row:
        p = crop_patch(img, b, Wpx, Hpx)
        if p is not None and p.size:
            bank_add(('latin', b['canon']), p)
        tpl_boxes.append((b, 'latin'))
    plates.append({'boxes': tpl_boxes, 'Wpx': Wpx, 'Hpx': Hpx, '_canvas': img})
    n_templates += 1

print(f'✅ قوالب صالحة: {n_templates} | تخطّينا: {n_skip}')
print(f'🏦 مفاتيح البنك (صف×صنف): {len(BANK)}')
print(f'📐 توزيع (عدد فوق، عدد تحت): {dict(sorted(rowcount.items(), key=lambda x:-x[1])[:8])}')
assert plates, '⚠️ ما فيه قالب صالح — صوّر هذا الإخراج لكلود'

# الأصناف المتاحة للزرع لكل (صف، نوع) — عندها رقع كافية
def avail(row, charset):
    return sorted(c for c in charset if len(BANK.get((row, c), [])) >= MIN_BANK_PER_CLASS)
AV = {('arabic', 'L'): avail('arabic', LETTER_SET), ('arabic', 'D'): avail('arabic', DIGIT_SET),
      ('latin', 'L'):  avail('latin', LETTER_SET),  ('latin', 'D'):  avail('latin', DIGIT_SET)}
print(f"🔤 متاح للزرع — عربي: {len(AV[('arabic','L')])}ح/{len(AV[('arabic','D')])}ر | "
      f"لاتيني: {len(AV[('latin','L')])}ح/{len(AV[('latin','D')])}ر")
assert all(AV.values()), '⚠️ بنك ناقص لأحد (صف،نوع) — قلّل MIN_BANK_PER_CLASS'

def photometric_aug(img):
    out = img.astype(np.float32)
    out *= random.uniform(0.55, 1.25); out += random.uniform(-18, 18)
    if random.random() < 0.5:
        out += np.random.normal(0, random.uniform(2, 13), out.shape)
    out = np.clip(out, 0, 255).astype(np.uint8)
    if random.random() < 0.45:
        k = random.choice([3, 3, 5]); out = cv2.GaussianBlur(out, (k, k), 0)
    if random.random() < 0.25:
        h, w = out.shape[:2]; s = random.uniform(0.55, 0.85)
        small = cv2.resize(out, (max(1, int(w * s)), max(1, int(h * s))), interpolation=cv2.INTER_AREA)
        out = cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)
    return out

def local_bg(img, b, Wpx, Hpx):
    x1 = int(max(0, b['bx1'])); y1 = int(max(0, b['by1']))
    x2 = int(min(Wpx, b['bx2'])); y2 = int(min(Hpx, b['by2']))
    ox1, oy1 = max(0, x1 - 3), max(0, y1 - 3); ox2, oy2 = min(Wpx, x2 + 3), min(Hpx, y2 + 3)
    region = img[oy1:oy2, ox1:ox2]
    if region.size == 0:
        return (245, 245, 245)
    if region.shape[0] > 4 and region.shape[1] > 4:
        border = np.concatenate([region[0:2].reshape(-1, 3), region[-2:].reshape(-1, 3),
                                 region[:, 0:2].reshape(-1, 3), region[:, -2:].reshape(-1, 3)])
        return tuple(int(v) for v in np.median(border, axis=0))
    return tuple(int(v) for v in np.median(region.reshape(-1, 3), axis=0))

def plant(canvas, b, patch, Wpx, Hpx):
    x1 = max(0, int(round(b['bx1']))); y1 = max(0, int(round(b['by1'])))
    x2 = min(Wpx, int(round(b['bx2']))); y2 = min(Hpx, int(round(b['by2'])))
    tw, th = x2 - x1, y2 - y1
    if tw <= 1 or th <= 1:
        return
    cv2.rectangle(canvas, (x1, y1), (x2, y2), local_bg(canvas, b, Wpx, Hpx), thickness=-1)
    canvas[y1:y2, x1:x2] = cv2.resize(patch, (tw, th), interpolation=cv2.INTER_AREA)

# ============ توليد N_RECOMB لوحة — كل خانة تُزرع مستقلة برمز حقيقي بنفس النوع ============
os.makedirs('/content/recomb_preview', exist_ok=True)
made = attempts = 0; MAX_ATTEMPTS = N_RECOMB * 4; previews = []
while made < N_RECOMB and attempts < MAX_ATTEMPTS:
    attempts += 1
    tpl = random.choice(plates); Wpx, Hpx = tpl['Wpx'], tpl['Hpx']
    canvas = tpl['_canvas'].copy(); lines = []
    boxes_drawn = []
    for b, row in tpl['boxes']:
        typ = 'L' if b['canon'] in LETTER_SET else 'D'
        pool_classes = AV[(row, typ)]
        new_canon = random.choice(pool_classes)
        patch = random.choice(BANK[(row, new_canon)])
        plant(canvas, b, patch, Wpx, Hpx)
        lines.append(f"{CLS_ID[str(new_canon)]} {b['cx']:.6f} {b['cy']:.6f} {b['w']:.6f} {b['h']:.6f}")
        boxes_drawn.append(b)
    canvas = photometric_aug(canvas)
    cv2.imwrite(f'{IMG_DIR}/recomb_{made:05d}.jpg', canvas, [cv2.IMWRITE_JPEG_QUALITY, JPEG_Q])
    open(f'{LBL_DIR}/recomb_{made:05d}.txt', 'w', encoding='utf-8').write('\n'.join(lines))
    if made < 6:
        prev = canvas.copy()
        for b in boxes_drawn:
            cv2.rectangle(prev, (int(b['bx1']), int(b['by1'])), (int(b['bx2']), int(b['by2'])), (0, 200, 0), 1)
        cv2.imwrite(f'/content/recomb_preview/prev_{made}.jpg', prev); previews.append(prev)
    made += 1
    if made % 1500 == 0:
        print(f'  ولّدنا {made}/{N_RECOMB}')

print(f'✅ ولّدنا {made} لوحة مركّبة (recomb_*) من {n_templates} قالب حقيقي.')

if KEEP_REAL and REAL_OVERSAMPLE > 1:
    dups = 0
    for img_path in real_imgs:
        stem = os.path.splitext(os.path.basename(img_path))[0]; ext = os.path.splitext(img_path)[1]
        lbl = f'{LBL_DIR}/{stem}.txt'
        if not os.path.exists(lbl):
            continue
        for j in range(1, REAL_OVERSAMPLE):
            shutil.copy(img_path, f'{IMG_DIR}/{stem}_dup{j}{ext}')
            shutil.copy(lbl, f'{LBL_DIR}/{stem}_dup{j}.txt')
            dups += 1
    print(f'🔁 ضاعفنا {len(real_imgs)} حقيقية ×{REAL_OVERSAMPLE} (+{dups})')

n_img = len(glob.glob(IMG_DIR + '/*')); n_lbl = len(glob.glob(LBL_DIR + '/*.txt'))
print(f'📊 إجمالي train: {n_img} صورة | {n_lbl} label')
print('🔍 صوّر العيّنة تحت وتأكد إن الصناديق الخضراء على الحروف صح قبل التدريب 2️⃣')
if previews:
    from IPython.display import display
    from PIL import Image as _PILImage
    for prev in previews[:3]:
        display(_PILImage.fromarray(cv2.cvtColor(prev, cv2.COLOR_BGR2RGB)))


In [ ]:
# 2️⃣ تدريب موديل الحروف — خلية واحدة على A100 (epochs مرّة واحدة، بدون تقسيم دفعات)
# تشتغل بعد: 1️⃣ التجهيز (يبني DATA_YAML) ثم 1.5️⃣ التوليد بإعادة ترتيب الحروف.
import os, shutil

# ---- إعدادات قابلة للضبط ----
MODEL_ARCH = os.environ.get('CHAR_ARCH', 'yolov8s.pt')   # s متوازن؛ m لدقة أعلى على A100
EPOCHS     = int(os.environ.get('CHAR_EPOCHS', 100))     # A100 يخلّصها بجلسة واحدة
IMGSZ      = 640
BATCH      = int(os.environ.get('CHAR_BATCH', 64))       # A100 40GB يتحمّل 64 (لو OOM نزّله 32)

DATA_YAML = globals().get('DATA_YAML', '/content/char_train/ds/data.yaml')
assert os.path.isfile(DATA_YAML), '⚠️ شغّل خلية التجهيز 1️⃣ أول (DATA_YAML غير موجود): %s' % DATA_YAML

DRIVE_FINAL = '/content/drive/MyDrive/plate_chars.pt'
APP_MODELS  = '/content/app/models'
try:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    SAVE_TO_DRIVE = True
except Exception as e:
    print('ℹ️ Drive غير متاح (%s) — نحفظ محلياً فقط.' % e)
    SAVE_TO_DRIVE = False

print('🎓 تدريب موديل الحروف | arch=%s epochs=%s imgsz=%s batch=%s' % (MODEL_ARCH, EPOCHS, IMGSZ, BATCH))
from ultralytics import YOLO
model = YOLO(MODEL_ARCH)
model.train(
    data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
    device=0, workers=8, cache=True,            # cache=RAM: ما يعيد قراءة الصور كل لفة (أسرع كثير)
    fliplr=0.0, flipud=0.0,                      # قلب يفسد هوية الرمز (B↔مرآة، أرقام)
    degrees=8, perspective=0.0005, scale=0.5, shear=3, translate=0.1,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.5,
    patience=30, project='/content/runs', name='chars_v46', exist_ok=True,
)

# ---- الحفظ: best.pt (أدق من last) → Drive + مجلد التطبيق ----
run_dir = '/content/runs/chars_v46/weights'
src = os.path.join(run_dir, 'best.pt')
if not os.path.exists(src):
    src = os.path.join(run_dir, 'last.pt')
assert os.path.exists(src), '⚠️ ما لقينا أوزان مدرّبة في %s' % run_dir
os.makedirs(APP_MODELS, exist_ok=True)
shutil.copy(src, os.path.join(APP_MODELS, 'plate_chars.pt'))
print('✅ نُسخ best.pt → %s/plate_chars.pt (للجلسة)' % APP_MODELS)
# 💾 باك أب للموديل القديم قبل الكتابة فوقه (ما يضيع)
if SAVE_TO_DRIVE and os.path.exists(DRIVE_FINAL):
    try:
        shutil.copy(DRIVE_FINAL, '/content/drive/MyDrive/plate_chars_BACKUP.pt')
        print('💾 احتفظنا بالموديل القديم ← MyDrive/plate_chars_BACKUP.pt')
    except Exception as e:
        print('⚠️ تعذّر الباك أب:', e)
if SAVE_TO_DRIVE:
    try:
        shutil.copy(src, DRIVE_FINAL); print('✅ نُسخ → %s (يبقى بعد الجلسة)' % DRIVE_FINAL)
    except Exception as e:
        print('⚠️ تعذّر النسخ إلى Drive: %s' % e)
print('=' * 55)
print('✅ التدريب خلص (لفة واحدة، %s epochs). شغّل خلية التشغيل عشان النظام يستخدم الموديل الجديد.' % EPOCHS)
print("   التذكير: PRIMARY_ENGINE='easyocr' (افتراضي) موديل الحروف احتياط | 'chars' أساس | 'hybrid' أرقام منه")


In [ ]:
# ⬇️ نزّل الموديل الجديد لجهازك (نسخة احتياطية إضافية عندك — ينزل بمجلد Downloads)
from google.colab import files
files.download('/content/app/models/plate_chars.pt')


# 🍓 ⑦ (اختياري) ربط Raspberry Pi (بوابة فعلية)

In [ ]:
%%writefile raspberry_gate.py
# يُشغّل على Raspberry Pi (مو على Colab): python raspberry_gate.py
from flask import Flask, request, jsonify
import RPi.GPIO as GPIO
import time

app = Flask(__name__)
SERVO_PIN = 18

GPIO.setmode(GPIO.BCM)
GPIO.setup(SERVO_PIN, GPIO.OUT)
pwm = GPIO.PWM(SERVO_PIN, 50)   # 50Hz للسيرفو
pwm.start(0)

def move(angle):
    duty = 2.5 + (angle / 180.0) * 10.0
    pwm.ChangeDutyCycle(duty)
    time.sleep(0.8)
    pwm.ChangeDutyCycle(0)

@app.route('/open', methods=['POST'])
def open_gate():
    move(90)        # يفتح
    time.sleep(3)   # يبقى مفتوح 3 ثواني
    move(0)         # يغلق
    return jsonify({'status': 'opened'})

@app.route('/health')
def health():
    return jsonify({'status': 'ok'})

if __name__ == '__main__':
    try:
        app.run(host='0.0.0.0', port=8000)
    finally:
        pwm.stop()
        GPIO.cleanup()